# 🧬 Tensegrity LM — a byte-level language model, 1M → 115M parameters, on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/recursive-ai-dev/tensegrity/blob/main/notebooks/Tensegrity_Colab.ipynb)

**One notebook, six model sizes, no tokenizer.** Tensegrity reads and writes raw UTF-8 bytes, so it handles any language, code or markup out of the box. Pick a size, press *Run all*, and walk away: the notebook auto-detects your GPU, sizes the run to the hours you have, checkpoints to Google Drive, and picks up where it left off if Colab disconnects.

### Quick start
1. **Runtime → Change runtime type → GPU** (a free **T4** is enough for 1M–10M; 50M and 115M want an L4/A100 or patience).
2. Set **§1 · Configure** (model size and the hours you can spare). Everything else has a sensible default.
3. **Runtime → Run all.** Watch the live loss curve and the sample text improve.

### The size ladder

| size | parameters | layers × width | heads (q/kv) | FFN | context | default byte budget | what it is for |
|---|---:|---|---|---:|---:|---:|---|
| **1M** | 1.02M | 5 × 128 | 4/2 | 384 | 1024 | 300M | Smoke-test scale. Learns spelling, word shapes and simple sentence rhythm in minutes. |
| **2M** | 1.99M | 5 × 192 | 6/3 | 480 | 1024 | 400M | Fastest rung that produces locally coherent sentences on narrow corpora. |
| **5M** | 5.06M | 7 × 256 | 8/4 | 672 | 1024 | 600M | Sweet spot for a free-tier GPU: a few hours for a clearly coherent story model. |
| **10M** | 10.05M | 9 × 320 | 10/5 | 832 | 1024 | 800M | Noticeably better long-range consistency; still comfortable on a T4. |
| **50M** | 50.29M | 15 × 512 | 16/8 | 1664 | 1024 | 1,200M | Serious small model. T4: ~a day of compute; L4/A100 recommended. |
| **115M** | 115.69M | 18 × 768 | 24/12 | 2016 | 1024 | 1,750M | Top of the ladder (GPT-2-small class parameter count, byte-level). A100/L4 or patience. |

Every rung has `head_dim = 32`, GQA 2:1 and the same attention pattern, so a trained rung can be **grown into the next one with exact function preservation** (§6): the bigger model starts with *identical* outputs and a verified-trainable set of new parameters, and the notebook checks both properties every time it grows a model. Growth is a tool for *reuse and incremental extension* — keep what a small model learned, extend a run when you get more time. It is not a speed-up: in the small equal-compute test in the repository README, training the bigger model from scratch did as well or better.

### What's inside
* **Architecture** — RoPE · grouped-query attention · alternating banded-local / global layers (one fused kernel, no Python chunk loop) · SwiGLU · Q/K-norm · learnable residual gains · logit soft-cap + z-loss · optional ByteMix (causal FIR over recent byte embeddings) · confidence head that drives adaptive sampling temperature.
* **Training** — AdamW or Muon · warmup-stable-decay schedule sized to your time budget · weight EMA · bf16 on Ampere+, fp16 + loss-scaling on T4 (no accidental emulated bf16) · exact resume (weights, optimizer, EMA, every RNG, data order) · wall-clock stop · non-finite-step protection.
* **Extras** — live dashboard · confidence-shaded text · function-preserving growth · EWC + replay for continual learning · safetensors / INT8 export with a model card · optional Hugging Face upload.

> **Honest scope.** These are small research models. The individual mechanisms are established in the literature; the *combination* is not benchmarked against other architectures here — the ablations behind the defaults are in the repository README. Expect fluent-looking, narrow-domain text, not a general assistant.

## 0 · Setup

In [ ]:
#@title 0.1 · Install the few missing dependencies { display-mode: "form" }
import importlib, subprocess, sys

def _ensure(pkg, module=None):
    """Import `module`; pip-install `pkg` only if it is genuinely missing (Colab ships torch/numpy)."""
    try:
        importlib.import_module(module or pkg)
        return
    except ImportError:
        pass
    for extra in ([], ["--break-system-packages"]):  # second form for externally-managed Pythons
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg, *extra],
                           capture_output=True, text=True)
        if r.returncode == 0:
            importlib.invalidate_caches()
            importlib.import_module(module or pkg)
            return
    raise RuntimeError(f"could not install {pkg!r}:\n{r.stderr[-800:]}\nInstall it manually: pip install {pkg}")

for _pkg, _mod in [("numpy", "numpy"), ("torch", "torch"), ("safetensors", "safetensors"), ("pyarrow", "pyarrow")]:
    _ensure(_pkg, _mod)
print("dependencies OK")

<a name="source"></a>
### The library (click ▸ to expand)
The cells below write the `tensegrity` package to disk — about 3,500 lines, identical to the files in the GitHub repository (a CI test enforces this). They are collapsed by default; you never need to touch them. **Run this whole section once per session.**

In [ ]:
import os
os.makedirs("tensegrity_src/tensegrity", exist_ok=True)
print("writing the tensegrity package to ./tensegrity_src/tensegrity/ ...")

In [ ]:
%%writefile tensegrity_src/tensegrity/config.py
"""TensegrityConfig: the single description of a model's architecture.

Backward-compatibility contract
-------------------------------
Every field that existed in the original single-file release keeps its name,
type and default. Every field added since (``norm_eps``, ``confidence_detach``,
``byte_mix_kernel``, ``x0_mix``) defaults to the value that reproduces the
original behaviour exactly. A checkpoint saved by the old code therefore loads
into the new code (via :meth:`TensegrityConfig.from_dict`) as the very same
network, and the *presets* (see ``presets.py``) are what opt into new features.
"""

from __future__ import annotations

import dataclasses
import math
import warnings
from dataclasses import dataclass
from typing import Any, Dict, Optional


def ffn_hidden_dim(n_embd: int, mult: float, align: int) -> int:
    """SwiGLU hidden width: ``int(n_embd * mult)`` rounded UP to a multiple of ``align``."""
    assert n_embd > 0, f"n_embd must be positive, got {n_embd}"
    assert mult > 0, f"mult must be positive, got {mult}"
    assert align >= 1, f"align must be a positive integer, got {align}"
    hidden = int(n_embd * mult)
    hidden = align * ((hidden + align - 1) // align)
    return max(hidden, align)


@dataclass
class TensegrityConfig:
    # ------------------------------------------------------------------ shape
    vocab_size: int = 259
    block_size: int = 4096
    n_layer: int = 28
    n_embd: int = 2560
    n_head: int = 20
    n_kv_head: int = 4
    dropout: float = 0.0

    # -------------------------------------------------------------- attention
    sliding_window: int = 1024
    global_every: int = 4          # layer i is global iff (i + 1) % global_every == 0
    rope_base: float = 12000.0

    # -------------------------------------------------------------------- FFN
    ffn_mult: float = 8.0 / 3.0
    ffn_align: int = 64

    # ------------------------------------------- homeostasis / stochastic depth
    homeo_target: float = 1.0
    homeo_weight: float = 0.01
    gain_init: Optional[float] = None
    layerdrop_max: float = 0.1

    # ------------------------------------------------------- confidence head
    use_confidence_head: bool = True
    confidence_weight: float = 0.05

    # ------------------------------------------------------------------- misc
    bias: bool = False
    init_std: float = 0.02
    grad_checkpoint: bool = True
    grad_checkpoint_every: int = 1  # checkpoint every Nth block; 1 = all blocks

    use_qk_norm: bool = True        # per-head RMSNorm on q,k before RoPE
    z_loss_weight: float = 0.0      # PaLM/Gemma z-loss coefficient (0 = off)
    logit_soft_cap: Optional[float] = None  # Gemma-2 tanh soft-cap (None = off)
    embed_scale: bool = True        # multiply embeddings by sqrt(n_embd)

    # ---------------------------------------------------- added after v1 (all
    # defaults reproduce v1 behaviour exactly; see module docstring)
    norm_eps: float = 1e-6          # eps of the model-width RMSNorms (ln1/ln2/ln_f)
    confidence_detach: bool = False  # True: confidence head is a stop-gradient probe
    byte_mix_kernel: int = 0        # K>=2: learned causal FIR over the previous K-1 byte embeddings
    byte_mix_bilinear: bool = False  # gated bilinear n-gram interaction inside ByteMix
    x0_mix: bool = False            # per-layer learnable re-injection of the embedding stream
    v0_mix: bool = False            # per-layer learnable mixing of layer 0's value projection
    n_loops: int = 1                # number of recursive passes through the block stack (>=1)

    # --------------------------------------------------------------- validate
    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError(f"n_embd ({self.n_embd}) must be divisible by n_head ({self.n_head})")
        if (self.n_embd // self.n_head) % 2 != 0:
            raise ValueError(f"head_dim ({self.n_embd // self.n_head}) must be even for RoPE")
        if self.n_head % self.n_kv_head != 0:
            raise ValueError(f"n_head ({self.n_head}) must be divisible by n_kv_head ({self.n_kv_head})")
        if self.n_layer < 1:
            raise ValueError(f"n_layer must be >= 1, got {self.n_layer}")
        if self.n_loops < 1:
            raise ValueError(f"n_loops must be >= 1, got {self.n_loops}")
        if self.global_every < 1:
            raise ValueError(f"global_every must be >= 1, got {self.global_every}")
        if self.grad_checkpoint_every < 1:
            raise ValueError(f"grad_checkpoint_every must be >= 1, got {self.grad_checkpoint_every}")
        if not 0.0 <= self.layerdrop_max < 1.0:
            raise ValueError(f"layerdrop_max must be in [0, 1), got {self.layerdrop_max}")
        if self.sliding_window < 1:
            raise ValueError(f"sliding_window must be >= 1, got {self.sliding_window}")
        if self.byte_mix_kernel != 0 and self.byte_mix_kernel < 2:
            raise ValueError(f"byte_mix_kernel must be 0 (off) or >= 2, got {self.byte_mix_kernel}")
        if self.norm_eps <= 0:
            raise ValueError(f"norm_eps must be > 0, got {self.norm_eps}")
        if self.gain_init is None:
            self.gain_init = 1.0 / math.sqrt(2.0 * self.n_layer * self.n_loops)

    # ------------------------------------------------------------ constructors
    @classmethod
    def field_names(cls) -> set:
        return {f.name for f in dataclasses.fields(cls)}

    @classmethod
    def from_dict(cls, d: Dict[str, Any], warn_unknown: bool = True) -> "TensegrityConfig":
        """Build from a (possibly older/newer) config dict. Missing keys take defaults
        (== legacy behaviour for fields added later); unknown keys are dropped with a warning
        instead of being silently ignored."""
        known = cls.field_names()
        unknown = sorted(k for k in d if k not in known)
        if unknown and warn_unknown:
            warnings.warn(f"TensegrityConfig.from_dict: ignoring unknown key(s) {unknown}", stacklevel=2)
        return cls(**{k: v for k, v in d.items() if k in known})

    def to_dict(self) -> Dict[str, Any]:
        return dataclasses.asdict(self)

    def replace(self, **overrides: Any) -> "TensegrityConfig":
        """Copy with field overrides. ``gain_init`` is recomputed unless given explicitly."""
        d = self.to_dict()
        if "gain_init" not in overrides and ("n_layer" in overrides or "n_loops" in overrides):
            d["gain_init"] = None
        d.update(overrides)
        return TensegrityConfig(**d)

    # ----------------------------------------------------------------- derived
    @property
    def head_dim(self) -> int:
        return self.n_embd // self.n_head

    @property
    def gqa_ratio(self) -> int:
        return self.n_head // self.n_kv_head

    @property
    def ffn_hidden(self) -> int:
        return ffn_hidden_dim(self.n_embd, self.ffn_mult, self.ffn_align)

    def is_global_layer(self, layer_idx: int) -> bool:
        return (layer_idx + 1) % self.global_every == 0

    def n_params(self) -> int:
        """Exact parameter count (tied embedding counted once). Verified against a real
        instantiation for every preset in tests/test_presets.py."""
        d, hd, h = self.n_embd, self.head_dim, self.ffn_hidden
        q_out, kv_out = self.n_head * hd, self.n_kv_head * hd
        per_block = 2 * d                                   # ln1, ln2
        per_block += d * q_out + 2 * d * kv_out + q_out * d  # q,k,v,o
        per_block += 3 * d * h                              # SwiGLU
        per_block += 2                                      # attn_gain, ffn_gain
        if self.bias:
            per_block += q_out + 2 * kv_out + d + 2 * h + d
        if self.use_qk_norm:
            per_block += 2 * hd
        if self.x0_mix:
            per_block += 1
        if self.n_loops > 1:
            per_block += (self.n_loops - 1) * (2 + (1 if self.x0_mix else 0))
        total = self.vocab_size * d + self.n_layer * per_block + d  # embedding + blocks + ln_f
        if self.v0_mix and self.n_layer > 1:
            total += (self.n_layer - 1) * self.n_kv_head
        if self.use_confidence_head:
            total += d + 1
        if self.byte_mix_kernel:
            n_taps = (self.byte_mix_kernel - 1) * d
            total += n_taps * (2 if self.byte_mix_bilinear else 1)
        return total

    def n_params_non_embedding(self) -> int:
        return self.n_params() - self.vocab_size * self.n_embd

    def flops_per_token(self, seq_len: Optional[int] = None, training: bool = True) -> float:
        """Approximate matmul FLOPs per token (2 per multiply-add; training = 3x forward).
        Includes the lm_head and the attention score/value products at the *effective*
        number of keys each layer attends to (full causal for global layers, the window for local)."""
        T = seq_len or self.block_size
        d, hd, h = self.n_embd, self.head_dim, self.ffn_hidden
        q_out, kv_out = self.n_head * hd, self.n_kv_head * hd
        proj = d * q_out + 2 * d * kv_out + q_out * d + 3 * d * h
        W = self.sliding_window
        keys_global = (T + 1) / 2.0
        keys_local = (T + 1) / 2.0 if T <= W else W - W * (W - 1) / (2.0 * T)
        attn = 0.0
        for i in range(self.n_layer):
            keys = keys_global if self.is_global_layer(i) else keys_local
            attn += 4.0 * keys * q_out        # QK^T and AV, 2 FLOPs per MAC
        fwd = 2.0 * (self.n_loops * self.n_layer * proj + self.vocab_size * d) + self.n_loops * attn
        return fwd * (3.0 if training else 1.0)

    def describe(self) -> str:
        n = self.n_params()
        kinds = "".join("G" if self.is_global_layer(i) else "l" for i in range(self.n_layer))
        loops = f"x{self.n_loops}" if self.n_loops > 1 else ""
        return (f"{n / 1e6:.2f}M params | L={self.n_layer}{loops} d={self.n_embd} "
                f"heads={self.n_head}/{self.n_kv_head}kv (hd={self.head_dim}) ffn={self.ffn_hidden} "
                f"ctx={self.block_size} win={self.sliding_window} layers[{kinds}]")

In [ ]:
%%writefile tensegrity_src/tensegrity/tokenizer.py
"""Byte-level tokenizer: 256 raw bytes + 3 specials. Zero training, zero dependencies."""

from __future__ import annotations

from typing import Any, List, Tuple


class ByteTokenizer:
    """
    Vocabulary = 256 raw byte values + 3 special tokens:
        256 = BOS (beginning of sequence)
        257 = EOS (end of sequence / document separator)
        258 = PAD (padding; also used as left-context filler by the byte-mix layer)

    Losslessness:
      * ``encode_bytes`` / ``decode_bytes`` are lossless for ANY bytes input.
      * ``encode`` / ``decode`` round-trip any well-formed ``str`` (UTF-8 is
        bijective on valid unicode). With the default ``errors="replace"`` they are
        NOT lossless for content strict UTF-8 cannot represent (e.g. lone
        surrogates); pass ``errors="strict"`` to fail loudly instead.
    """

    VOCAB_SIZE = 259
    BOS = 256
    EOS = 257
    PAD = 258
    _SPECIAL_IDS = frozenset((BOS, EOS, PAD))
    _SPECIAL_TOKEN_STRINGS = {BOS: "<|bos|>", EOS: "<|eos|>", PAD: "<|pad|>"}

    def __len__(self) -> int:
        return self.VOCAB_SIZE

    # ---------------------------------------------------------------- str <-> ids
    def encode(self, text: str, errors: str = "replace",
               add_bos: bool = False, add_eos: bool = False) -> List[int]:
        ids = list(text.encode("utf-8", errors=errors))
        if add_bos:
            ids.insert(0, self.BOS)
        if add_eos:
            ids.append(self.EOS)
        return ids

    @staticmethod
    def _as_list(ids: Any) -> list:
        if hasattr(ids, "tolist"):
            return ids.tolist()
        return ids if isinstance(ids, (list, tuple)) else list(ids)

    def _validate(self, ids: list, who: str) -> None:
        bad = [int(i) for i in ids if not (0 <= int(i) < self.VOCAB_SIZE)]
        if bad:
            raise ValueError(f"{who}: invalid token id(s) outside vocab: {bad[:5]}"
                             f"{'...' if len(bad) > 5 else ''}")

    def decode(self, ids: Any, errors: str = "replace",
               skip_special_tokens: bool = True, strict: bool = False) -> str:
        """
        skip_special_tokens: drop ids >= 256 (default). If False, specials render as
            ``<|bos|>`` / ``<|eos|>`` / ``<|pad|>``.
        strict: raise ValueError on any id outside [0, VOCAB_SIZE) instead of dropping it.
        """
        ids = self._as_list(ids)
        if strict:
            self._validate(ids, "decode")
        if skip_special_tokens:
            return bytes(int(i) for i in ids if 0 <= int(i) <= 255).decode("utf-8", errors=errors)

        parts: List[str] = []
        cur = bytearray()
        for raw_id in ids:
            i = int(raw_id)
            if 0 <= i <= 255:
                cur.append(i)
            elif i in self._SPECIAL_TOKEN_STRINGS:
                if cur:
                    parts.append(cur.decode("utf-8", errors=errors))
                    cur.clear()
                parts.append(self._SPECIAL_TOKEN_STRINGS[i])
        if cur:
            parts.append(cur.decode("utf-8", errors=errors))
        return "".join(parts)

    # -------------------------------------------------------------- bytes <-> ids
    def encode_bytes(self, data: bytes, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        if not isinstance(data, (bytes, bytearray)):
            raise TypeError(f"encode_bytes expects bytes/bytearray, got {type(data).__name__}")
        ids = list(data)
        if add_bos:
            ids.insert(0, self.BOS)
        if add_eos:
            ids.append(self.EOS)
        return ids

    def decode_bytes(self, ids: Any, strict: bool = False) -> bytes:
        """Strip specials, return raw bytes (no UTF-8 interpretation): the lossless path."""
        ids = self._as_list(ids)
        if strict:
            self._validate(ids, "decode_bytes")
        return bytes(int(i) for i in ids if 0 <= int(i) <= 255)

    # ---------------------------------------------------------------------- batch
    def encode_batch(self, texts: List[str], **kwargs) -> List[List[int]]:
        return [self.encode(t, **kwargs) for t in texts]

    def decode_batch(self, batches: List[List[int]], **kwargs) -> List[str]:
        return [self.decode(ids, **kwargs) for ids in batches]

    # ------------------------------------------------------------- introspection
    def is_special(self, token_id: int) -> bool:
        return token_id in self._SPECIAL_IDS

    def special_tokens(self) -> Tuple[int, int, int]:
        return (self.BOS, self.EOS, self.PAD)

In [ ]:
%%writefile tensegrity_src/tensegrity/layers.py
"""Building blocks: RMSNorm, RoPE, grouped-query attention (global + banded-local), SwiGLU, Block.

Parameter *names* are identical to the original single-file release for every
legacy feature, so old ``state_dict``s load unchanged.
"""

from __future__ import annotations

from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint  # noqa: F401  (explicit: `import torch` alone does not guarantee it)

from .config import TensegrityConfig

KV = Tuple[torch.Tensor, torch.Tensor]


# =============================================================================
# SDPA capability probing
# =============================================================================
#
# The original code decided whether SDPA supports `enable_gqa` with
# `inspect.signature(F.scaled_dot_product_attention)`. On real torch builds that
# raises ValueError (it is a C builtin with no introspectable signature), so the
# flag was *always* False. We probe by calling the function instead.

def _probe_enable_gqa() -> bool:
    try:
        q = torch.zeros(1, 2, 1, 2)
        kv = torch.zeros(1, 1, 1, 2)
        F.scaled_dot_product_attention(q, kv, kv, enable_gqa=True)
        return True
    except (TypeError, RuntimeError, ValueError):
        return False


SDPA_HAS_ENABLE_GQA: bool = _probe_enable_gqa()

_NATIVE_GQA_CACHE: Dict[Tuple[int, torch.dtype], bool] = {}


def native_gqa_ok(t: torch.Tensor) -> bool:
    """Use SDPA's native ``enable_gqa`` only where it hits the fused FlashAttention kernel.

    PyTorch's memory-efficient kernel (the only fused kernel on a Turing T4, sm75) does
    not implement GQA; with ``enable_gqa=True`` SDPA would silently fall back to the
    *math* backend and materialise the full (T, T) score matrix. So: native GQA only on
    CUDA, compute capability >= 8, fp16/bf16. Everywhere else K/V are expanded instead,
    which keeps the fused kernels engaged.
    """
    if not SDPA_HAS_ENABLE_GQA or not t.is_cuda or t.dtype not in (torch.float16, torch.bfloat16):
        return False
    key = (t.device.index if t.device.index is not None else torch.cuda.current_device(), t.dtype)
    ok = _NATIVE_GQA_CACHE.get(key)
    if ok is None:
        ok = torch.cuda.get_device_capability(t.device)[0] >= 8
        _NATIVE_GQA_CACHE[key] = ok
    return ok


def _expand_kv(k: torch.Tensor, v: torch.Tensor, n_rep: int) -> Tuple[torch.Tensor, torch.Tensor]:
    if n_rep == 1:
        return k, v
    B, Hkv, T, D = k.shape
    return (k[:, :, None, :, :].expand(B, Hkv, n_rep, T, D).reshape(B, Hkv * n_rep, T, D),
            v[:, :, None, :, :].expand(B, Hkv, n_rep, T, D).reshape(B, Hkv * n_rep, T, D))


# =============================================================================
# Normalisation, RoPE, small helpers
# =============================================================================

_HAS_FUSED_RMS_NORM: bool = hasattr(F, "rms_norm")


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if _HAS_FUSED_RMS_NORM:
            w = self.weight if x.dtype == self.weight.dtype else self.weight.to(x.dtype)
            return F.rms_norm(x, self.weight.shape, w, self.eps)
        in_dtype = x.dtype
        xf = x.float()
        rms = torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (xf * rms * self.weight.float()).to(in_dtype)


def precompute_rope(head_dim: int, max_seq_len: int, base: float = 10000.0,
                    device=None, dtype=torch.float32) -> Tuple[torch.Tensor, torch.Tensor]:
    """cos/sin tables of shape (max_seq_len, head_dim // 2)."""
    assert head_dim % 2 == 0
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device, dtype=torch.float32) / head_dim))
    t = torch.arange(max_seq_len, device=device, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)
    return freqs.cos().to(dtype), freqs.sin().to(dtype)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """x: (B, H, T, D). cos/sin: (>=T, D/2) or pre-shaped (1, 1, T, D/2)."""
    B, H, T, D = x.shape
    half = D // 2
    if cos.ndim == 4:
        c, s = cos, sin
    else:
        assert T <= cos.shape[0], (
            f"sequence length {T} exceeds RoPE table length {cos.shape[0]}; "
            f"call precompute_rope with a larger max_seq_len")
        c = cos[:T].float().view(1, 1, T, half)
        s = sin[:T].float().view(1, 1, T, half)
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    return torch.cat([x1 * c - x2 * s, x2 * c + x1 * s], dim=-1).to(x.dtype)


def activation_rms(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Mean over batch/tokens of the per-token feature RMS (eps-stabilised sqrt)."""
    return torch.sqrt(x.float().pow(2).mean(dim=-1) + eps).mean()


def apply_logit_soft_cap(logits: torch.Tensor, soft_cap: float) -> torch.Tensor:
    """Gemma-2 soft-capping: ``cap * tanh(logits / cap)`` (computed in fp32)."""
    return (soft_cap * torch.tanh(logits.float() / soft_cap)).to(logits.dtype)


# =============================================================================
# Attention
# =============================================================================

_BAND_MASK_CACHE: Dict[tuple, torch.Tensor] = {}


def _band_mask(n_chunks: int, window: int, batch: int, device: torch.device) -> torch.Tensor:
    """Boolean mask (batch * n_chunks, 1, W, 2W) for chunked sliding-window attention.

    Chunk c's queries (positions cW + i, i in [0, W)) see a 2W-key strip = [chunk c-1 | chunk c]
    whose key j sits at absolute position (c-1)W + j. A query may attend iff
        key_pos <= q_pos   <=>  j <= i + W
        key_pos >  q_pos - W  <=>  j >  i
    i.e. exactly W keys (including itself) -- the same window as the dense
    ``j <= i and j > i - W`` mask. Chunk 0 has no predecessor, so keys j < W are masked out.
    """
    key = (n_chunks, window, batch, str(device))
    m = _BAND_MASK_CACHE.get(key)
    if m is None:
        i = torch.arange(window, device=device).view(-1, 1)
        j = torch.arange(2 * window, device=device).view(1, -1)
        band = (j > i) & (j <= i + window)                     # (W, 2W)
        masks = band.unsqueeze(0).repeat(n_chunks, 1, 1)       # (nC, W, 2W)
        masks[0] &= (j >= window)
        m = masks.unsqueeze(0).expand(batch, -1, -1, -1).reshape(batch * n_chunks, 1, window, 2 * window)
        if len(_BAND_MASK_CACHE) >= 16:
            _BAND_MASK_CACHE.pop(next(iter(_BAND_MASK_CACHE)))
        _BAND_MASK_CACHE[key] = m
    return m


def local_causal_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, window: int,
                           n_rep: int, dropout_p: float = 0.0) -> torch.Tensor:
    """Sliding-window causal attention in ONE fused SDPA call.

    The sequence is cut into chunks of ``window`` queries; each chunk attends to the
    2W-key strip [previous chunk | itself] under a banded mask (see ``_band_mask``).
    Chunks are folded into the batch dimension, so there is no Python loop over chunks
    and no (T, T) tensor. Result equals dense masked attention exactly.

    q: (B, H, T, D); k, v: (B, Hkv, T, D). Returns (B, H, T, D).
    """
    B, H, T, D = q.shape
    Hkv = k.shape[1]
    W = window
    if T <= W:  # window covers everything -> plain causal attention
        if n_rep > 1 and native_gqa_ok(q):
            return F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True, enable_gqa=True)
        if n_rep > 1:
            k, v = _expand_kv(k, v, n_rep)
        return F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True)

    n_chunks = (T + W - 1) // W
    pad = n_chunks * W - T
    if pad:  # pad at the END: padded keys lie in the future of every real query, so are masked
        q = F.pad(q, (0, 0, 0, pad))
    kp = F.pad(k, (0, 0, W, pad)).view(B, Hkv, n_chunks + 1, W, D)
    vp = F.pad(v, (0, 0, W, pad)).view(B, Hkv, n_chunks + 1, W, D)
    k2 = torch.cat([kp[:, :, :-1], kp[:, :, 1:]], dim=3)       # (B, Hkv, nC, 2W, D)
    v2 = torch.cat([vp[:, :, :-1], vp[:, :, 1:]], dim=3)

    qf = q.view(B, H, n_chunks, W, D).transpose(1, 2).reshape(B * n_chunks, H, W, D)
    kf = k2.transpose(1, 2).reshape(B * n_chunks, Hkv, 2 * W, D)
    vf = v2.transpose(1, 2).reshape(B * n_chunks, Hkv, 2 * W, D)
    if n_rep > 1:
        kf, vf = _expand_kv(kf, vf, n_rep)

    mask = _band_mask(n_chunks, W, B, q.device)
    yf = F.scaled_dot_product_attention(qf, kf, vf, attn_mask=mask, dropout_p=dropout_p)
    y = yf.view(B, n_chunks, H, W, D).transpose(1, 2).reshape(B, H, n_chunks * W, D)
    return y[:, :, :T] if pad else y


class CausalSelfAttention(nn.Module):
    """GQA attention. ``is_global`` layers attend causally to everything; local layers to
    the trailing ``sliding_window`` positions."""

    def __init__(self, cfg: TensegrityConfig, is_global: bool, layer_idx: int = 0):
        super().__init__()
        self.n_head = cfg.n_head
        self.n_kv_head = cfg.n_kv_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.n_rep = cfg.n_head // cfg.n_kv_head
        self.is_global = is_global
        self.sliding_window = cfg.sliding_window
        self.dropout_p = cfg.dropout
        self.q_proj = nn.Linear(cfg.n_embd, cfg.n_head * self.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.n_embd, cfg.n_kv_head * self.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.n_embd, cfg.n_kv_head * self.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * self.head_dim, cfg.n_embd, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        if cfg.use_qk_norm:
            self.q_norm = RMSNorm(self.head_dim)
            self.k_norm = RMSNorm(self.head_dim)
        else:
            self.q_norm = None
            self.k_norm = None
        if cfg.v0_mix and layer_idx > 0:
            self.v0_gain = nn.Parameter(torch.full((cfg.n_kv_head,), 0.5))
            self.v0_gain._no_decay = True  # type: ignore[attr-defined]
        else:
            self.v0_gain = None

    def _project(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                 v0: Optional[torch.Tensor] = None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v_raw = v
        if self.v0_gain is not None and v0 is not None:
            v = v + self.v0_gain.to(v.dtype).view(1, -1, 1, 1) * v0
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        return apply_rope(q, cos, sin), apply_rope(k, cos, sin), v, v_raw

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                past: Optional[KV] = None, return_kv: bool = False,
                v0: Optional[torch.Tensor] = None, return_v0: bool = False):
        """
        Training / prefill (``past is None``): full-sequence attention starting at position 0.
        Decode (``past`` given): ``x`` must be ONE new token; its K/V are appended to ``past``.
        With ``return_kv`` the (trimmed) cache for subsequent decode steps is returned too.
        """
        B, T, _ = x.shape
        q, k, v, v_raw = self._project(x, cos, sin, v0)
        dropout_p = self.dropout_p if self.training else 0.0

        if past is not None:
            assert T == 1, "cached decode supports a single new token per step"
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
            if not self.is_global and k.shape[2] > self.sliding_window:
                k = k[:, :, -self.sliding_window:]
                v = v[:, :, -self.sliding_window:]
            ke, ve = _expand_kv(k, v, self.n_rep)
            y = F.scaled_dot_product_attention(q, ke, ve)  # every cached key is causally visible
        elif self.is_global:
            if self.n_rep > 1 and native_gqa_ok(q):
                y = F.scaled_dot_product_attention(q, k, v, dropout_p=dropout_p, is_causal=True,
                                                   enable_gqa=True)
            else:
                ke, ve = _expand_kv(k, v, self.n_rep)
                y = F.scaled_dot_product_attention(q, ke, ve, dropout_p=dropout_p, is_causal=True)
        else:
            y = local_causal_attention(q, k, v, self.sliding_window, self.n_rep, dropout_p)

        y = y.transpose(1, 2).contiguous().view(B, T, self.n_head * self.head_dim)
        y = self.resid_dropout(self.o_proj(y))
        if not return_kv:
            return (y, v_raw) if return_v0 else y
        if not self.is_global and k.shape[2] > self.sliding_window:
            k, v = k[:, :, -self.sliding_window:], v[:, :, -self.sliding_window:]
        kv = (k.contiguous(), v.contiguous())
        return (y, kv, v_raw) if return_v0 else (y, kv)


# =============================================================================
# Feed-forward and block
# =============================================================================

class SwiGLU(nn.Module):
    def __init__(self, n_embd: int, hidden_dim: int, bias: bool, dropout: float):
        super().__init__()
        self.w_gate = nn.Linear(n_embd, hidden_dim, bias=bias)
        self.w_up = nn.Linear(n_embd, hidden_dim, bias=bias)
        self.w_down = nn.Linear(hidden_dim, n_embd, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


class Block(nn.Module):
    """Pre-norm transformer block with learnable residual gains ("homeostatic" gains).

        x <- x + x0_gain * x0                      (only if cfg.x0_mix)
        x <- x + attn_gain * Attn(RMSNorm(x))
        x <- x + ffn_gain  * SwiGLU(RMSNorm(x))

    LayerDrop is decided by the *model* (one CPU draw per forward), not here: a block
    never touches RNG or the network, so it is safe under gradient checkpointing,
    torch.compile and DDP.
    """

    def __init__(self, cfg: TensegrityConfig, layer_idx: int):
        super().__init__()
        self.layer_idx = layer_idx
        self.ln1 = RMSNorm(cfg.n_embd, cfg.norm_eps)
        self.attn = CausalSelfAttention(cfg, is_global=cfg.is_global_layer(layer_idx), layer_idx=layer_idx)
        self.ln2 = RMSNorm(cfg.n_embd, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.n_embd, cfg.ffn_hidden, cfg.bias, cfg.dropout)
        self.attn_gain = nn.Parameter(torch.tensor(float(cfg.gain_init)))
        self.ffn_gain = nn.Parameter(torch.tensor(float(cfg.gain_init)))
        self.x0_gain = nn.Parameter(torch.zeros(())) if cfg.x0_mix else None
        if cfg.n_loops > 1:
            n_extra = cfg.n_loops - 1
            self.loop_attn_gains = nn.Parameter(torch.full((n_extra,), float(cfg.gain_init)))
            self.loop_ffn_gains = nn.Parameter(torch.full((n_extra,), float(cfg.gain_init)))
            self.loop_x0_gains = nn.Parameter(torch.zeros(n_extra)) if cfg.x0_mix else None
        else:
            self.loop_attn_gains = None
            self.loop_ffn_gains = None
            self.loop_x0_gains = None
        self.homeo_target = cfg.homeo_target
        self.track_homeo = cfg.homeo_weight > 0.0
        # depth-scaled stochastic-depth probability (consumed by TensegrityLM)
        self.drop_p = cfg.layerdrop_max * (layer_idx / (cfg.n_layer - 1)) if cfg.n_layer > 1 else 0.0

    def _homeo(self, sub_out: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
        if self.track_homeo and torch.is_grad_enabled():
            return (activation_rms(sub_out) - self.homeo_target).pow(2)
        return ref.new_zeros((), dtype=torch.float32)

    def _gains(self, loop_idx: int):
        if loop_idx == 0 or self.loop_attn_gains is None:
            return self.attn_gain, self.ffn_gain, self.x0_gain
        k = loop_idx - 1
        xg = self.loop_x0_gains[k] if self.loop_x0_gains is not None else None
        return self.loop_attn_gains[k], self.loop_ffn_gains[k], xg

    def _forward_impl(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                      x0: Optional[torch.Tensor] = None, v0: Optional[torch.Tensor] = None,
                      loop_idx: int = 0, need_v0: bool = False):
        ag, fg, xg = self._gains(loop_idx)
        if xg is not None and x0 is not None:
            x = x + xg * x0
        if need_v0:
            attn_out, v0_out = self.attn(self.ln1(x), cos, sin, v0=v0, return_v0=True)
        else:
            attn_out, v0_out = self.attn(self.ln1(x), cos, sin, v0=v0), v0
        homeo = self._homeo(attn_out, x)
        x = x + ag * attn_out
        ffn_out = self.ffn(self.ln2(x))
        homeo = homeo + self._homeo(ffn_out, x)
        x = x + fg * ffn_out
        return x, homeo, v0_out

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                x0: Optional[torch.Tensor] = None, v0: Optional[torch.Tensor] = None,
                loop_idx: int = 0) -> Tuple[torch.Tensor, torch.Tensor]:
        x, homeo, _ = self._forward_impl(x, cos, sin, x0, v0, loop_idx, need_v0=False)
        return x, homeo

    def forward_incremental(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor,
                            x0: Optional[torch.Tensor], past: Optional[KV],
                            v0: Optional[torch.Tensor] = None, loop_idx: int = 0,
                            return_v0: bool = False):
        """KV-cached forward for inference (eval mode; no LayerDrop, no homeostasis loss)."""
        ag, fg, xg = self._gains(loop_idx)
        if xg is not None and x0 is not None:
            x = x + xg * x0
        if return_v0:
            attn_out, kv, v0_out = self.attn(self.ln1(x), cos, sin, past=past, return_kv=True, v0=v0, return_v0=True)
        else:
            attn_out, kv = self.attn(self.ln1(x), cos, sin, past=past, return_kv=True, v0=v0)
            v0_out = v0
        x = x + ag * attn_out
        x = x + fg * self.ffn(self.ln2(x))
        return (x, kv, v0_out) if return_v0 else (x, kv)


class ByteMix(nn.Module):
    """Learned causal FIR filter (and optional gated bilinear n-gram mixer) over the previous K-1 byte embeddings:

        y_t = e_t + sum_{j=1..K-1} (w_j + bi_j * tanh(e_t / scale)) * e_{t-j}

    A byte is a fragment of a character/word, so the first layers of a byte LM spend
    capacity just assembling local n-grams. This gives every position direct, cheap
    access to its recent bytes and multiplicative byte-pair interactions. Zero-initialised,
    so it starts as the identity and preserves function under width growth.
    """

    def __init__(self, kernel: int, n_embd: int, bilinear: bool = False,
                 init_std: float = 0.02, embed_scale: bool = False):
        super().__init__()
        self.kernel = kernel
        self.init_std = init_std
        self.embed_scale = embed_scale
        self.weight = nn.Parameter(torch.zeros(kernel - 1, n_embd))
        self.weight._no_decay = True  # type: ignore[attr-defined]
        if bilinear:
            self.bi_weight = nn.Parameter(torch.zeros(kernel - 1, n_embd))
            self.bi_weight._no_decay = True  # type: ignore[attr-defined]
        else:
            self.bi_weight = None

    def forward(self, e: torch.Tensor, n_prefix: int) -> torch.Tensor:
        """e: (B, n_prefix + T, C) embeddings whose first ``n_prefix`` (= K-1) rows are
        left context. Returns (B, T, C)."""
        T = e.shape[1] - n_prefix
        cur = e[:, n_prefix:]
        out = cur
        w = self.weight if e.dtype == self.weight.dtype else self.weight.to(e.dtype)
        if self.bi_weight is not None:
            bw = self.bi_weight if e.dtype == self.bi_weight.dtype else self.bi_weight.to(e.dtype)
            scale = self.init_std * (e.shape[-1] ** 0.5 if self.embed_scale else 1.0)
            g = torch.tanh(cur * (1.0 / scale))
            for j in range(1, self.kernel):
                prev = e[:, n_prefix - j: n_prefix - j + T]
                out = out + (w[j - 1] + bw[j - 1] * g) * prev
            return out
        for j in range(1, self.kernel):
            out = out + w[j - 1] * e[:, n_prefix - j: n_prefix - j + T]
        return out

In [ ]:
%%writefile tensegrity_src/tensegrity/model.py
"""TensegrityLM: a byte-level decoder-only language model.

Mechanisms (each is established in the literature; the *combination* is a
coherent but unvalidated recombination -- measure before claiming it beats a
vanilla transformer of equal size):

  * RoPE positions                        * RMSNorm pre-norm, optional Q/K RMSNorm
  * Grouped-query attention               * alternating local (banded) / global attention
  * SwiGLU feed-forward                   * learnable per-layer residual gains
  * activation-RMS "homeostasis" loss     * depth-scaled LayerDrop
  * z-loss, logit soft-capping            * optional confidence head (adaptive sampling temperature)
  * optional ByteMix (causal FIR over recent byte embeddings) and x0-mix (embedding re-injection)
"""

from __future__ import annotations

from typing import Dict, Iterator, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

from .config import TensegrityConfig
from .layers import (Block, ByteMix, KV, RMSNorm, apply_logit_soft_cap, precompute_rope)
from .tokenizer import ByteTokenizer


class TensegrityLM(nn.Module):
    def __init__(self, cfg: TensegrityConfig, verbose: bool = False):
        super().__init__()
        self.cfg = cfg
        self.grad_checkpoint = cfg.grad_checkpoint
        self.grad_checkpoint_every = cfg.grad_checkpoint_every

        self.token_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.byte_mix = None
        if cfg.byte_mix_kernel:
            if cfg.vocab_size <= ByteTokenizer.PAD:
                raise ValueError("byte_mix_kernel needs vocab_size > PAD id (258)")
            self.byte_mix = ByteMix(cfg.byte_mix_kernel, cfg.n_embd,
                                    bilinear=cfg.byte_mix_bilinear, init_std=cfg.init_std,
                                    embed_scale=cfg.embed_scale)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.n_layer)])
        self.ln_f = RMSNorm(cfg.n_embd, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.token_emb.weight = self.lm_head.weight  # weight tying

        self.confidence_head = nn.Linear(cfg.n_embd, 1, bias=True) if cfg.use_confidence_head else None

        # RoPE tables live outside the module state (not buffers): they stay fp32 under
        # `.half()` and never appear in a state_dict. Built lazily per device.
        self._rope: Dict[str, Tuple[torch.Tensor, torch.Tensor]] = {}

        # LayerDrop draws come from a private CPU generator: no GPU sync, no collective,
        # and identical on every DDP rank as long as every rank seeds it identically.
        self._ld_gen = torch.Generator(device="cpu")
        self._ld_gen.manual_seed(0)
        self._drop_ps: List[float] = [b.drop_p for b in self.blocks]

        self.apply(self._init_weights)
        if verbose:
            print(f"[TensegrityLM] {cfg.describe()}")

    # ------------------------------------------------------------------ setup
    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=self.cfg.init_std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=self.cfg.init_std)

    def num_parameters(self, non_embedding: bool = False) -> int:
        n = sum(p.numel() for p in self.parameters())
        return n - self.token_emb.weight.numel() if non_embedding else n

    def summary(self) -> str:
        return f"{self.cfg.describe()} | counted {self.num_parameters():,} params"

    def _rope_tables(self, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
        key = str(device)
        tabs = self._rope.get(key)
        if tabs is None:
            tabs = precompute_rope(self.cfg.head_dim, self.cfg.block_size, self.cfg.rope_base, device=device)
            self._rope[key] = tabs
        return tabs

    # ------------------------------------------------------------- layerdrop
    def seed_layerdrop(self, seed: int) -> None:
        self._ld_gen.manual_seed(int(seed))

    def layerdrop_state(self) -> torch.Tensor:
        return self._ld_gen.get_state()

    def set_layerdrop_state(self, state: torch.Tensor) -> None:
        self._ld_gen.set_state(state.cpu())

    def _layerdrop_plan(self) -> List[bool]:
        n = len(self.blocks)
        if not self.training or not any(p > 0.0 for p in self._drop_ps):
            return [False] * n
        u = torch.rand(n, generator=self._ld_gen).tolist()
        return [u[i] < self._drop_ps[i] for i in range(n)]

    # --------------------------------------------------------------- embedding
    def _embed(self, idx: torch.Tensor, ctx: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Token ids -> embedding stream x0, shape (B, T, C).

        ``ctx`` holds the (up to K-1) tokens that immediately precede ``idx`` (used only by
        ByteMix during cached decoding); missing left context is filled with PAD, which is
        exactly what training sees at the start of every window.
        """
        cfg = self.cfg
        if self.byte_mix is None:
            x = self.token_emb(idx)
            if cfg.embed_scale:
                x = x * (cfg.n_embd ** 0.5)
            return self.drop(x)

        n_prefix = cfg.byte_mix_kernel - 1
        B = idx.size(0)
        if ctx is None or ctx.numel() == 0:
            prefix = idx.new_full((B, n_prefix), ByteTokenizer.PAD)
        else:
            prefix = ctx[:, -n_prefix:]
            short = n_prefix - prefix.size(1)
            if short > 0:
                prefix = torch.cat([idx.new_full((B, short), ByteTokenizer.PAD), prefix], dim=1)
        e = self.token_emb(torch.cat([prefix, idx], dim=1))
        if cfg.embed_scale:
            e = e * (cfg.n_embd ** 0.5)
        return self.drop(self.byte_mix(e, n_prefix))

    def _head(self, x: torch.Tensor) -> torch.Tensor:
        logits = self.lm_head(self.ln_f(x))
        if self.cfg.logit_soft_cap is not None:
            logits = apply_logit_soft_cap(logits, self.cfg.logit_soft_cap)
        return logits

    # ----------------------------------------------------------------- forward
    def forward(self, idx: torch.Tensor, targets: Optional[torch.Tensor] = None
                ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]], Optional[torch.Tensor]]:
        """
        idx:     (B, T) token ids, T <= block_size
        targets: (B, T) next-token ids (-1 = ignore) or None

        Returns ``(logits, losses, confidence)``:
          * with targets: ``losses`` is a dict (ce_loss, homeo_loss, total_loss, [z_loss],
            [confidence_loss]) and ``confidence`` is None;
          * without: ``losses`` is None and ``confidence`` is (B, T) in [0, 1] (or None).
        """
        B, T = idx.shape
        if T > self.cfg.block_size:
            raise ValueError(f"sequence length {T} exceeds block_size {self.cfg.block_size}")

        x0 = self._embed(idx)
        x = x0
        x0_arg = x0 if self.cfg.x0_mix else None
        cos_all, sin_all = self._rope_tables(idx.device)
        cos = cos_all[:T].view(1, 1, T, -1)
        sin = sin_all[:T].view(1, 1, T, -1)

        plan = self._layerdrop_plan()
        use_ckpt = self.grad_checkpoint and self.training and x.requires_grad
        total_homeo = x.new_zeros((), dtype=torch.float32)
        n_active = 0
        v0: Optional[torch.Tensor] = None
        need_v0_flag = self.cfg.v0_mix
        for loop_idx in range(self.cfg.n_loops):
            for i, block in enumerate(self.blocks):
                if plan[i]:
                    continue
                need_v0 = need_v0_flag and v0 is None
                if use_ckpt and i % self.grad_checkpoint_every == 0:
                    x, homeo, v0 = torch.utils.checkpoint.checkpoint(
                        block._forward_impl, x, cos, sin, x0_arg, v0, loop_idx, need_v0, use_reentrant=False)
                else:
                    x, homeo, v0 = block._forward_impl(x, cos, sin, x0_arg, v0, loop_idx, need_v0)
                total_homeo = total_homeo + homeo
                n_active += 1

        x = self.ln_f(x)
        logits = self.lm_head(x)
        if self.cfg.logit_soft_cap is not None:
            logits = apply_logit_soft_cap(logits, self.cfg.logit_soft_cap)

        if targets is None:
            confidence = None
            if self.confidence_head is not None:
                confidence = torch.sigmoid(self.confidence_head(x).squeeze(-1))
            return logits, None, confidence

        cfg = self.cfg
        logits_f = logits.float()
        ce_loss = F.cross_entropy(logits_f.view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-1)
        avg_homeo = total_homeo / max(1, n_active * 2)
        total = ce_loss + cfg.homeo_weight * avg_homeo
        losses: Dict[str, torch.Tensor] = {"ce_loss": ce_loss, "homeo_loss": avg_homeo}

        if cfg.z_loss_weight > 0.0:
            z = cfg.z_loss_weight * torch.logsumexp(logits_f, dim=-1).pow(2).mean()
            losses["z_loss"] = z
            total = total + z

        if self.confidence_head is not None:
            h = x.detach() if cfg.confidence_detach else x
            conf_logits = self.confidence_head(h).squeeze(-1)
            with torch.no_grad():
                correct = (logits.argmax(dim=-1) == targets).float()
                valid = (targets != -1).float()
            raw = F.binary_cross_entropy_with_logits(conf_logits.float(), correct, reduction="none")
            conf_loss = (raw * valid).sum() / valid.sum().clamp_min(1.0)
            losses["confidence_loss"] = conf_loss
            total = total + cfg.confidence_weight * conf_loss

        losses["total_loss"] = total
        return logits, losses, None

    # ------------------------------------------------------ cached incremental
    def forward_incremental(self, idx: torch.Tensor,
                            past_kv_list: Optional[List[Optional[KV]]] = None,
                            pos_start: int = 0,
                            ctx: Optional[torch.Tensor] = None,
                            ) -> Tuple[torch.Tensor, List[KV], Optional[torch.Tensor]]:
        """KV-cached forward for inference.

        * prefill: ``past_kv_list=None``, ``idx`` is a whole window starting at position 0.
        * decode:  ``past_kv_list`` from the previous call, ``idx`` is ONE new token at
          absolute window position ``pos_start``; ``ctx`` = the K-1 tokens before it
          (needed only when ``byte_mix_kernel`` is set).

        Returns (logits, new_past_kv_list, confidence) for the T new positions.
        """
        B, T = idx.shape
        if pos_start + T > self.cfg.block_size:
            raise ValueError(f"positions [{pos_start}, {pos_start + T}) exceed block_size {self.cfg.block_size}")
        n_total = len(self.blocks) * self.cfg.n_loops
        if past_kv_list is None:
            past_kv_list = [None] * n_total

        x0 = self._embed(idx, ctx)
        x = x0
        x0_arg = x0 if self.cfg.x0_mix else None
        cos_all, sin_all = self._rope_tables(idx.device)
        cos = cos_all[pos_start:pos_start + T].view(1, 1, T, -1)
        sin = sin_all[pos_start:pos_start + T].view(1, 1, T, -1)

        new_past: List[KV] = []
        v0: Optional[torch.Tensor] = None
        need_v0_flag = self.cfg.v0_mix
        kv_idx = 0
        for loop_idx in range(self.cfg.n_loops):
            for block in self.blocks:
                past = past_kv_list[kv_idx]
                kv_idx += 1
                if need_v0_flag and v0 is None:
                    x, kv, v0 = block.forward_incremental(x, cos, sin, x0_arg, past, v0=None, loop_idx=loop_idx, return_v0=True)
                else:
                    x, kv = block.forward_incremental(x, cos, sin, x0_arg, past, v0=v0, loop_idx=loop_idx, return_v0=False)
                new_past.append(kv)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        if self.cfg.logit_soft_cap is not None:
            logits = apply_logit_soft_cap(logits, self.cfg.logit_soft_cap)
        confidence = None
        if self.confidence_head is not None:
            confidence = torch.sigmoid(self.confidence_head(x).squeeze(-1))
        return logits, new_past, confidence

    # --------------------------------------------------------------- sampling
    @staticmethod
    def _sample_next(logits: torch.Tensor, temperature: float, top_k: Optional[int],
                     top_p: Optional[float], min_p: Optional[float],
                     conf: Optional[torch.Tensor], conf_temp_min: float, conf_temp_max: float
                     ) -> torch.Tensor:
        """logits: (B, V) -> next token ids (B, 1)."""
        logits = logits.float()
        if temperature == 0.0:  # an explicit greedy request always wins over confidence scaling
            return logits.argmax(dim=-1, keepdim=True)

        if conf is not None:
            # low confidence -> hotter, high confidence -> cooler (per sequence)
            scale = conf_temp_max - conf.float() * (conf_temp_max - conf_temp_min)
            logits = logits / (temperature * scale).clamp_min(1e-4).unsqueeze(-1)
        else:
            logits = logits / max(temperature, 1e-4)
        fallback = logits.argmax(dim=-1, keepdim=True)

        if top_k is not None:
            kth = torch.topk(logits, min(top_k, logits.size(-1))).values[:, -1:]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        if top_p is not None:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
            cum = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            cut = cum > top_p
            cut[:, 1:] = cut[:, :-1].clone()  # keep the first token that crosses top_p
            cut[:, 0] = False
            logits = torch.full_like(logits, float("-inf")).scatter(
                1, sorted_idx, sorted_logits.masked_fill(cut, float("-inf")))
        if min_p is not None:
            p = F.softmax(logits, dim=-1)
            logits = logits.masked_fill(p < min_p * p.max(dim=-1, keepdim=True).values, float("-inf"))

        probs = F.softmax(logits, dim=-1)
        bad = ~torch.isfinite(probs).all(dim=-1) | (probs.sum(dim=-1) <= 0.0)
        if bad.any():  # degenerate row (all -inf / NaN): fall back to greedy for that row only
            probs = torch.where(bad.unsqueeze(-1), torch.ones_like(probs), torch.nan_to_num(probs, nan=0.0))
            sampled = torch.multinomial(probs, 1)
            return torch.where(bad.unsqueeze(-1), fallback, sampled)
        return torch.multinomial(probs, 1)

    def _generate_iter(self, idx: torch.Tensor, max_new_tokens: int, temperature: float,
                       top_k: Optional[int], top_p: Optional[float], min_p: Optional[float],
                       use_confidence_temp: bool, conf_temp_min: float, conf_temp_max: float,
                       stop_on_eos: bool, use_cache: bool, rebuild_stride: int
                       ) -> Iterator[Tuple[torch.Tensor, Optional[torch.Tensor]]]:
        """Yield ``(next_tok (B,1), confidence (B,) | None)`` for each generated position."""
        if temperature < 0.0:
            raise ValueError(f"temperature must be >= 0, got {temperature}")
        if top_k is not None and top_k <= 0:
            raise ValueError(f"top_k must be None or positive, got {top_k}")
        if idx.dim() != 2 or idx.size(1) < 1:
            raise ValueError("idx must be (B, T) with T >= 1")
        bs = self.cfg.block_size
        if not 1 <= rebuild_stride <= bs:
            raise ValueError(f"rebuild_stride must be in [1, block_size={bs}], got {rebuild_stride}")
        K = self.cfg.byte_mix_kernel

        was_training = self.training
        self.eval()
        try:
            B = idx.size(0)
            finished = torch.zeros(B, dtype=torch.bool, device=idx.device) if stop_on_eos else None
            window = idx[:, -bs:]  # the tokens the next prediction is conditioned on
            past: Optional[List[KV]] = None
            for _ in range(max_new_tokens):
                if window.size(1) > bs:
                    # Context overflow: re-prefill from a trimmed window. stride=1 keeps exactly the
                    # trailing block_size tokens every step (== non-cached semantics, O(T^2)/step);
                    # a larger stride re-prefills only every `stride` tokens (amortised).
                    window = window[:, -(bs - rebuild_stride + 1):]
                    past = None
                n = window.size(1)
                if use_cache:
                    if past is None:
                        logits, past, conf = self.forward_incremental(window, None, 0)
                    else:
                        ctx = window[:, max(0, n - K): n - 1] if K and n > 1 else None
                        logits, past, conf = self.forward_incremental(window[:, -1:], past, n - 1, ctx)
                else:
                    logits, _, conf = self.forward(window)
                logits = logits[:, -1, :]
                conf = conf[:, -1] if conf is not None else None

                next_tok = self._sample_next(
                    logits, temperature, top_k, top_p, min_p,
                    conf if (use_confidence_temp and conf is not None) else None,
                    conf_temp_min, conf_temp_max)
                if stop_on_eos:
                    # freeze finished rows instead of letting them drift while others finish
                    next_tok = torch.where(finished.unsqueeze(-1),
                                           torch.full_like(next_tok, ByteTokenizer.EOS), next_tok)
                    finished = finished | (next_tok.squeeze(-1) == ByteTokenizer.EOS)
                window = torch.cat([window, next_tok], dim=1)
                yield next_tok, conf
                if stop_on_eos and bool(finished.all()):
                    break
        finally:
            self.train(was_training)

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0,
                 top_k: Optional[int] = None, top_p: Optional[float] = None,
                 min_p: Optional[float] = None, use_confidence_temp: bool = False,
                 conf_temp_min: float = 0.5, conf_temp_max: float = 1.5,
                 stop_on_eos: bool = True, use_cache: bool = True, rebuild_stride: int = 1,
                 return_confidence: bool = False):
        """Autoregressive sampling.

        Returns ``(B, T + new)`` ids (shorter if every row hit EOS; finished rows are padded
        with EOS). With ``return_confidence=True`` returns ``(ids, conf)`` where ``conf`` is
        (B, new) of confidence-head scores (None if the model has no head).

        use_cache: reuse a per-layer KV cache (bit-identical to ``use_cache=False`` -- see
            tests). ``rebuild_stride`` only matters once the running sequence exceeds
            ``block_size``: 1 reproduces the non-cached "last block_size tokens" semantics
            exactly; a larger value amortises the re-prefill (slightly shorter effective
            context right after each rebuild).
        """
        toks, confs = [idx], []
        for next_tok, conf in self._generate_iter(
                idx, max_new_tokens, temperature, top_k, top_p, min_p, use_confidence_temp,
                conf_temp_min, conf_temp_max, stop_on_eos, use_cache, rebuild_stride):
            toks.append(next_tok)
            confs.append(conf)
        out = torch.cat(toks, dim=1)
        if not return_confidence:
            return out
        conf_out = torch.stack(confs, dim=1) if confs and confs[0] is not None else None
        return out, conf_out

    @torch.no_grad()
    def stream(self, idx: torch.Tensor, max_new_tokens: int, **kwargs) -> Iterator[Tuple[int, Optional[float]]]:
        """Yield ``(token_id, confidence)`` one byte at a time for a single sequence (B == 1).
        Accepts the sampling kwargs of :meth:`generate`."""
        if idx.size(0) != 1:
            raise ValueError("stream() supports a single sequence (batch size 1)")
        kw = dict(temperature=1.0, top_k=None, top_p=None, min_p=None, use_confidence_temp=False,
                  conf_temp_min=0.5, conf_temp_max=1.5, stop_on_eos=True, use_cache=True, rebuild_stride=1)
        unknown = set(kwargs) - set(kw)
        if unknown:
            raise TypeError(f"stream() got unexpected keyword argument(s) {sorted(unknown)}")
        kw.update(kwargs)
        for next_tok, conf in self._generate_iter(idx, max_new_tokens, **kw):
            tid = int(next_tok[0, 0])
            yield tid, (float(conf[0]) if conf is not None else None)
            if kw["stop_on_eos"] and tid == ByteTokenizer.EOS:
                return

    # --------------------------------------------------------------- optimizer
    def configure_optimizer(self, weight_decay: float, learning_rate: float,
                            betas: Tuple[float, float], device_type: str, **kwargs) -> torch.optim.Optimizer:
        from .optim import configure_optimizer
        return configure_optimizer(self, weight_decay, learning_rate, betas, device_type, **kwargs)

In [ ]:
%%writefile tensegrity_src/tensegrity/optim.py
"""Optimizers: AdamW (fused when available) and a single-device Muon + auxiliary AdamW.

Muon is vendored from https://github.com/KellerJordan/Muon (SingleDeviceMuonWithAuxAdam),
the reference implementation behind the modded-nanogpt speedrun. The Newton-Schulz
coefficients and the momentum / scale formulas are reproduced verbatim -- they are exactly
the numbers those results are evidence for, not values to casually retune.

Deviations from upstream (numerics only, not the algorithm):
  * Newton-Schulz runs in bf16 only on hardware with *native* bf16 (compute capability >= 8).
    On a Turing T4 it runs in fp16 (tensor-core fast); on CPU in fp32. The original release
    used `torch.cuda.is_bf16_supported()`, which is True on a T4 through emulation.
  * The Frobenius normalisation is done in fp32 before down-casting, so tiny gradients cannot
    underflow to a zero norm in fp16.
  * The gradient is not modified in place.
"""

from __future__ import annotations

import inspect
from typing import Any, Dict, List, Tuple

import torch
import torch.nn as nn


# =============================================================================
# Muon
# =============================================================================

def _ns_dtype(device: torch.device) -> torch.dtype:
    if device.type == "cuda":
        return torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16
    return torch.float32


def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5) -> torch.Tensor:
    """Quintic Newton-Schulz iteration: approximate orthogonalisation of G (all singular
    values pushed to ~U(0.5, 1.5)), the cheap stand-in for the SVD-based polar factor."""
    assert G.ndim >= 2
    a, b, c = (3.4445, -4.7750, 2.0315)
    X = G.float()
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    X = X.to(_ns_dtype(G.device))
    transposed = G.size(-2) > G.size(-1)
    if transposed:
        X = X.mT
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A
        X = a * X + B @ X
    if transposed:
        X = X.mT
    return X.to(G.dtype)


def muon_update(grad: torch.Tensor, momentum: torch.Tensor, beta: float = 0.95,
                ns_steps: int = 5, nesterov: bool = True) -> torch.Tensor:
    momentum.lerp_(grad, 1 - beta)
    update = torch.lerp(grad, momentum, beta) if nesterov else momentum
    if update.ndim == 4:  # conv filters; unused by this model, kept for fidelity to upstream
        update = update.view(len(update), -1)
    update = zeropower_via_newtonschulz5(update, steps=ns_steps)
    return update * max(1.0, update.size(-2) / update.size(-1)) ** 0.5


def adam_update(grad: torch.Tensor, buf1: torch.Tensor, buf2: torch.Tensor, step: int,
                betas: Tuple[float, float], eps: float) -> torch.Tensor:
    buf1.lerp_(grad, 1 - betas[0])
    buf2.lerp_(grad.square(), 1 - betas[1])
    return (buf1 / (1 - betas[0] ** step)) / ((buf2 / (1 - betas[1] ** step)).sqrt() + eps)


class SingleDeviceMuonWithAuxAdam(torch.optim.Optimizer):
    """One optimizer object: Muon on groups with ``use_muon=True``, AdamW on the rest."""

    def __init__(self, param_groups: List[Dict[str, Any]]):
        for g in param_groups:
            if "use_muon" not in g:
                raise ValueError("every param group needs a 'use_muon' flag")
            if g["use_muon"]:
                g.setdefault("lr", 0.02)
                g.setdefault("momentum", 0.95)
                g.setdefault("weight_decay", 0.0)
            else:
                g.setdefault("lr", 3e-4)
                g.setdefault("betas", (0.9, 0.95))
                g.setdefault("eps", 1e-10)
                g.setdefault("weight_decay", 0.0)
        super().__init__(param_groups, dict())

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            if group["use_muon"]:
                lr, beta, wd = group["lr"], group["momentum"], group["weight_decay"]
                buckets: Dict[Tuple[Any, ...], List[nn.Parameter]] = {}
                for p in group["params"]:
                    if p.grad is None:
                        continue
                    state = self.state[p]
                    if len(state) == 0:
                        state["momentum_buffer"] = torch.zeros_like(p)
                    if p.ndim == 2:
                        buckets.setdefault((p.shape, p.device, p.dtype), []).append(p)
                    else:
                        update = muon_update(p.grad, state["momentum_buffer"], beta=beta)
                        if wd != 0.0:
                            p.mul_(1 - lr * wd)
                        p.add_(update.reshape(p.shape), alpha=-lr)
                for (shape, _, _), params in buckets.items():
                    if len(params) == 1:
                        p = params[0]
                        update = muon_update(p.grad, self.state[p]["momentum_buffer"], beta=beta)
                        if wd != 0.0:
                            p.mul_(1 - lr * wd)
                        p.add_(update.reshape(shape), alpha=-lr)
                    else:
                        grads = [p.grad for p in params]
                        bufs = [self.state[p]["momentum_buffer"] for p in params]
                        torch._foreach_lerp_(bufs, grads, 1.0 - beta)
                        stacked = torch.stack([torch.lerp(g, m, beta) for g, m in zip(grads, bufs)], dim=0)
                        orth = zeropower_via_newtonschulz5(stacked, steps=5)
                        scale = max(1.0, shape[-2] / shape[-1]) ** 0.5
                        if wd != 0.0:
                            torch._foreach_mul_(params, 1.0 - lr * wd)
                        torch._foreach_add_(params, list(orth.unbind(0)), alpha=-lr * scale)
            else:
                for p in group["params"]:
                    if p.grad is None:
                        continue
                    state = self.state[p]
                    if len(state) == 0:
                        state["exp_avg"] = torch.zeros_like(p)
                        state["exp_avg_sq"] = torch.zeros_like(p)
                        state["step"] = 0
                    state["step"] += 1
                    update = adam_update(p.grad, state["exp_avg"], state["exp_avg_sq"],
                                         state["step"], group["betas"], group["eps"])
                    if group["weight_decay"] != 0.0:
                        p.mul_(1 - group["lr"] * group["weight_decay"])
                    p.add_(update, alpha=-group["lr"])
        return loss


# =============================================================================
# Parameter partitioning + factory
# =============================================================================

def _is_no_decay(p: nn.Parameter) -> bool:
    return p.dim() < 2 or getattr(p, "_no_decay", False)


def configure_optimizer(model: nn.Module, weight_decay: float, learning_rate: float,
                        betas: Tuple[float, float], device_type: str,
                        use_8bit: bool = False, use_muon: bool = False,
                        muon_lr: float = 0.02, muon_momentum: float = 0.95,
                        muon_weight_decay: float = None,
                        scalar_lr_mult: float = 3.0) -> torch.optim.Optimizer:
    """
    AdamW (default): >=2-D weights are decayed; norms, scalars (residual gains, x0/v0 gains),
    biases and ByteMix taps are not.

    use_muon=True: every >=2-D weight inside ``model.blocks`` (attention + SwiGLU matrices) is
    trained with Muon; everything else -- the tied token embedding / lm_head, the confidence
    head, ByteMix, and every 1-D / scalar parameter -- stays on AdamW. This is Muon's own
    guidance (hidden matrices only). The walk covers the whole model twice (blocks, then the
    remainder), so a parameter added to the model later can never silently fall through.

    muon_weight_decay: decoupled decay for the Muon group (default: ``weight_decay``).
    scalar_lr_mult: multiplier applied to ``learning_rate`` for the non-decayed auxiliary
        group (norms, residual gains, ByteMix taps) when ``use_muon=True``, so O(1)-scale
        gates/taps keep pace with the orthogonalised Muon steps.
    """
    seen = set()

    def take(params):
        out = []
        for p in params:
            if p.requires_grad and id(p) not in seen:
                seen.add(id(p))
                out.append(p)
        return out

    if use_muon:
        blocks = getattr(model, "blocks", [])
        muon_params = take(p for p in blocks.parameters() if p.dim() >= 2)
        decay = take(p for p in model.parameters() if not _is_no_decay(p))
        no_decay = take(model.parameters())
        mwd = weight_decay if muon_weight_decay is None else muon_weight_decay
        nd_lr = min(muon_lr, learning_rate * scalar_lr_mult)
        groups = [
            dict(params=muon_params, lr=muon_lr, momentum=muon_momentum, weight_decay=mwd, use_muon=True),
            dict(params=decay, lr=learning_rate, betas=betas, eps=1e-10, weight_decay=weight_decay, use_muon=False),
            dict(params=no_decay, lr=nd_lr, betas=betas, eps=1e-10, weight_decay=0.0, use_muon=False),
        ]
        return SingleDeviceMuonWithAuxAdam([g for g in groups if g["params"]])

    decay = take(p for p in model.parameters() if not _is_no_decay(p))
    no_decay = take(model.parameters())
    groups = [{"params": decay, "weight_decay": weight_decay},
              {"params": no_decay, "weight_decay": 0.0}]
    groups = [g for g in groups if g["params"]]

    if use_8bit and device_type == "cuda" and torch.cuda.is_available():
        try:
            import bitsandbytes as bnb
            return bnb.optim.AdamW8bit(groups, lr=learning_rate, betas=betas)
        except ImportError:
            pass

    extra: Dict[str, Any] = {}
    if device_type == "cuda" and torch.cuda.is_available():
        try:
            if "fused" in inspect.signature(torch.optim.AdamW).parameters:
                extra["fused"] = True
        except (TypeError, ValueError):
            pass
    return torch.optim.AdamW(groups, lr=learning_rate, betas=betas, **extra)

In [ ]:
%%writefile tensegrity_src/tensegrity/data.py
"""Data pipeline: streaming ``prepare``, deterministic/resumable batching, dataset downloads.

On-disk format (unchanged from v1): ``train.bin`` / ``val.bin`` are flat ``uint16`` token
streams (bytes 0-255, EOS = 257 between documents) plus ``meta.json``. Existing prepared
directories keep working.
"""

from __future__ import annotations

import glob
import json
import os
import random
import re
import socket
import time
import urllib.error
import urllib.request
from typing import Any, Dict, Iterator, List, Optional, Sequence, Tuple

import numpy as np
import torch

from .tokenizer import ByteTokenizer

SUPPORTED_EXTS = {".txt", ".md", ".parquet", ".jsonl"}
_TEXT_KEYS = ("text", "content", "body", "story")
_EOS = ByteTokenizer.EOS


# =============================================================================
# Streaming encoder: raw bytes -> uint16 tokens, optional document separator -> EOS
# =============================================================================

def _partial_suffix_len(tail: bytes, sep: bytes) -> int:
    """Length of the longest proper prefix of ``sep`` that ``tail`` ends with."""
    for k in range(min(len(sep) - 1, len(tail)), 0, -1):
        if tail.endswith(sep[:k]):
            return k
    return 0


class _StreamEncoder:
    """Feeds byte chunks in, emits uint16 token arrays. A literal ``sep`` (e.g. TinyStories'
    ``<|endoftext|>``) becomes ONE real EOS token instead of 13 ordinary bytes the model
    would have to learn to spell. A separator split across chunk boundaries is handled."""

    def __init__(self, sink, sep: Optional[bytes] = None):
        self.sink = sink
        self.sep = sep or None
        self._rx = re.compile(re.escape(self.sep)) if self.sep else None
        self.carry = b""
        self.n_tokens = 0
        self.n_docs = 0

    def _emit(self, arr: np.ndarray) -> None:
        if arr.size:
            self.n_tokens += int(arr.size)
            self.sink(arr)

    def _encode(self, data: bytes) -> None:
        if not data:
            return
        arr = np.frombuffer(data, dtype=np.uint8).astype(np.uint16)
        if self._rx is None:
            self._emit(arr)
            return
        pos = np.fromiter((m.start() for m in self._rx.finditer(data)), dtype=np.int64)
        if pos.size:
            arr[pos] = _EOS
            if len(self.sep) > 1:
                keep = np.ones(arr.size, dtype=bool)
                for off in range(1, len(self.sep)):
                    keep[pos + off] = False
                arr = arr[keep]
            self.n_docs += int(pos.size)
        self._emit(arr)

    def feed(self, raw: bytes) -> None:
        data = self.carry + raw
        self.carry = b""
        if self._rx is not None:
            last_end = 0
            for m in self._rx.finditer(data):
                last_end = m.end()
            hold = _partial_suffix_len(data[last_end:], self.sep)
            if hold:
                self.carry, data = data[-hold:], data[:-hold]
        self._encode(data)

    def end_document(self) -> None:
        """Flush any held partial separator as ordinary bytes, then emit one EOS."""
        if self.carry:
            tail, self.carry = self.carry, b""
            self._encode(tail)
        self._emit(np.array([_EOS], dtype=np.uint16))
        self.n_docs += 1


# =============================================================================
# File readers
# =============================================================================

def _iter_jsonl_docs(path: str) -> Iterator[bytes]:
    bad = total = 0
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            total += 1
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                bad += 1
                continue
            val = None
            if isinstance(obj, dict):
                for k in _TEXT_KEYS:
                    if obj.get(k):
                        val = obj[k]
                        break
                if val is None:
                    val = "\n".join(str(v) for v in obj.values() if isinstance(v, (str, int, float)))
            elif isinstance(obj, str):
                val = obj
            if val:
                yield str(val).encode("utf-8", errors="replace")
    if total and bad / total > 0.01:
        print(f"[prepare] warning: {bad}/{total} unparseable lines in {path}")


def _pick_text_column(names: Sequence[str], string_cols: Sequence[str], requested: str, path: str) -> str:
    if requested != "auto":
        if requested not in names:
            raise ValueError(f"{path}: column {requested!r} not found; available: {list(names)}")
        return requested
    for k in _TEXT_KEYS:
        if k in string_cols:
            return k
    if string_cols:
        return string_cols[0]
    raise ValueError(f"{path}: no string column to use as text; columns: {list(names)}")


def _iter_parquet_docs(path: str, text_column: str) -> Iterator[bytes]:
    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
    except ImportError as e:  # pragma: no cover - environment dependent
        raise ImportError("reading .parquet needs `pip install pyarrow`") from e
    pf = pq.ParquetFile(path)
    schema = pf.schema_arrow
    string_cols = [f.name for f in schema if pa.types.is_string(f.type) or pa.types.is_large_string(f.type)]
    col = _pick_text_column(schema.names, string_cols, text_column, path)
    for batch in pf.iter_batches(columns=[col], batch_size=4096):
        for s in batch.column(0).to_pylist():
            if s:
                yield str(s).encode("utf-8", errors="replace")


def _iter_text_chunks(path: str, chunk_bytes: int) -> Iterator[bytes]:
    with open(path, "rb") as fh:
        while True:
            chunk = fh.read(chunk_bytes)
            if not chunk:
                return
            yield chunk


def list_source_files(data_path: str) -> List[str]:
    if os.path.isdir(data_path):
        allf = sorted(f for f in glob.glob(os.path.join(data_path, "**", "*"), recursive=True) if os.path.isfile(f))
        files = [f for f in allf if os.path.splitext(f)[1].lower() in SUPPORTED_EXTS]
        return files or allf  # fall back to everything if nothing has a known extension
    return [data_path]


# =============================================================================
# prepare
# =============================================================================

def prepare_data(data_path: str, out_dir: str, val_fraction: float = 0.1, seed: int = 1337,
                 shuffle_files: bool = False, doc_sep: Optional[str] = None,
                 text_column: str = "auto", max_bytes: Optional[int] = None,
                 chunk_bytes: int = 64 << 20, verbose: bool = True) -> Dict[str, Any]:
    """
    Stream text / markdown / parquet / jsonl file(s) into ``out_dir/{train,val}.bin``.

    * Memory use is O(chunk_bytes), not O(corpus): a multi-GB file is processed in chunks.
    * Plain-text files: one EOS after each file, plus one at every occurrence of ``doc_sep``
      (a literal string such as ``"<|endoftext|>"``). Parquet / jsonl: one EOS per row/record.
    * ``max_bytes``: stop after this many *source* bytes (cheap way to take a slice).
    * The last ``val_fraction`` of the token stream becomes the validation set.

    Returns the metadata dict that is also written to ``meta.json``.
    """
    if not 0.0 < val_fraction < 1.0:
        raise ValueError(f"val_fraction must be in (0, 1), got {val_fraction}")
    files = list_source_files(data_path)
    if not files:
        raise ValueError(f"No files found at {data_path}")
    if shuffle_files:
        files = list(files)
        random.Random(seed).shuffle(files)
    os.makedirs(out_dir, exist_ok=True)

    sep = doc_sep.encode("utf-8") if doc_sep else None
    tmp_path = os.path.join(out_dir, "stream.tmp")
    n_src = 0
    t0 = time.time()
    with open(tmp_path, "wb") as sink_f:
        enc = _StreamEncoder(lambda a: a.tofile(sink_f), sep)
        budget = max_bytes
        for f in files:
            if budget is not None and n_src >= budget:
                break
            ext = os.path.splitext(f)[1].lower()
            if ext == ".parquet" or ext == ".jsonl":
                docs = _iter_parquet_docs(f, text_column) if ext == ".parquet" else _iter_jsonl_docs(f)
                for doc in docs:
                    if budget is not None and n_src >= budget:
                        break
                    enc.feed(doc)
                    enc.end_document()
                    n_src += len(doc)
            else:
                for chunk in _iter_text_chunks(f, chunk_bytes):
                    if budget is not None and n_src + len(chunk) > budget:
                        chunk = chunk[: budget - n_src]
                    enc.feed(chunk)
                    n_src += len(chunk)
                    if verbose:
                        print(f"\r[prepare] {n_src / 1e6:8.1f} MB read, {enc.n_tokens / 1e6:8.1f}M tokens", end="")
                    if budget is not None and n_src >= budget:
                        break
                enc.end_document()
        n = enc.n_tokens
        n_docs = enc.n_docs
    if verbose:
        print()
    if n == 0:
        os.remove(tmp_path)
        raise ValueError(f"All files at {data_path} are empty; nothing to tokenize.")

    n_val = max(1, int(n * val_fraction))
    n_train = n - n_val
    if n_train <= 0:
        os.remove(tmp_path)
        raise ValueError("Dataset too small after train/val split; provide more data.")

    # split in place: copy the tail to val.bin, truncate the stream to become train.bin
    val_path = os.path.join(out_dir, "val.bin")
    train_path = os.path.join(out_dir, "train.bin")
    stream = np.memmap(tmp_path, dtype=np.uint16, mode="r", shape=(n,))
    with open(val_path, "wb") as vf:
        for s in range(n_train, n, 1 << 24):
            np.asarray(stream[s: min(n, s + (1 << 24))]).tofile(vf)
    del stream
    os.truncate(tmp_path, n_train * 2)
    os.replace(tmp_path, train_path)

    meta = {
        "vocab_size": ByteTokenizer.VOCAB_SIZE,
        "dtype": "uint16",
        "n_train_tokens": int(n_train),
        "n_val_tokens": int(n_val),
        "n_documents": int(n_docs),
        "source_bytes": int(n_src),
        "source_files": files,
        "val_fraction": val_fraction,
        "shuffle_files": shuffle_files,
        "seed": seed if shuffle_files else None,
        "doc_sep": doc_sep,
        "max_bytes": max_bytes,
        "prepare_seconds": round(time.time() - t0, 1),
    }
    with open(os.path.join(out_dir, "meta.json"), "w") as fh:
        json.dump(meta, fh, indent=2)
    if verbose:
        print(f"[prepare] {len(files)} file(s) -> {n_train:,} train / {n_val:,} val tokens "
              f"({n_docs:,} documents) in {meta['prepare_seconds']}s -> {out_dir}")
    return meta


def load_meta(data_dir: str) -> Dict[str, Any]:
    with open(os.path.join(data_dir, "meta.json")) as fh:
        return json.load(fh)


# =============================================================================
# Batching
# =============================================================================

class ByteDataset:
    """Random-window sampler over one or more prepared directories.

    * Owns a private ``numpy.random.Generator``: batch sampling never perturbs (or is
      perturbed by) global RNG use elsewhere, and its state is checkpointable
      (``state_dict`` / ``load_state_dict``) so a resumed run continues the exact sequence.
    * With several directories, each row of a batch is drawn from directory ``i`` with
      probability ``weights[i]`` (replay / continual-learning mixes).
    * ``fixed_batches`` gives a deterministic evaluation set, so validation numbers are
      comparable from step to step (the v1 loop drew fresh random batches every eval).
    """

    def __init__(self, data_dirs, weights: Optional[Sequence[float]] = None, seed: int = 1337, rank: int = 0):
        if isinstance(data_dirs, str):
            data_dirs = [data_dirs]
        self.data_dirs = list(data_dirs)
        if not self.data_dirs:
            raise ValueError("data_dirs is empty")
        w = np.asarray(weights if weights is not None else [1.0] * len(self.data_dirs), dtype=np.float64)
        if len(w) != len(self.data_dirs) or (w < 0).any() or w.sum() <= 0:
            raise ValueError("weights must be non-negative, sum > 0, one per data dir")
        self.weights = w / w.sum()
        self.rng = np.random.default_rng([seed, rank])
        self._arrs: Dict[Tuple[str, str], np.memmap] = {}

    def array(self, data_dir: str, split: str) -> np.memmap:
        key = (data_dir, split)
        arr = self._arrs.get(key)
        if arr is None:
            path = os.path.join(data_dir, f"{split}.bin")
            if not os.path.exists(path):
                raise FileNotFoundError(f"{path} not found; run prepare_data first")
            arr = np.memmap(path, dtype=np.uint16, mode="r")
            self._arrs[key] = arr
        return arr

    def n_tokens(self, split: str, dir_index: int = 0) -> int:
        return len(self.array(self.data_dirs[dir_index], split))

    def _check(self, arr: np.ndarray, path: str, split: str, block_size: int) -> None:
        if len(arr) <= block_size + 1:
            raise ValueError(f"{path}/{split}.bin has only {len(arr)} tokens, need > block_size+1 "
                             f"({block_size + 1}). Use a larger corpus or a smaller block_size.")

    def _gather(self, arr: np.ndarray, starts: np.ndarray, block_size: int) -> Tuple[np.ndarray, np.ndarray]:
        idx = starts[:, None] + np.arange(block_size + 1)[None, :]
        xy = arr[idx].astype(np.int64)
        return xy[:, :-1], xy[:, 1:]

    def _to_device(self, x: np.ndarray, y: np.ndarray, device: str) -> Tuple[torch.Tensor, torch.Tensor]:
        xt = torch.from_numpy(np.ascontiguousarray(x))
        yt = torch.from_numpy(np.ascontiguousarray(y))
        if str(device).startswith("cuda"):
            return xt.pin_memory().to(device, non_blocking=True), yt.pin_memory().to(device, non_blocking=True)
        return xt.to(device), yt.to(device)

    def get_batch(self, split: str, batch_size: int, block_size: int, device: str):
        x = np.empty((batch_size, block_size), dtype=np.int64)
        y = np.empty((batch_size, block_size), dtype=np.int64)
        if len(self.data_dirs) == 1:
            choices = np.zeros(batch_size, dtype=np.int64)
        else:
            choices = self.rng.choice(len(self.data_dirs), size=batch_size, p=self.weights)
        for d_idx, d in enumerate(self.data_dirs):
            rows = np.nonzero(choices == d_idx)[0]
            if rows.size == 0:
                continue
            arr = self.array(d, split)
            self._check(arr, d, split, block_size)
            starts = self.rng.integers(0, len(arr) - block_size, size=rows.size)
            x[rows], y[rows] = self._gather(arr, starts, block_size)
        return self._to_device(x, y, device)

    def fixed_batches(self, split: str, n_batches: int, batch_size: int, block_size: int,
                      device: str, dir_index: int = 0, seed: int = 0):
        """Deterministic evaluation windows from ``data_dirs[dir_index]`` (same every call)."""
        arr = self.array(self.data_dirs[dir_index], split)
        self._check(arr, self.data_dirs[dir_index], split, block_size)
        rng = np.random.default_rng([seed, 7919])
        out = []
        for _ in range(n_batches):
            starts = rng.integers(0, len(arr) - block_size, size=batch_size)
            x, y = self._gather(arr, starts, block_size)
            out.append(self._to_device(x, y, device))
        return out

    def state_dict(self) -> Dict[str, Any]:
        return {"bit_generator": self.rng.bit_generator.state}

    def load_state_dict(self, state: Optional[Dict[str, Any]]) -> None:
        if state and "bit_generator" in state:
            self.rng.bit_generator.state = state["bit_generator"]


# =============================================================================
# Downloads (resumable, retrying, with a clean cut at a document boundary)
# =============================================================================

_UA = {"User-Agent": "tensegrity-lm/2.0 (+https://github.com/recursive-ai-dev/tensegrity)"}


def _truncate_at_boundary(path: str, boundary: bytes, window: int = 8 << 20) -> None:
    """Cut ``path`` right after the LAST ``boundary`` in its final ``window`` bytes so a
    byte-limited slice never ends mid-document."""
    size = os.path.getsize(path)
    start = max(0, size - window)
    with open(path, "r+b") as fh:
        fh.seek(start)
        tail = fh.read()
        k = tail.rfind(boundary)
        if k >= 0:
            fh.truncate(start + k + len(boundary))


def download_file(url: str, dest: str, max_bytes: Optional[int] = None, retries: int = 5,
                  timeout: float = 30.0, chunk: int = 1 << 20, boundary: Optional[bytes] = None,
                  verbose: bool = True) -> str:
    """
    Download ``url`` to ``dest`` (streaming, never holds the file in memory).

    * Resumes a partial ``dest + '.part'`` with an HTTP Range request (restarts cleanly if
      the server ignores Range).
    * ``max_bytes``: stop after that many bytes (only that range is requested); with
      ``boundary`` the slice is then trimmed back to the last complete document.
    * Retries network errors with exponential backoff; raises after ``retries`` failures.
    """
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        if verbose:
            print(f"[download] using cached {dest} ({os.path.getsize(dest) / 1e6:.1f} MB)")
        return dest
    os.makedirs(os.path.dirname(os.path.abspath(dest)), exist_ok=True)
    part = dest + ".part"
    last_err: Optional[BaseException] = None
    for attempt in range(retries):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        if max_bytes is not None and have >= max_bytes:
            break
        headers = dict(_UA)
        if have or max_bytes is not None:
            end = "" if max_bytes is None else str(max_bytes - 1)
            headers["Range"] = f"bytes={have}-{end}"
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                status = getattr(resp, "status", 200)
                total = resp.headers.get("Content-Length")
                total = int(total) + (have if status == 206 else 0) if total else None
                mode = "ab" if (status == 206 and have) else "wb"
                if mode == "wb":
                    have = 0
                got, t_last = have, time.time()
                with open(part, mode) as out:
                    while True:
                        buf = resp.read(chunk)
                        if not buf:
                            break
                        if max_bytes is not None and got + len(buf) > max_bytes:
                            buf = buf[: max_bytes - got]
                        out.write(buf)
                        got += len(buf)
                        if verbose and time.time() - t_last > 2.0:
                            pct = f" ({100 * got / total:5.1f}%)" if total else ""
                            print(f"\r[download] {got / 1e6:9.1f} MB{pct}", end="")
                            t_last = time.time()
                        if max_bytes is not None and got >= max_bytes:
                            break
            if total is not None and max_bytes is None and got < total:
                raise ConnectionError(f"short read: {got} of {total} bytes")
            last_err = None
            break
        except (urllib.error.URLError, ConnectionError, socket.timeout, TimeoutError, OSError) as e:
            last_err = e
            wait = min(60, 2 ** attempt)
            if verbose:
                print(f"\n[download] attempt {attempt + 1}/{retries} failed ({e}); retrying in {wait}s")
            time.sleep(wait)
    if last_err is not None:
        raise RuntimeError(f"download of {url} failed after {retries} attempts: {last_err}") from last_err
    if verbose:
        print()
    if boundary:
        _truncate_at_boundary(part, boundary)
    os.replace(part, dest)
    if verbose:
        print(f"[download] saved {dest} ({os.path.getsize(dest) / 1e6:.1f} MB)")
    return dest


# name -> spec.  `doc_sep` is the literal separator inside the text file.
DATASETS: Dict[str, Dict[str, Any]] = {
    "shakespeare": dict(
        url="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        filename="tinyshakespeare.txt", doc_sep=None, val_fraction=0.05, approx_mb=1.1,
        description="Tiny Shakespeare (1.1 MB). Smoke tests / quick demos."),
    "tinystories-valid": dict(
        url="https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt",
        filename="TinyStories-valid.txt", doc_sep="<|endoftext|>", val_fraction=0.05, approx_mb=19,
        description="TinyStories validation split (19 MB). Good for the 1M-5M models."),
    "tinystories": dict(
        url="https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt",
        filename="TinyStories-train.txt", doc_sep="<|endoftext|>", val_fraction=0.01, approx_mb=1920,
        description="TinyStories train split (1.9 GB; use max_mb to take a slice)."),
}


def fetch_dataset(name: str, root: str, max_mb: Optional[float] = None, verbose: bool = True) -> Dict[str, Any]:
    """Download (cached) a named dataset into ``root`` and return its spec plus ``path``."""
    if name not in DATASETS:
        raise ValueError(f"unknown dataset {name!r}; available: {sorted(DATASETS)}")
    spec = dict(DATASETS[name])
    max_bytes = int(max_mb * 1e6) if max_mb else None
    suffix = f".{int(max_mb)}mb" if max_bytes else ""
    dest = os.path.join(root, spec["filename"] + suffix)
    boundary = spec["doc_sep"].encode() if (max_bytes and spec["doc_sep"]) else None
    spec["path"] = download_file(spec["url"], dest, max_bytes=max_bytes, boundary=boundary, verbose=verbose)
    return spec


def fetch_hf_streaming(repo: str, config: Optional[str], split: str, text_field: str, out_path: str,
                       max_mb: float, doc_sep: str = "<|endoftext|>", verbose: bool = True) -> str:
    """Stream documents from a Hugging Face dataset (needs ``pip install datasets``) into one
    text file separated by ``doc_sep`` until ``max_mb`` megabytes were written."""
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return out_path
    try:
        from datasets import load_dataset
    except ImportError as e:  # pragma: no cover - environment dependent
        raise ImportError("this dataset needs `pip install datasets`") from e
    os.makedirs(os.path.dirname(os.path.abspath(out_path)), exist_ok=True)
    ds = load_dataset(repo, config, split=split, streaming=True)
    budget, written, n_docs, t_last = int(max_mb * 1e6), 0, 0, time.time()
    part = out_path + ".part"
    with open(part, "wb") as fh:
        for row in ds:
            text = (row.get(text_field) or "").replace(doc_sep, "")
            if not text:
                continue
            blob = text.encode("utf-8", errors="replace") + doc_sep.encode()
            fh.write(blob)
            written += len(blob)
            n_docs += 1
            if verbose and time.time() - t_last > 2.0:
                print(f"\r[stream] {written / 1e6:8.1f} / {max_mb:.0f} MB ({n_docs:,} docs)", end="")
                t_last = time.time()
            if written >= budget:
                break
    os.replace(part, out_path)
    if verbose:
        print(f"\n[stream] saved {out_path} ({written / 1e6:.1f} MB, {n_docs:,} docs)")
    return out_path

In [ ]:
%%writefile tensegrity_src/tensegrity/checkpoint.py
"""Checkpointing: atomic writes, safe-to-unpickle RNG state, v1 -> v2 compatible loading.

Format v2 differences from v1 (readers of v1 files are fully supported):
  * RNG state is stored as tensors / ints only, so checkpoints load with
    ``torch.load(weights_only=True)`` -- no arbitrary-code-execution surface.
  * ``ckpt_best.pt`` is weights-only by default (a third of the size of a full checkpoint).
  * Extra training state (EMA, dataset sampler, LayerDrop generator, run plan) is optional.
"""

from __future__ import annotations

import os
import pickle
import random
import warnings
from typing import Any, Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn

from .config import TensegrityConfig
from .model import TensegrityLM

CHECKPOINT_FORMAT_VERSION = 2
_REQUIRED = ("model_state_dict", "config", "step", "best_val_loss")


# =============================================================================
# RNG state (safe representation)
# =============================================================================

def capture_rng_state(model: Optional[nn.Module] = None, dataset: Any = None) -> Dict[str, Any]:
    st = random.getstate()
    ns = np.random.get_state()
    state: Dict[str, Any] = {
        "torch": torch.get_rng_state(),
        "python": {"version": st[0], "state": list(st[1]), "gauss": st[2]},
        "numpy": {"name": ns[0], "keys": torch.from_numpy(ns[1].astype(np.int64)),
                  "pos": int(ns[2]), "has_gauss": int(ns[3]), "cached": float(ns[4])},
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    raw = getattr(model, "module", model)
    if raw is not None and hasattr(raw, "layerdrop_state"):
        state["layerdrop"] = raw.layerdrop_state()
    if dataset is not None:
        state["dataset"] = dataset.state_dict()
    return state


def restore_rng_state(state: Optional[Dict[str, Any]], model: Optional[nn.Module] = None,
                      dataset: Any = None) -> None:
    """Best-effort restore; accepts both v2 (safe) and v1 (raw tuple / ndarray) layouts."""
    if not state:
        return
    try:
        py = state.get("python", state.get("random"))
        if isinstance(py, dict):
            random.setstate((py["version"], tuple(py["state"]), py["gauss"]))
        elif py is not None:
            random.setstate(py)
    except Exception as e:  # noqa: BLE001
        warnings.warn(f"failed to restore python RNG state ({e})")
    if "torch" in state:
        t = state["torch"]
        torch.set_rng_state(t.cpu() if isinstance(t, torch.Tensor) else t)
    if "numpy" in state:
        ns = state["numpy"]
        try:
            if isinstance(ns, dict):
                np.random.set_state((ns["name"], ns["keys"].numpy().astype(np.uint32),
                                     ns["pos"], ns["has_gauss"], ns["cached"]))
            else:
                np.random.set_state(ns)
        except Exception as e:  # noqa: BLE001
            warnings.warn(f"failed to restore numpy RNG state ({e})")
    if "cuda" in state and torch.cuda.is_available():
        try:
            cs = [s.cpu() if isinstance(s, torch.Tensor) else s for s in state["cuda"]]
            n = torch.cuda.device_count()
            if len(cs) == n:
                torch.cuda.set_rng_state_all(cs)
            else:  # different GPU count than when saved: restore what exists
                for dev in range(min(len(cs), n)):
                    torch.cuda.set_rng_state(cs[dev], device=dev)
        except Exception as e:  # noqa: BLE001
            warnings.warn(f"failed to restore CUDA RNG state ({e})")
    raw = getattr(model, "module", model)
    if "layerdrop" in state and raw is not None and hasattr(raw, "set_layerdrop_state"):
        raw.set_layerdrop_state(state["layerdrop"])
    if dataset is not None and "dataset" in state:
        dataset.load_state_dict(state["dataset"])


# =============================================================================
# save / load
# =============================================================================

def atomic_torch_save(obj: Any, path: str) -> None:
    """torch.save to ``path + '.tmp'``, fsync, then atomic rename: a crash (or a Colab
    disconnect) mid-write can never leave a truncated file that looks like a checkpoint."""
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    with open(tmp, "r+b") as f:
        os.fsync(f.fileno())
    os.replace(tmp, path)


def save_checkpoint(path: str, model: nn.Module, optimizer: Optional[torch.optim.Optimizer],
                    cfg: TensegrityConfig, step: int, best_val_loss: float,
                    rng_state: Optional[Dict[str, Any]] = None,
                    scaler: Optional[torch.amp.GradScaler] = None,
                    ema_state: Optional[Dict[str, Any]] = None,
                    extra: Optional[Dict[str, Any]] = None) -> None:
    """``optimizer=None`` writes a weights-only checkpoint (still loadable by
    :func:`load_checkpoint` / :func:`load_model`)."""
    raw = getattr(model, "module", model)
    ckpt: Dict[str, Any] = {
        "format_version": CHECKPOINT_FORMAT_VERSION,
        "model_state_dict": raw.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "ema_state_dict": ema_state,
        "config": cfg.to_dict(),
        "step": int(step),
        "best_val_loss": float(best_val_loss),
        "rng_state": rng_state,
    }
    if extra:
        ckpt.update(extra)
    atomic_torch_save(ckpt, path)


def load_checkpoint(path: str, device: str = "cpu", allow_legacy_pickle: bool = True) -> Dict[str, Any]:
    """
    Load a checkpoint dict. v2 files load under ``weights_only=True``. Files written by the
    original release (v1) contain numpy RNG objects and need full unpickling; that is only
    attempted when ``allow_legacy_pickle`` is True, with a warning -- **only load checkpoint
    files you trust** (pickle can execute code).
    """
    try:
        ckpt = torch.load(path, map_location=device, weights_only=True)
    except (pickle.UnpicklingError, RuntimeError, AttributeError, ValueError) as e:
        if not allow_legacy_pickle:
            raise
        warnings.warn(f"{os.path.basename(path)}: not loadable with weights_only=True ({type(e).__name__}); "
                      f"treating it as a legacy (v1) checkpoint and unpickling it. Only do this for files you trust.",
                      stacklevel=2)
        ckpt = torch.load(path, map_location=device, weights_only=False)
    missing = [k for k in _REQUIRED if k not in ckpt]
    if missing:
        raise ValueError(f"{path} is missing checkpoint key(s) {missing}; not a TensegrityLM checkpoint.")
    ver = ckpt.get("format_version")
    if ver not in (1, CHECKPOINT_FORMAT_VERSION):
        warnings.warn(f"{path} has format_version={ver!r}; this code reads 1 and {CHECKPOINT_FORMAT_VERSION}.")
    return ckpt


def load_model(path: str, device: str = "cpu", use_ema: bool = False, verbose: bool = False
               ) -> Tuple[TensegrityLM, TensegrityConfig, Dict[str, Any]]:
    """Build the model described by a checkpoint and load its weights. ``use_ema=True`` loads
    the EMA weights when the checkpoint has them."""
    ckpt = load_checkpoint(path, device)
    cfg = TensegrityConfig.from_dict(ckpt["config"])
    model = TensegrityLM(cfg, verbose=verbose)
    state = ckpt["model_state_dict"]
    if use_ema and ckpt.get("ema_state_dict"):
        from .train import EMA  # local import: train imports this module
        EMA.apply_state_to_model(ckpt["ema_state_dict"], model)
    else:
        model.load_state_dict(state)
    model.to(device)
    return model, cfg, ckpt

In [ ]:
%%writefile tensegrity_src/tensegrity/hw.py
"""Hardware helpers: device / dtype selection, GPU facts, micro-batch autotune, throughput.

The headline fix over the original release: ``torch.cuda.is_bf16_supported()`` returns True on
a Turing T4 (Colab's free GPU) because it counts *emulation*; picking bf16 there is far slower
than fp16. :func:`pick_dtype` decides from the compute capability instead.
"""

from __future__ import annotations

import time
import warnings
from typing import Any, Dict, Optional

import torch

from .config import TensegrityConfig

# Dense fp16/bf16 tensor-core peak, TFLOPs (vendor figures; used only to display an MFU estimate).
_PEAK_TFLOPS = (
    ("H200", 989.0), ("H100", 989.0), ("B200", 2250.0), ("A100", 312.0), ("L40S", 362.0),
    ("L4", 121.0), ("A10G", 70.0), ("A10", 125.0), ("V100", 125.0), ("T4", 65.0),
    ("RTX 4090", 165.0), ("RTX 4080", 97.5), ("RTX 4070", 58.0), ("RTX 4060", 44.0),
    ("RTX 3090", 71.0), ("RTX 3080", 59.5), ("RTX 3070", 40.6), ("RTX 3060", 25.4),
    ("P100", 18.7),
)


def pick_device(requested: str = "auto") -> str:
    if requested != "auto":
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def device_type_of(device: str) -> str:
    d = str(device)
    return "cuda" if "cuda" in d else ("mps" if d == "mps" else "cpu")


def native_bf16(device: str = "cuda") -> bool:
    """True only for GPUs with hardware bf16 (compute capability >= 8)."""
    if device_type_of(device) != "cuda" or not torch.cuda.is_available():
        return False
    return torch.cuda.get_device_capability(torch.device(device))[0] >= 8


def pick_dtype(requested: str, device: str) -> torch.dtype:
    """'auto' -> bf16 on Ampere+ (native), fp16 on older CUDA GPUs (T4/V100), fp32 elsewhere."""
    dev_type = device_type_of(device)
    if requested != "auto":
        dt = {"float32": torch.float32, "fp32": torch.float32, "bfloat16": torch.bfloat16,
              "bf16": torch.bfloat16, "float16": torch.float16, "fp16": torch.float16}[requested]
        if dt == torch.bfloat16 and dev_type == "cuda" and not native_bf16(device):
            warnings.warn("bfloat16 requested on a GPU without native bf16 (compute capability < 8): "
                          "it will be emulated and much slower than float16.", stacklevel=2)
        return dt
    if dev_type == "cuda":
        return torch.bfloat16 if native_bf16(device) else torch.float16
    return torch.float32


def amp_settings(dtype: torch.dtype, device: str) -> Dict[str, Any]:
    dev_type = device_type_of(device)
    use_amp = dtype in (torch.float16, torch.bfloat16) and dev_type in ("cuda", "cpu")
    return {
        "use_amp": use_amp,
        "amp_dtype": dtype if use_amp else torch.float32,
        "autocast_device": "cpu" if dev_type == "mps" else dev_type,
        "use_scaler": dtype == torch.float16 and dev_type == "cuda",
    }


def gpu_info() -> Dict[str, Any]:
    """Facts about the active GPU (or a CPU stub)."""
    if not torch.cuda.is_available():
        return {"available": False, "name": "cpu", "capability": None, "total_gb": 0.0,
                "native_bf16": False, "peak_tflops": None}
    p = torch.cuda.get_device_properties(0)
    peak = next((v for k, v in _PEAK_TFLOPS if k.lower() in p.name.lower()), None)
    return {"available": True, "name": p.name, "capability": (p.major, p.minor),
            "total_gb": p.total_memory / 2**30, "native_bf16": p.major >= 8, "peak_tflops": peak}


def set_fast_matmul() -> None:
    """Allow TF32 for the few fp32 matmuls that remain under autocast (Ampere+ only)."""
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
        torch.set_float32_matmul_precision("high")


def mfu(cfg: TensegrityConfig, tokens_per_sec: float, seq_len: int, peak_tflops: Optional[float]) -> Optional[float]:
    """Model FLOPs utilisation estimate in [0, 1] (None if the GPU's peak is unknown)."""
    if not peak_tflops:
        return None
    return cfg.flops_per_token(seq_len, training=True) * tokens_per_sec / (peak_tflops * 1e12)


# =============================================================================
# Autotune / benchmark
# =============================================================================

def _is_oom(e: BaseException) -> bool:
    oom_cls = getattr(torch.cuda, "OutOfMemoryError", ())
    return (oom_cls and isinstance(e, oom_cls)) or "out of memory" in str(e).lower()


def _fwd_bwd(model: torch.nn.Module, B: int, T: int, device: str, amp: Dict[str, Any]) -> None:
    x = torch.randint(0, 256, (B, T), device=device)
    y = torch.randint(0, 256, (B, T), device=device)
    with torch.autocast(device_type=amp["autocast_device"], dtype=amp["amp_dtype"], enabled=amp["use_amp"]):
        _, losses, _ = model(x, y)
    losses["total_loss"].backward()


def measure_throughput(model: torch.nn.Module, micro_batch: int, seq_len: int, device: str,
                       dtype: torch.dtype, steps: int = 5, warmup: int = 2) -> float:
    """Training tokens/second for fwd+bwd at the given micro-batch (no optimizer step)."""
    amp = amp_settings(dtype, device)
    model.train()
    sync = torch.cuda.synchronize if device_type_of(device) == "cuda" else (lambda: None)
    for _ in range(warmup):
        _fwd_bwd(model, micro_batch, seq_len, device, amp)
        model.zero_grad(set_to_none=True)
    sync()
    t0 = time.perf_counter()
    for _ in range(steps):
        _fwd_bwd(model, micro_batch, seq_len, device, amp)
        model.zero_grad(set_to_none=True)
    sync()
    return steps * micro_batch * seq_len / (time.perf_counter() - t0)


def autotune_micro_batch(model: torch.nn.Module, seq_len: int, device: str, dtype: torch.dtype,
                         target_batch_tokens: int, max_micro: int = 128, headroom: float = 0.9,
                         verbose: bool = True) -> Dict[str, Any]:
    """
    Find the largest power-of-two micro-batch whose fwd+bwd fits in memory, then choose
    ``grad_accum`` so one optimizer step sees >= ``target_batch_tokens`` tokens.

    If even micro-batch 1 does not fit, gradient checkpointing is switched on and the search
    repeats once. On CPU/MPS (no OOM signal) a small fixed micro-batch is used.
    """
    dev_type = device_type_of(device)
    if dev_type != "cuda":
        mb = max(1, min(8, target_batch_tokens // seq_len))
        accum = max(1, round(target_batch_tokens / (mb * seq_len)))
        return {"micro_batch": mb, "grad_accum": accum, "grad_checkpoint": False}

    amp = amp_settings(dtype, device)
    was_training = model.training
    model.train()
    total_mem = torch.cuda.get_device_properties(device).total_memory
    # The probe runs before the optimizer and the weight-EMA exist. AdamW keeps 2 fp32 moments
    # per parameter and the EMA 1 more copy, so reserve 3x the parameter bytes up front
    # (Muon needs 1x, so this is conservative for it).
    reserve = 3 * sum(p.numel() * p.element_size() for p in model.parameters())

    def search() -> Optional[int]:
        best, mb = None, 1
        while mb <= max_micro:
            try:
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                _fwd_bwd(model, mb, seq_len, device, amp)
                peak = torch.cuda.max_memory_allocated()
                model.zero_grad(set_to_none=True)
                if peak + reserve > headroom * total_mem:
                    break
                best = mb
                if verbose:
                    print(f"[autotune] micro_batch={mb:3d} fits (peak {peak / 2**30:.2f} GiB + "
                          f"{reserve / 2**30:.2f} GiB reserved for optimizer/EMA)")
                mb *= 2
            except Exception as e:  # noqa: BLE001
                model.zero_grad(set_to_none=True)
                if not _is_oom(e):
                    raise
                break
        torch.cuda.empty_cache()
        return best

    ckpt_flag = bool(getattr(model, "grad_checkpoint", False))
    best = search()
    if best is None and not ckpt_flag:
        if verbose:
            print("[autotune] nothing fits without gradient checkpointing; enabling it")
        model.grad_checkpoint = True
        ckpt_flag = True
        best = search()
    model.train(was_training)
    if best is None:
        raise RuntimeError("Even micro_batch=1 does not fit in GPU memory at this sequence length; "
                           "reduce block_size or pick a smaller model.")
    accum = max(1, -(-target_batch_tokens // (best * seq_len)))
    return {"micro_batch": best, "grad_accum": accum, "grad_checkpoint": ckpt_flag}

In [ ]:
%%writefile tensegrity_src/tensegrity/schedule.py
"""Learning-rate schedules and run planning (token budget / wall-clock budget -> steps)."""

from __future__ import annotations

import math
from dataclasses import asdict, dataclass
from typing import Any, Dict, Optional


def get_lr(step: int, warmup_steps: int, max_steps: int, max_lr: float, min_lr: float) -> float:
    """Linear warmup then cosine decay to ``min_lr`` ("temperature annealing")."""
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    ratio = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * ratio)) * (max_lr - min_lr)


def get_lr_wsd(step: int, warmup_steps: int, stable_steps: int, decay_steps: int,
               max_lr: float, min_lr: float) -> float:
    """
    Warmup-Stable-Decay (MiniCPM; also used by Llama 3.1 and DeepSeek-V2): linear warmup,
    hold at ``max_lr``, then linear decay to ``min_lr``. The decay only has to be as long as
    ``decay_steps`` -- the total budget need not be fixed at step 0.
    """
    assert warmup_steps >= 1, "warmup_steps must be >= 1"
    assert decay_steps >= 1, "decay_steps must be >= 1"
    decay_start = warmup_steps + stable_steps
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step < decay_start:
        return max_lr
    if step >= decay_start + decay_steps:
        return min_lr
    return max_lr - (step - decay_start) / decay_steps * (max_lr - min_lr)


@dataclass
class RunPlan:
    """A concrete, serialisable training plan. Persisted next to the checkpoints so a resumed
    run (even in a new Colab session) keeps exactly the same schedule."""
    tokens_per_step: int
    max_steps: int
    warmup_steps: int
    stable_steps: int
    decay_steps: int
    schedule: str = "wsd"

    @property
    def total_tokens(self) -> int:
        return self.tokens_per_step * self.max_steps

    def multiplier(self, step: int, floor_ratio: float) -> float:
        """LR as a fraction of the peak, in [floor_ratio, 1]."""
        if self.schedule == "wsd":
            return get_lr_wsd(step, self.warmup_steps, self.stable_steps, self.decay_steps, 1.0, floor_ratio)
        return get_lr(step, self.warmup_steps, self.max_steps, 1.0, floor_ratio)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def make_plan(total_tokens: int, tokens_per_step: int, schedule: str = "wsd",
              warmup_frac: float = 0.02, decay_frac: float = 0.20, min_warmup: int = 20) -> RunPlan:
    """Turn a token budget into steps. For WSD the final ``decay_frac`` of steps is the cool-down."""
    if schedule not in ("wsd", "cosine"):
        raise ValueError(f"schedule must be 'wsd' or 'cosine', got {schedule!r}")
    if tokens_per_step <= 0 or total_tokens <= 0:
        raise ValueError("total_tokens and tokens_per_step must be positive")
    max_steps = max(10, total_tokens // tokens_per_step)
    warmup = min(max(min_warmup, int(warmup_frac * max_steps)), max(1, max_steps // 4))
    decay = max(1, int(decay_frac * max_steps))
    stable = max(0, max_steps - warmup - decay)
    return RunPlan(tokens_per_step, max_steps, warmup, stable, decay, schedule)


def plan_for_time_budget(tokens_per_sec: float, hours: float, tokens_per_step: int,
                         max_tokens: Optional[int] = None, safety: float = 0.92, **kw) -> RunPlan:
    """Largest plan expected to finish within ``hours`` of pure training time (``safety`` leaves
    room for evaluation, checkpointing and throughput jitter), capped at ``max_tokens``."""
    if tokens_per_sec <= 0 or hours <= 0:
        raise ValueError("tokens_per_sec and hours must be positive")
    budget = int(tokens_per_sec * hours * 3600.0 * safety)
    if max_tokens is not None:
        budget = min(budget, int(max_tokens))
    return make_plan(max(budget, tokens_per_step * 10), tokens_per_step, **kw)


def extend_plan(plan: RunPlan, extra_steps: int, at_step: int = 0) -> RunPlan:
    """Lengthen a WSD run by ``extra_steps`` (e.g. "I have two more hours"). The extra steps join the
    *stable* phase, so the cool-down still happens at the end -- the property that makes WSD
    suitable for open-ended training. Only legal before the cool-down has started."""
    if plan.schedule != "wsd":
        raise ValueError("extend_plan needs a WSD plan (a cosine schedule's shape depends on its total length)")
    if extra_steps < 1:
        raise ValueError("extra_steps must be >= 1")
    if at_step > plan.warmup_steps + plan.stable_steps:
        raise ValueError(f"the cool-down already began at step {plan.warmup_steps + plan.stable_steps}; "
                         f"continue as a NEW run initialised from the weights instead")
    return RunPlan(plan.tokens_per_step, plan.max_steps + extra_steps, plan.warmup_steps,
                   plan.stable_steps + extra_steps, plan.decay_steps, "wsd")

In [ ]:
%%writefile tensegrity_src/tensegrity/train.py
"""Training engine: one resumable, preemption-safe loop for notebooks, scripts and DDP.

What it does that the v1 loops did not
--------------------------------------
* **Correct resume.** Base learning rates are captured from the *fresh* optimizer before any
  state is loaded. (v1 captured them after ``optimizer.load_state_dict``, i.e. the already
  decayed LR, and decayed it a second time.) Model, optimizer, GradScaler, EMA, all RNGs and the
  data sampler are restored, so a resumed run continues the exact batch sequence.
* **Preemption safety.** Checkpoints every N minutes as well as every eval; atomic writes;
  an optional wall-clock limit that stops cleanly with a checkpoint (Colab sessions end).
* **No hidden syncs.** Losses accumulate on-device and are read only when logging.
* **Non-finite protection.** A NaN/Inf gradient skips the step instead of poisoning the weights.
* **Deterministic validation** (fixed windows) reported as loss, bits-per-byte and perplexity,
  for the raw weights and, if enabled, the EMA weights; the better one becomes ``ckpt_best.pt``.
"""

from __future__ import annotations

import contextlib
import dataclasses
import json
import math
import os
import time
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Sequence

import torch
import torch.distributed as dist
import torch.nn as nn

from .checkpoint import capture_rng_state, load_checkpoint, restore_rng_state, save_checkpoint
from .config import TensegrityConfig
from .data import ByteDataset
from .hw import amp_settings, device_type_of, gpu_info, mfu, pick_device, pick_dtype, set_fast_matmul
from .model import TensegrityLM
from .schedule import RunPlan
from .tokenizer import ByteTokenizer

Callback = Callable[[str, Dict[str, Any]], None]


# =============================================================================
# EMA of weights
# =============================================================================

class EMA:
    """Exponential moving average of the model's parameters (kept in fp32).

    The effective decay warms up as ``min(decay, (1 + n) / (10 + n))`` so the average is
    meaningful from the first steps. ``swapped(model)`` temporarily loads the averaged
    weights (bit-exact restore on exit), e.g. for evaluation or export.
    """

    def __init__(self, model: nn.Module, decay: float):
        if not 0.0 < decay < 1.0:
            raise ValueError(f"EMA decay must be in (0, 1), got {decay}")
        self.decay = decay
        self.num_updates = 0
        named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
        self.shadow: Dict[str, torch.Tensor] = {n: p.detach().clone().float() for n, p in named}
        # cached, aligned lists: the parameters are the same objects for the model's lifetime
        self._params = [p for _, p in named]
        self._shadows = [self.shadow[n] for n, _ in named]

    @torch.no_grad()
    def update(self, model: Optional[nn.Module] = None) -> None:
        """Fold the current parameters into the average (``model`` is accepted for symmetry
        with ``swapped`` but the parameters captured at construction are what is averaged)."""
        self.num_updates += 1
        d = min(self.decay, (1.0 + self.num_updates) / (10.0 + self.num_updates))
        params = [p.detach() for p in self._params]
        try:
            torch._foreach_lerp_(self._shadows, params, 1.0 - d)
        except (AttributeError, RuntimeError, TypeError):
            for s, p in zip(self._shadows, params):
                s.lerp_(p, 1.0 - d)

    @torch.no_grad()
    def _swap(self, model: nn.Module) -> None:
        for n, p in model.named_parameters():
            s = self.shadow.get(n)
            if s is not None:
                tmp = p.data.clone()
                p.data.copy_(s)
                s.copy_(tmp)

    @contextlib.contextmanager
    def swapped(self, model: nn.Module):
        self._swap(model)
        try:
            yield model
        finally:
            self._swap(model)

    def state_dict(self) -> Dict[str, Any]:
        return {"decay": self.decay, "num_updates": self.num_updates,
                "shadow": {k: v.detach().cpu() for k, v in self.shadow.items()}}

    def load_state_dict(self, state: Dict[str, Any]) -> None:
        self.decay = float(state["decay"])
        self.num_updates = int(state["num_updates"])
        for k, v in state["shadow"].items():
            if k in self.shadow:
                self.shadow[k].copy_(v)

    @staticmethod
    @torch.no_grad()
    def apply_state_to_model(state: Dict[str, Any], model: nn.Module) -> None:
        """Load EMA weights from a saved EMA state straight into ``model``."""
        for n, p in model.named_parameters():
            if n in state["shadow"]:
                p.data.copy_(state["shadow"][n].to(p.device, p.dtype))


def default_ema_decay(max_steps: int) -> float:
    """Average over roughly the last 5% of training (capped at 0.9995)."""
    return float(min(0.9995, 1.0 - 1.0 / max(10.0, 0.05 * max_steps)))


# =============================================================================
# Config
# =============================================================================

@dataclass
class TrainConfig:
    # batch
    batch_size: int = 16            # micro-batch, sequences
    grad_accum_steps: int = 1
    block_size: Optional[int] = None  # training sequence length (default: model block_size)
    # optimisation
    learning_rate: float = 3e-4
    min_lr: float = 3e-5
    optimizer: str = "adamw"        # "adamw" | "muon"
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    muon_lr: float = 0.02
    muon_momentum: float = 0.95
    muon_weight_decay: Optional[float] = None
    use_8bit_optimizer: bool = False
    # numerics
    dtype: str = "auto"
    compile: bool = False
    # averaging
    ema_decay: float = 0.0          # 0 = off; see default_ema_decay()
    # evaluation / logging
    log_interval: int = 10
    eval_interval: int = 250
    eval_batches: int = 20
    eval_batch_size: int = 16
    sample_every_evals: int = 1     # 0 = never
    sample_prompt: str = ""
    sample_len: int = 200
    # persistence
    ckpt_interval_minutes: float = 10.0
    max_minutes: Optional[float] = None
    max_nonfinite: int = 50
    seed: int = 1337
    verbose: bool = True

    # Per-session settings that must NOT stick to a run: re-running with a different wall-clock
    # limit or verbosity has to take effect, instead of replaying whatever the interrupted
    # session used (a persisted ``max_minutes`` would stop every resume immediately).
    EPHEMERAL = ("max_minutes", "verbose")

    def to_dict(self) -> Dict[str, Any]:
        return dataclasses.asdict(self)

    def persistent_dict(self) -> Dict[str, Any]:
        return {k: v for k, v in self.to_dict().items() if k not in self.EPHEMERAL}


# =============================================================================
# Evaluation
# =============================================================================

@torch.no_grad()
def evaluate_batches(model: nn.Module, batches: Sequence, device: str, dtype: torch.dtype) -> Dict[str, float]:
    """Mean cross-entropy over fixed (x, y) batches -> loss (nats/byte), bits-per-byte, perplexity."""
    amp = amp_settings(dtype, device)
    was_training = model.training
    model.eval()
    try:
        total = 0.0
        for x, y in batches:
            with torch.autocast(device_type=amp["autocast_device"], dtype=amp["amp_dtype"], enabled=amp["use_amp"]):
                _, losses, _ = model(x, y)
            total += float(losses["ce_loss"])
        loss = total / len(batches)
    finally:
        model.train(was_training)
    return {"loss": loss, "bpb": loss / math.log(2.0), "ppl": math.exp(min(loss, 50.0))}


def evaluate(model: nn.Module, data_dir: str, cfg: TensegrityConfig, device: str, dtype: Optional[torch.dtype] = None,
             n_batches: int = 20, batch_size: int = 16, split: str = "val") -> Dict[str, float]:
    """Convenience: evaluate on a fixed set of windows from ``data_dir``."""
    dtype = dtype or pick_dtype("auto", device)
    ds = ByteDataset(data_dir)
    batches = ds.fixed_batches(split, n_batches, batch_size, cfg.block_size, device)
    return evaluate_batches(model, batches, device, dtype)


# =============================================================================
# Trainer
# =============================================================================

def _cfg_mismatch(a: Dict[str, Any], b: Dict[str, Any]) -> List[str]:
    skip = {"grad_checkpoint", "grad_checkpoint_every"}  # memory knobs: free to differ between runs
    return [f"{k}: checkpoint={a.get(k)!r} vs model={b.get(k)!r}"
            for k in sorted(set(a) | set(b)) if k not in skip and a.get(k) != b.get(k)]


def preflight(model: TensegrityLM, tc: TrainConfig, device: str = "auto", steps: int = 4) -> Dict[str, Any]:
    """
    Run a few REAL optimisation steps (the configured optimizer, dtype, loss scaling; at 10% of the
    peak LR) on random bytes and report whether everything stayed finite -- a cheap way to catch a numerics problem on the
    user's actual GPU before committing hours. The model's weights are restored afterwards; nothing
    is written to disk.

    Returns ``{"ok": bool, "optimizer": str, "dtype": str, "reason": str | None, "loss_first": float, "loss_last": float}``.
    """
    device = pick_device(device)
    dtype = pick_dtype(tc.dtype, device)
    amp = amp_settings(dtype, device)
    snapshot = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    was_training, ld = model.training, model.layerdrop_state()
    model.to(device).train()
    model.seed_layerdrop(tc.seed)
    opt = model.configure_optimizer(
        weight_decay=tc.weight_decay, learning_rate=tc.learning_rate, betas=(tc.beta1, tc.beta2),
        device_type=device_type_of(device), use_8bit=False, use_muon=(tc.optimizer == "muon"),
        muon_lr=tc.muon_lr, muon_momentum=tc.muon_momentum, muon_weight_decay=tc.muon_weight_decay)
    for g_ in opt.param_groups:      # numerics check, not a stability test: no warmup here, so use a gentle LR
        g_["lr"] *= 0.1
    scaler = torch.amp.GradScaler("cuda", enabled=amp["use_scaler"])
    T = min(tc.block_size or model.cfg.block_size, 128)
    g = torch.Generator().manual_seed(0)
    out: Dict[str, Any] = {"ok": True, "optimizer": tc.optimizer, "dtype": str(dtype).replace("torch.", ""),
                           "reason": None, "loss_first": float("nan"), "loss_last": float("nan")}
    try:
        for i in range(steps):
            x = torch.randint(0, 256, (2, T), generator=g).to(device)
            with torch.autocast(device_type=amp["autocast_device"], dtype=amp["amp_dtype"], enabled=amp["use_amp"]):
                _, losses, _ = model(x, torch.roll(x, -1, 1))
            loss = losses["total_loss"]
            (scaler.scale(loss) if scaler.is_enabled() else loss).backward()
            if scaler.is_enabled():
                scaler.unscale_(opt)
            gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if scaler.is_enabled():
                scaler.step(opt)
                scaler.update()
            elif torch.isfinite(gnorm):
                opt.step()
            opt.zero_grad(set_to_none=True)
            out["loss_last"] = float(loss.detach())
            if i == 0:
                out["loss_first"] = out["loss_last"]
            if not math.isfinite(out["loss_last"]):
                out.update(ok=False, reason=f"non-finite loss at step {i}")
                break
        else:
            bad = [n for n, p in model.named_parameters() if not torch.isfinite(p).all()]
            if bad:
                out.update(ok=False, reason=f"non-finite weights after the steps: {bad[:3]}")
            elif out["loss_last"] > out["loss_first"] * 2.0 + 2.0:
                out.update(ok=False, reason=f"loss diverged ({out['loss_first']:.2f} -> {out['loss_last']:.2f})")
    except Exception as e:  # noqa: BLE001 - any failure of the optimizer path is a failed preflight
        out.update(ok=False, reason=f"{type(e).__name__}: {e}")
    finally:
        model.load_state_dict(snapshot)
        model.set_layerdrop_state(ld)
        model.train(was_training)
        del opt
    return out


def preflight_or_fallback(model: TensegrityLM, tc: TrainConfig, device: str = "auto", verbose: bool = True) -> TrainConfig:
    """Run :func:`preflight`; if the configured optimizer fails (non-finite loss/weights, an exception)
    switch ``tc`` to plain AdamW and re-check. Raises only if AdamW fails too. Mutates and returns ``tc``."""
    r = preflight(model, tc, device)
    say = print if verbose else (lambda *a, **k: None)
    if r["ok"]:
        say(f"[preflight] {r['optimizer']} / {r['dtype']} OK (loss {r['loss_first']:.2f} -> {r['loss_last']:.2f} over 4 test steps)")
        return tc
    if tc.optimizer == "adamw":
        raise RuntimeError(f"preflight failed with AdamW: {r['reason']}")
    say(f"[preflight] WARNING: {tc.optimizer} failed on this device ({r['reason']}); falling back to AdamW.")
    tc.optimizer = "adamw"
    r2 = preflight(model, tc, device)
    if not r2["ok"]:
        raise RuntimeError(f"preflight failed with AdamW as well: {r2['reason']}")
    say(f"[preflight] adamw / {r2['dtype']} OK")
    return tc


def load_run_plan(out_dir: str):
    """Read ``run_plan.json`` written by a previous ``Trainer.fit`` in ``out_dir``.
    Returns ``(RunPlan, TrainConfig, TensegrityConfig)`` or None if the directory has no record.
    Per-session fields (``TrainConfig.EPHEMERAL``: wall-clock limit, verbosity) are not restored --
    set them on the returned config for the session you are about to run."""
    path = os.path.join(out_dir, "run_plan.json")
    if not os.path.exists(path):
        return None
    with open(path) as fh:
        d = json.load(fh)
    known = {f.name for f in dataclasses.fields(TrainConfig)}
    return (RunPlan(**d["plan"]), TrainConfig(**{k: v for k, v in d["train_config"].items() if k in known}),
            TensegrityConfig.from_dict(d["model_config"]))


class Trainer:
    """
    Parameters
    ----------
    model : TensegrityLM (any device; moved to ``device``)
    tc    : TrainConfig            plan : RunPlan (steps + LR schedule)
    data_dirs / data_weights : one or more prepared corpora (validation always uses data_dirs[0])
    out_dir : checkpoints (``ckpt_latest.pt`` full, ``ckpt_best.pt`` weights-only), ``train_log.jsonl``
    extra_loss : optional ``fn(model) -> scalar tensor`` added to the loss (e.g. an EWC penalty)
    callbacks : ``fn(event, payload)`` for events: start, log, eval, sample, checkpoint, end
    """

    def __init__(self, model: TensegrityLM, tc: TrainConfig, plan: RunPlan, data_dirs, out_dir: str,
                 data_weights: Optional[Sequence[float]] = None, device: str = "auto",
                 extra_loss: Optional[Callable[[nn.Module], torch.Tensor]] = None,
                 callbacks: Optional[Sequence[Callback]] = None,
                 init_optimizer: Optional[Callable[[torch.optim.Optimizer], None]] = None):
        self.model = model
        self._raw = model
        self._log_file = None
        self.cfg: TensegrityConfig = model.cfg
        self.tc, self.plan = tc, plan
        self.data_dirs = [data_dirs] if isinstance(data_dirs, str) else list(data_dirs)
        self.data_weights = data_weights
        self.out_dir = out_dir
        self.device_arg = device
        self.extra_loss = extra_loss
        self.callbacks: List[Callback] = list(callbacks or [])
        self.init_optimizer = init_optimizer
        self.history: List[Dict[str, Any]] = []
        self.best_val_loss = float("inf")
        self.best_source = "raw"
        self.tokens_seen = 0
        self.step = 0
        self.skipped_steps = 0
        self.ema: Optional[EMA] = None
        self.optimizer: Optional[torch.optim.Optimizer] = None

    # ------------------------------------------------------------------ utils
    def _emit(self, event: str, payload: Dict[str, Any]) -> None:
        for cb in self.callbacks:
            cb(event, payload)

    def _print(self, msg: str) -> None:
        if self.tc.verbose and self.master:
            print(msg, flush=True)

    @property
    def raw_model(self) -> TensegrityLM:
        return self._raw

    def _build_optimizer(self) -> torch.optim.Optimizer:
        tc = self.tc
        if tc.optimizer not in ("adamw", "muon"):
            raise ValueError(f"optimizer must be 'adamw' or 'muon', got {tc.optimizer!r}")
        return self.raw_model.configure_optimizer(
            weight_decay=tc.weight_decay, learning_rate=tc.learning_rate, betas=(tc.beta1, tc.beta2),
            device_type=self.device_type, use_8bit=tc.use_8bit_optimizer, use_muon=(tc.optimizer == "muon"),
            muon_lr=tc.muon_lr, muon_momentum=tc.muon_momentum, muon_weight_decay=tc.muon_weight_decay)

    def _write_log(self, rec: Dict[str, Any]) -> None:
        self.history.append(rec)
        if self._log_file is not None:
            self._log_file.write(json.dumps(rec) + "\n")
            self._log_file.flush()

    # ----------------------------------------------------------- checkpointing
    def _extra_state(self) -> Dict[str, Any]:
        return {"format_train": 2, "tokens_seen": self.tokens_seen, "best_source": self.best_source,
                "plan": self.plan.to_dict(), "train_config": self.tc.persistent_dict(),
                "skipped_steps": self.skipped_steps}

    def save_latest(self, step: int) -> str:
        path = os.path.join(self.out_dir, "ckpt_latest.pt")
        save_checkpoint(path, self.raw_model, self.optimizer, self.cfg, step, self.best_val_loss,
                        rng_state=capture_rng_state(self.raw_model, self.ds), scaler=self.scaler,
                        ema_state=self.ema.state_dict() if self.ema else None, extra=self._extra_state())
        self._emit("checkpoint", {"path": path, "step": step, "kind": "latest"})
        return path

    def save_best(self, step: int, use_ema: bool) -> str:
        path = os.path.join(self.out_dir, "ckpt_best.pt")
        ctx = self.ema.swapped(self.raw_model) if (use_ema and self.ema) else contextlib.nullcontext()
        with ctx:
            save_checkpoint(path, self.raw_model, None, self.cfg, step, self.best_val_loss,
                            extra={"tokens_seen": self.tokens_seen, "best_source": self.best_source,
                                   "weights_only": True})
        self._emit("checkpoint", {"path": path, "step": step, "kind": "best"})
        return path

    def _resume(self, path: str) -> int:
        ckpt = load_checkpoint(path, self.device)
        bad = _cfg_mismatch(ckpt["config"], self.cfg.to_dict())
        if bad:
            raise ValueError(f"cannot resume from {path}: model config differs from the checkpoint's:\n  "
                             + "\n  ".join(bad))
        self.raw_model.load_state_dict(ckpt["model_state_dict"])
        if ckpt.get("optimizer_state_dict") is None:
            raise ValueError(f"{path} is a weights-only checkpoint; it has no optimizer state to resume from. "
                             f"Resume from ckpt_latest.pt, or start a new run initialised from these weights.")
        try:
            self.optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        except (ValueError, KeyError, RuntimeError) as e:
            raise ValueError(f"optimizer state in {path} does not match tc.optimizer={self.tc.optimizer!r}: {e}") from e
        if ckpt.get("scaler_state_dict"):
            self.scaler.load_state_dict(ckpt["scaler_state_dict"])
        if self.ema is not None and ckpt.get("ema_state_dict"):
            self.ema.load_state_dict(ckpt["ema_state_dict"])
        restore_rng_state(ckpt.get("rng_state"), self.raw_model, self.ds)
        self.best_val_loss = float(ckpt["best_val_loss"])
        self.best_source = ckpt.get("best_source", "raw")
        self.tokens_seen = int(ckpt.get("tokens_seen", 0))
        self.skipped_steps = int(ckpt.get("skipped_steps", 0))
        # v1 stored the step just completed; v2 stores the number of completed steps.
        return int(ckpt["step"]) + (1 if ckpt.get("format_version", 1) == 1 else 0)

    # ----------------------------------------------------------------- sampling
    def sample_text(self, use_ema: bool = False) -> str:
        tc = self.tc
        tok = ByteTokenizer()
        ids = [ByteTokenizer.BOS] + tok.encode(tc.sample_prompt)
        idx = torch.tensor([ids], dtype=torch.long, device=self.device)
        amp = amp_settings(self.dtype, self.device)
        ctx = self.ema.swapped(self.raw_model) if (use_ema and self.ema) else contextlib.nullcontext()
        devices = [torch.device(self.device)] if self.device_type == "cuda" else []
        with torch.random.fork_rng(devices=devices), ctx:  # sampling must not disturb training RNG
            torch.manual_seed(1234)
            with torch.autocast(device_type=amp["autocast_device"], dtype=amp["amp_dtype"], enabled=amp["use_amp"]):
                out = self.raw_model.generate(idx, tc.sample_len, temperature=0.8, top_k=40, min_p=0.02,
                                              stop_on_eos=True)
        return tok.decode(out[0].tolist())

    # --------------------------------------------------------------------- fit
    def fit(self, resume: bool = True) -> Dict[str, Any]:
        tc, plan = self.tc, self.plan
        # ---- distributed / device setup
        self.is_ddp = int(os.environ.get("RANK", -1)) != -1
        if self.is_ddp:
            from torch.nn.parallel import DistributedDataParallel as DDP
            cuda = torch.cuda.is_available()
            dist.init_process_group(backend="nccl" if cuda else "gloo")
            self.rank, self.world = int(os.environ["RANK"]), int(os.environ["WORLD_SIZE"])
            local = int(os.environ.get("LOCAL_RANK", 0))
            self.device = f"cuda:{local}" if cuda else "cpu"
            if cuda:
                torch.cuda.set_device(self.device)
        else:
            self.rank, self.world = 0, 1
            self.device = pick_device(self.device_arg)
        self.master = self.rank == 0
        self.device_type = device_type_of(self.device)
        self.dtype = pick_dtype(tc.dtype, self.device)
        amp = amp_settings(self.dtype, self.device)
        set_fast_matmul()

        torch.manual_seed(tc.seed + self.rank)
        self.raw_model.seed_layerdrop(tc.seed)  # identical on every rank -> identical skip decisions
        os.makedirs(self.out_dir, exist_ok=True)
        self.model.to(self.device)
        T = tc.block_size or self.cfg.block_size
        if T > self.cfg.block_size:
            raise ValueError(f"training block_size {T} exceeds the model's block_size {self.cfg.block_size}")
        if self.master:  # lets a re-run in a fresh session reuse exactly this schedule (see load_run_plan)
            with open(os.path.join(self.out_dir, "run_plan.json"), "w") as fh:
                json.dump({"plan": plan.to_dict(), "train_config": tc.persistent_dict(),
                           "model_config": self.cfg.to_dict()}, fh, indent=2)
        self.ds = ByteDataset(self.data_dirs, self.data_weights, seed=tc.seed, rank=self.rank)
        self.scaler = torch.amp.GradScaler("cuda", enabled=amp["use_scaler"])
        self.optimizer = self._build_optimizer()
        base_lrs = [g["lr"] for g in self.optimizer.param_groups]  # BEFORE any state is loaded (see module doc)
        floor = tc.min_lr / tc.learning_rate if tc.learning_rate else 0.0
        if tc.ema_decay > 0:
            self.ema = EMA(self.raw_model, tc.ema_decay)

        start_step = 0
        latest = os.path.join(self.out_dir, "ckpt_latest.pt")
        if resume and os.path.exists(latest):
            start_step = self._resume(latest)
            if self.ema is not None and self.ema.num_updates == 0:
                self.ema = EMA(self.raw_model, tc.ema_decay)  # checkpoint had no EMA: start averaging now
            self._print(f"[resume] {latest} -> continuing at step {start_step} "
                        f"({self.tokens_seen / 1e6:.1f}M tokens seen, best_val {self.best_val_loss:.4f})")
        else:
            if self.init_optimizer is not None:
                self.init_optimizer(self.optimizer)
            if resume:
                self._print(f"[resume] no checkpoint at {latest}; starting from scratch")

        compiled = self.model
        if tc.compile:
            try:
                compiled = torch.compile(self.model)
                self._print("[train] torch.compile enabled")
            except Exception as e:  # noqa: BLE001
                self._print(f"[train] torch.compile unavailable ({e}); continuing eagerly")
        if self.is_ddp:
            ids = [torch.device(self.device).index] if self.device_type == "cuda" else None
            compiled = DDP(compiled, device_ids=ids, find_unused_parameters=self.cfg.layerdrop_max > 0)
        run_model = compiled  # what forward/backward go through; self.model stays the plain module

        eval_batches = self.ds.fixed_batches("val", tc.eval_batches, tc.eval_batch_size, T, self.device)
        self._log_file = open(os.path.join(self.out_dir, "train_log.jsonl"), "a") if self.master else None
        info = gpu_info()
        tokens_per_step = tc.batch_size * tc.grad_accum_steps * T * self.world
        if tokens_per_step != plan.tokens_per_step:
            self._print(f"[train] note: plan assumed {plan.tokens_per_step:,} tokens/step, this config does "
                        f"{tokens_per_step:,}; the schedule is defined in steps, tokens are reported as run")
        self._print(f"[train] {self.cfg.describe()}")
        self._print(f"[train] device={self.device} dtype={self.dtype} optimizer={tc.optimizer} "
                    f"steps={plan.max_steps:,} ({plan.total_tokens / 1e6:.0f}M tokens plan) "
                    f"batch={tc.batch_size}x{tc.grad_accum_steps}x{T}")
        self._emit("start", {"step": start_step, "plan": plan.to_dict(), "tc": tc.to_dict()})

        status = "completed"
        t_start = time.time()
        t_log, tok_log = t_start, self.tokens_seen
        t_ckpt = t_start
        n_evals = 0
        consecutive_bad = 0
        step = start_step
        last_val: Dict[str, float] = {}
        self.model.train()
        self.optimizer.zero_grad(set_to_none=True)
        try:
            while step < plan.max_steps:
                mult = plan.multiplier(step, floor)
                for g, b in zip(self.optimizer.param_groups, base_lrs):
                    g["lr"] = b * mult

                sums: Dict[str, torch.Tensor] = {}
                for micro in range(tc.grad_accum_steps):
                    x, y = self.ds.get_batch("train", tc.batch_size, T, self.device)
                    sync = run_model.no_sync() if (self.is_ddp and micro < tc.grad_accum_steps - 1) \
                        else contextlib.nullcontext()
                    with sync:
                        with torch.autocast(device_type=amp["autocast_device"], dtype=amp["amp_dtype"],
                                            enabled=amp["use_amp"]):
                            _, losses, _ = run_model(x, y)
                            loss = losses["total_loss"]
                            if self.extra_loss is not None:
                                extra = self.extra_loss(self.raw_model)
                                losses = dict(losses, extra_loss=extra)
                                loss = loss + extra
                        scaled = loss / tc.grad_accum_steps
                        (self.scaler.scale(scaled) if self.scaler.is_enabled() else scaled).backward()
                    for k, v in losses.items():
                        sums[k] = sums.get(k, 0.0) + v.detach().float() / tc.grad_accum_steps

                if self.scaler.is_enabled():
                    self.scaler.unscale_(self.optimizer)
                gnorm = (torch.nn.utils.clip_grad_norm_(self.raw_model.parameters(), tc.grad_clip)
                         if tc.grad_clip > 0 else None)
                probe = gnorm if gnorm is not None else sums["total_loss"]
                finite = bool(torch.isfinite(probe))
                if self.scaler.is_enabled():
                    self.scaler.step(self.optimizer)   # skips itself on inf/nan gradients
                    self.scaler.update()
                elif finite:
                    self.optimizer.step()
                self.optimizer.zero_grad(set_to_none=True)
                if finite:
                    consecutive_bad = 0
                    if self.ema is not None:
                        self.ema.update(self.raw_model)
                else:
                    consecutive_bad += 1
                    self.skipped_steps += 1
                    # fp16 + GradScaler legitimately skips a few steps while the loss scale calibrates
                    if consecutive_bad > tc.max_nonfinite:
                        raise FloatingPointError(
                            f"{consecutive_bad} consecutive non-finite steps at step {step}; training has diverged. "
                            f"Lower the learning rate or check the data.")
                self.tokens_seen += tokens_per_step
                self.step = step + 1

                # ---- logging
                if self.master and (step % tc.log_interval == 0 or step == plan.max_steps - 1):
                    now = time.time()
                    tps = (self.tokens_seen - tok_log) / (now - t_log) if step > start_step and now > t_log else None
                    rec = {"step": step, "lr": self.optimizer.param_groups[0]["lr"], "lr_mult": mult,
                           "tokens": self.tokens_seen, "elapsed": now - t_start,
                           "grad_norm": float(gnorm) if gnorm is not None else None,
                           "skipped": self.skipped_steps, "tok_per_s": tps,
                           "mfu": mfu(self.cfg, tps, T, info["peak_tflops"]) if tps else None}
                    rec.update({k: float(v) for k, v in sums.items()})
                    t_log, tok_log = now, self.tokens_seen
                    self._write_log(rec)
                    self._emit("log", rec)
                    extra_s = f" conf {rec['confidence_loss']:.3f}" if "confidence_loss" in rec else ""
                    self._print(f"step {step:6d} | lr {rec['lr']:.2e} | loss {rec['total_loss']:.4f} "
                                f"(ce {rec['ce_loss']:.4f} = {rec['ce_loss'] / math.log(2):.3f} bpb{extra_s}) "
                                f"| gnorm {rec['grad_norm'] if rec['grad_norm'] is not None else float('nan'):.2f} "
                                f"| {(tps or 0) / 1e3:.1f}k tok/s | {rec['elapsed'] / 60:.1f} min")

                # ---- evaluation + checkpoint
                done_step = step + 1
                is_last = done_step >= plan.max_steps
                stop_now = False
                if tc.max_minutes is not None and (time.time() - t_start) / 60.0 >= tc.max_minutes:
                    stop_now = True
                if self.is_ddp:
                    flag = torch.tensor([1.0 if stop_now else 0.0], device=self.device)
                    dist.all_reduce(flag, op=dist.ReduceOp.MAX)
                    stop_now = bool(flag.item())
                do_eval = (done_step % tc.eval_interval == 0) or is_last or stop_now
                if do_eval and self.master:
                    last_val = self._eval_and_maybe_save_best(eval_batches, done_step)
                    n_evals += 1
                    if tc.sample_every_evals and n_evals % tc.sample_every_evals == 0:
                        text = self.sample_text(use_ema=last_val.get("source") == "ema")
                        self._emit("sample", {"step": done_step, "text": text})
                        if tc.verbose:
                            print("      sample> " + repr(text[:160]))
                    self.save_latest(done_step)
                    t_ckpt = time.time()
                elif self.master and (time.time() - t_ckpt) / 60.0 >= tc.ckpt_interval_minutes:
                    self.save_latest(done_step)
                    t_ckpt = time.time()
                step = done_step
                if stop_now and not is_last:
                    status = "time_limit"
                    self._print(f"[train] wall-clock limit ({tc.max_minutes} min) reached at step {step}; "
                                f"checkpoint saved. Re-run to continue.")
                    break
        finally:
            if self._log_file is not None:
                self._log_file.close()
            if self.is_ddp:
                dist.destroy_process_group()

        result = {"status": status, "step": step, "best_val_loss": self.best_val_loss,
                  "best_source": self.best_source, "tokens_seen": self.tokens_seen,
                  "elapsed": time.time() - t_start, "last_val": last_val, "skipped_steps": self.skipped_steps}
        self._emit("end", result)
        if status == "completed":
            self._print(f"[train] done: step {step}, best val loss {self.best_val_loss:.4f} "
                        f"({self.best_val_loss / math.log(2):.3f} bpb, from {self.best_source} weights)")
        return result

    def _eval_and_maybe_save_best(self, eval_batches, done_step: int) -> Dict[str, Any]:
        raw = evaluate_batches(self.raw_model, eval_batches, self.device, self.dtype)
        rec: Dict[str, Any] = {"step": done_step, "val_loss": raw["loss"], "val_bpb": raw["bpb"],
                               "val_ppl": raw["ppl"], "tokens": self.tokens_seen}
        best_loss, source = raw["loss"], "raw"
        if self.ema is not None:
            with self.ema.swapped(self.raw_model):
                e = evaluate_batches(self.raw_model, eval_batches, self.device, self.dtype)
            rec.update(val_loss_ema=e["loss"], val_bpb_ema=e["bpb"])
            if e["loss"] < best_loss:
                best_loss, source = e["loss"], "ema"
        rec["source"] = source
        self._write_log(rec)
        self._emit("eval", rec)
        msg = f"step {done_step:6d} | val {raw['loss']:.4f} ({raw['bpb']:.3f} bpb)"
        if self.ema is not None:
            msg += f" | ema {rec['val_loss_ema']:.4f} ({rec['val_bpb_ema']:.3f} bpb)"
        self._print(msg)
        if best_loss < self.best_val_loss:
            self.best_val_loss, self.best_source = best_loss, source
            self.save_best(done_step, use_ema=(source == "ema"))
            self._print(f"      new best val loss {best_loss:.4f} ({source}) -> ckpt_best.pt")
        return rec

In [ ]:
%%writefile tensegrity_src/tensegrity/growth.py
"""Function-preserving model growth (Net2Net-style) and continual-learning tools (EWC).

``grow_model`` turns a trained small model into a larger one that computes *exactly the same
function* at the moment of growth -- and, unlike v1, whose new capacity can actually learn.

Why v1 growth along ``n_embd`` was a trap
-----------------------------------------
Every new weight region was zero-filled on BOTH sides of a product (new head: q=k=v=o=0; new
hidden unit: gate=up=down=0; new residual channel: every reader AND every writer = 0). A
product of two zeros has zero gradient w.r.t. both factors, forever, so the grown model matched
the old logits yet could never use its extra width (verified: every new region had exactly 0.0
gradient). The rule used here is *one factor zero, the other random*:

  * WRITERS into the residual stream (``o_proj`` columns/rows, ``w_down`` columns/rows, the
    embedding's new channels) are zero  -> the old function is untouched, and
  * READERS (q/k/v, gate/up input columns, new-head projections) keep a small random init
    -> the zero writers receive non-zero gradient on the first step.

RMSNorm statistics are kept exact by zero residual channels + the sqrt(old/new) weight rescale
(and the matching ``norm_eps`` rescale, stored in the config so it survives a save/load).
"""

from __future__ import annotations

import copy
import math
from typing import Dict, List, Optional, Tuple, Union

import torch
import torch.nn as nn

from .config import TensegrityConfig
from .data import ByteDataset
from .model import TensegrityLM

# Fields whose change would alter the computed function (or the parameter layout) in a way
# growth cannot compensate for. Everything else (dropout, loss weights, LayerDrop, init_std,
# checkpointing knobs, ...) is free to differ between rungs.
_LOCKED = (
    "vocab_size", "sliding_window", "global_every", "rope_base", "bias", "use_qk_norm",
    "embed_scale", "logit_soft_cap", "use_confidence_head", "byte_mix_kernel",
    "byte_mix_bilinear", "x0_mix", "v0_mix",
)


def _check_compatible(old: TensegrityConfig, new: TensegrityConfig) -> None:
    errs = []
    for f in _LOCKED:
        if getattr(old, f) != getattr(new, f):
            errs.append(f"{f}: {getattr(old, f)!r} -> {getattr(new, f)!r} (changes the function)")
    if old.head_dim != new.head_dim:
        errs.append(f"head_dim: {old.head_dim} -> {new.head_dim} (RoPE frequencies depend on head_dim; "
                    f"grow by adding heads, not by widening them)")
    if old.gqa_ratio != new.gqa_ratio:
        errs.append(f"GQA ratio: {old.gqa_ratio} -> {new.gqa_ratio} (would re-map query heads to KV groups)")
    for f in ("n_layer", "n_loops", "n_embd", "n_head", "n_kv_head", "ffn_hidden", "block_size"):
        if getattr(new, f) < getattr(old, f):
            errs.append(f"{f}: {getattr(old, f)} -> {getattr(new, f)} (growth cannot shrink)")
    if errs:
        raise ValueError("incompatible growth target:\n  " + "\n  ".join(errs))


def retarget(old: TensegrityConfig, size_from: TensegrityConfig, keep_context: bool = True) -> TensegrityConfig:
    """Config for growing ``old`` towards the *size* of ``size_from`` (e.g. the next preset).

    Takes the width / depth / heads / FFN from ``size_from`` but keeps every function-defining
    field of ``old`` (attention pattern, RoPE base, norms, ByteMix, ...), so a model trained with a
    customised window or context still grows cleanly into a rung of the ladder. ``keep_context``
    keeps ``old.block_size`` (pass False to adopt ``size_from``'s context length)."""
    keep = {f: getattr(old, f) for f in _LOCKED}
    if keep_context:
        keep["block_size"] = old.block_size
    if size_from.n_loops < old.n_loops:
        keep["n_loops"] = old.n_loops
    d = size_from.to_dict()
    d.update(keep)
    d["gain_init"] = None
    return TensegrityConfig(**d)


def derive_target(old: TensegrityConfig, new_n_layer: Optional[int] = None,
                  ffn_growth_mult: Optional[float] = None, new_n_embd: Optional[int] = None,
                  new_n_head: Optional[int] = None, new_n_kv_head: Optional[int] = None,
                  new_block_size: Optional[int] = None, new_n_loops: Optional[int] = None) -> TensegrityConfig:
    """Build the target config from the v1-style keyword arguments."""
    d = old.to_dict()
    if new_n_layer is not None:
        d["n_layer"] = new_n_layer
        d["gain_init"] = None if new_n_layer != old.n_layer else old.gain_init
    if new_n_loops is not None:
        d["n_loops"] = new_n_loops
        if new_n_loops != old.n_loops and new_n_layer is None:
            d["gain_init"] = None
    if ffn_growth_mult is not None:
        if ffn_growth_mult < 1.0:
            raise ValueError("ffn_growth_mult must be >= 1.0")
        d["ffn_mult"] = old.ffn_mult * ffn_growth_mult
    if new_n_embd is not None or new_n_head is not None:
        hd = old.head_dim
        n_embd = new_n_embd if new_n_embd is not None else new_n_head * hd
        if n_embd % hd != 0:
            raise ValueError(f"new_n_embd ({n_embd}) must be a multiple of head_dim ({hd}); e.g. "
                             f"{hd * (n_embd // hd + 1)}")
        implied = n_embd // hd
        if new_n_head is not None and new_n_head != implied:
            raise ValueError(f"new_n_head={new_n_head} would change head_dim (only {implied} keeps "
                             f"head_dim={hd}); omit new_n_head to have it inferred")
        d["n_embd"], d["n_head"] = n_embd, implied
        d["n_kv_head"] = new_n_kv_head if new_n_kv_head is not None else max(1, implied // old.gqa_ratio)
    if new_block_size is not None:
        d["block_size"] = new_block_size
    return TensegrityConfig(**d)


@torch.no_grad()
def _randomize_like_init(t: torch.Tensor, std: float) -> None:
    t.normal_(mean=0.0, std=std)


@torch.no_grad()
def _reader_std(w_old: torch.Tensor, min_std: float) -> float:
    return max(min_std, float(w_old.float().pow(2).mean().sqrt()))


@torch.no_grad()
def _grow_block(ob: nn.Module, nb: nn.Module, oc: TensegrityConfig, nc: TensegrityConfig) -> None:
    D0, D1 = oc.n_embd, nc.n_embd
    hd = oc.head_dim
    q0, kv0 = oc.n_head * hd, oc.n_kv_head * hd
    h0 = oc.ffn_hidden
    std = nc.init_std
    scale = (D0 / D1) ** 0.5
    wide = D1 > D0

    # ---- norms: sqrt(old/new) keeps x/rms(x) * w exact when the new channels are zero
    for name in ("ln1", "ln2"):
        w_new, w_old = getattr(nb, name).weight, getattr(ob, name).weight
        w_new[:D0].copy_(w_old * (scale if wide else 1.0))
    # ---- residual gains (embed_scale makes the stream sqrt(D1/D0) larger -> branches must follow)
    gain_scale = (D1 / D0) ** 0.5 if (wide and oc.embed_scale) else 1.0
    nb.attn_gain.copy_(ob.attn_gain * gain_scale)
    nb.ffn_gain.copy_(ob.ffn_gain * gain_scale)
    if ob.x0_gain is not None:
        nb.x0_gain.copy_(ob.x0_gain)  # x0 and the stream scale together: no extra factor
    if nb.loop_attn_gains is not None:
        nb.loop_attn_gains.zero_()
        nb.loop_ffn_gains.zero_()
        if nb.loop_x0_gains is not None:
            nb.loop_x0_gains.zero_()
        if ob.loop_attn_gains is not None:
            n_old = ob.loop_attn_gains.numel()
            nb.loop_attn_gains[:n_old].copy_(ob.loop_attn_gains * gain_scale)
            nb.loop_ffn_gains[:n_old].copy_(ob.loop_ffn_gains * gain_scale)
            if ob.loop_x0_gains is not None and nb.loop_x0_gains is not None:
                nb.loop_x0_gains[:n_old].copy_(ob.loop_x0_gains)

    # ---- attention.   READERS random (matched to parent weight RMS), WRITERS zero.
    a_o, a_n = ob.attn, nb.attn
    for lin in ("q_proj", "k_proj", "v_proj"):
        out0 = q0 if lin == "q_proj" else kv0
        w_old = getattr(a_o, lin).weight
        w = getattr(a_n, lin).weight
        r_std = _reader_std(w_old, std)
        if w.shape[0] > out0:
            _randomize_like_init(w[out0:, :], r_std)
        w[:out0, :D0].copy_(w_old)                           # old heads, old channels: exact copy
        if wide:                                             # old heads reading NEW channels: input is 0 at growth,
            _randomize_like_init(w[:out0, D0:], r_std)       # so random init is free and breaks the symmetry later
        if oc.bias:
            getattr(a_n, lin).bias[:out0].copy_(getattr(a_o, lin).bias)
    w_o = a_n.o_proj.weight
    w_o.zero_()                                              # writers: new heads -> all dims, all heads -> new dims
    w_o[:D0, :q0].copy_(a_o.o_proj.weight)
    if oc.bias:
        a_n.o_proj.bias.zero_()
        a_n.o_proj.bias[:D0].copy_(a_o.o_proj.bias)
    if a_o.q_norm is not None:                               # head_dim unchanged -> per-head norms copy 1:1
        a_n.q_norm.weight.copy_(a_o.q_norm.weight)
        a_n.k_norm.weight.copy_(a_o.k_norm.weight)
    if a_o.v0_gain is not None and a_n.v0_gain is not None:
        a_n.v0_gain[:oc.n_kv_head].copy_(a_o.v0_gain)

    # ---- feed-forward
    f_o, f_n = ob.ffn, nb.ffn
    for lin in ("w_gate", "w_up"):
        w_old = getattr(f_o, lin).weight
        w = getattr(f_n, lin).weight                        # (h1, D1): readers of the stream
        r_std = _reader_std(w_old, std)
        if w.shape[0] > h0:
            _randomize_like_init(w[h0:, :], r_std)
        w[:h0, :D0].copy_(w_old)
        if wide:
            _randomize_like_init(w[:h0, D0:], r_std)
        if oc.bias:
            getattr(f_n, lin).bias[:h0].copy_(getattr(f_o, lin).bias)
    w_d = f_n.w_down.weight                                  # (D1, h1): writer
    w_d.zero_()
    w_d[:D0, :h0].copy_(f_o.w_down.weight)
    if oc.bias:
        f_n.w_down.bias.zero_()
        f_n.w_down.bias[:D0].copy_(f_o.w_down.bias)


@torch.no_grad()
def grow_model(old_model: TensegrityLM, old_cfg: Optional[TensegrityConfig] = None, *,
               target: Optional[TensegrityConfig] = None,
               new_n_layer: Optional[int] = None, ffn_growth_mult: Optional[float] = None,
               new_n_embd: Optional[int] = None, new_n_head: Optional[int] = None,
               new_n_kv_head: Optional[int] = None, new_block_size: Optional[int] = None,
               new_n_loops: Optional[int] = None,
               device: Optional[Union[str, torch.device]] = None, seed: Optional[int] = 0
               ) -> Tuple[TensegrityLM, TensegrityConfig]:
    """
    Return ``(new_model, new_cfg)``: a larger model that, at the moment of growth, computes the
    same function as ``old_model`` (within float rounding) and whose new capacity is trainable.

    Pass either ``target`` (a full target :class:`TensegrityConfig`, e.g. the next preset), or
    the v1-style keywords (``new_n_layer``, ``new_n_embd``, ``ffn_growth_mult``, ...).

    Growth axes
      * depth   -- new blocks are appended with residual gains = 0 (exact no-op, ReZero-style start);
      * loops   -- extra loop passes are added with loop residual gains = 0 (exact no-op);
      * FFN     -- new hidden units: RMS-matched random gate/up (readers), zero down (writer);
      * width   -- more heads and residual channels, per the reader/writer rule in the module doc;
      * context -- ``block_size`` may grow (RoPE is relative; weights are unchanged).
    """
    old_cfg = old_cfg or old_model.cfg
    if target is None:
        target = derive_target(old_cfg, new_n_layer, ffn_growth_mult, new_n_embd, new_n_head,
                               new_n_kv_head, new_block_size, new_n_loops)
    _check_compatible(old_cfg, target)
    new_cfg = target
    device = device if device is not None else next(old_model.parameters()).device

    # RMSNorm exactness under zero-padded channels. With the stream scaled by sigma and the norm
    # weight by sqrt(D0/D1), out = x / sqrt(mean_new + eps_new) * w matches the old norm iff
    #     eps_new = eps_old * sigma^2 * D0 / D1,   sigma^2 = D1/D0 if embed_scale else 1.
    # So eps is unchanged when embed_scale is on (the sqrt(D1) embedding factor cancels the
    # dilution of the mean-square) and scales by D0/D1 when it is off. Stored in the config so a
    # saved/reloaded grown model is still exactly the same function.
    D0_, D1_ = old_cfg.n_embd, new_cfg.n_embd
    sigma2 = (D1_ / D0_) if (D1_ > D0_ and old_cfg.embed_scale) else 1.0
    new_cfg = new_cfg.replace(norm_eps=old_cfg.norm_eps * sigma2 * D0_ / D1_)

    if seed is not None:
        torch.manual_seed(seed)  # reproducible random init of the new (reader) regions
    new_model = TensegrityLM(new_cfg).to(device)
    old_model = old_model.to(device)
    D0, D1 = old_cfg.n_embd, new_cfg.n_embd
    scale = (D0 / D1) ** 0.5

    # ---- embedding (tied lm_head), final norm, confidence head, byte-mix
    emb = new_model.token_emb.weight
    emb.zero_()                                              # new channels = zero writers
    emb[:, :D0].copy_(old_model.token_emb.weight)
    new_model.ln_f.weight[:D0].copy_(old_model.ln_f.weight * (scale if D1 > D0 else 1.0))
    if old_model.confidence_head is not None:
        ch_n, ch_o = new_model.confidence_head, old_model.confidence_head
        ch_n.weight.zero_()                                  # single-factor zero: x_new becomes non-zero via writers
        ch_n.weight[:, :D0].copy_(ch_o.weight)
        ch_n.bias.copy_(ch_o.bias)
    if old_model.byte_mix is not None:
        new_model.byte_mix.weight.zero_()
        new_model.byte_mix.weight[:, :D0].copy_(old_model.byte_mix.weight)
        if old_model.byte_mix.bi_weight is not None and new_model.byte_mix.bi_weight is not None:
            new_model.byte_mix.bi_weight.zero_()
            new_model.byte_mix.bi_weight[:, :D0].copy_(old_model.byte_mix.bi_weight)

    # ---- blocks
    for i in range(old_cfg.n_layer):
        _grow_block(old_model.blocks[i], new_model.blocks[i], old_cfg, new_cfg)
    ref_blk = old_model.blocks[-1]
    for i in range(old_cfg.n_layer, new_cfg.n_layer):        # appended blocks: exact no-ops at init, RMS-matched weights
        blk = new_model.blocks[i]
        for sub_o, sub_n, names in (
            (ref_blk.attn, blk.attn, ("q_proj", "k_proj", "v_proj", "o_proj")),
            (ref_blk.ffn, blk.ffn, ("w_gate", "w_up", "w_down")),
        ):
            for lin_name in names:
                wo = getattr(sub_o, lin_name).weight
                wn = getattr(sub_n, lin_name).weight
                _randomize_like_init(wn, _reader_std(wo, new_cfg.init_std))
        blk.attn_gain.zero_()
        blk.ffn_gain.zero_()
        if blk.x0_gain is not None:
            blk.x0_gain.zero_()
        if blk.loop_attn_gains is not None:
            blk.loop_attn_gains.zero_()
            blk.loop_ffn_gains.zero_()
            if blk.loop_x0_gains is not None:
                blk.loop_x0_gains.zero_()
    return new_model, new_cfg


@torch.no_grad()
def grow_optimizer_state(old_model: nn.Module, old_opt_or_state: Any,
                         new_model: nn.Module, new_opt: torch.optim.Optimizer) -> int:
    """Transplant optimizer state (Muon ``momentum_buffer``, AdamW ``exp_avg`` / ``exp_avg_sq`` /
    ``step``) from ``old_model`` onto ``new_opt`` (built for ``new_model``).

    ``old_opt_or_state`` may be either a live :class:`torch.optim.Optimizer` or a saved
    ``optimizer_state_dict`` (e.g. from ``ckpt_latest.pt``). Newly grown slices of ``exp_avg_sq``
    are initialised to the mean second moment of the parent parameter so new channels/units do not
    take an uncalibrated step on the first update. Returns the number of transplanted parameters.
    """
    if old_opt_or_state is None:
        return 0
    old_by_name: Dict[str, Dict[str, Any]] = {}
    if isinstance(old_opt_or_state, torch.optim.Optimizer):
        for n, p in old_model.named_parameters():
            if p in old_opt_or_state.state and old_opt_or_state.state[p]:
                old_by_name[n] = old_opt_or_state.state[p]
    elif isinstance(old_opt_or_state, dict) and "state" in old_opt_or_state and "param_groups" in old_opt_or_state:
        # Reconstruct the flat parameter ordering configure_optimizer produces on old_model
        use_muon = any(g.get("use_muon", False) for g in old_opt_or_state["param_groups"])
        dummy = old_model.configure_optimizer(0.1, 1e-3, (0.9, 0.95), "cpu", use_muon=use_muon)
        id_to_name = {id(p): n for n, p in old_model.named_parameters()}
        ordered_names = [id_to_name[id(p)] for g in dummy.param_groups for p in g["params"]]
        state_map = old_opt_or_state["state"]
        for idx, name in enumerate(ordered_names):
            st = state_map.get(idx, state_map.get(str(idx)))
            if st:
                old_by_name[name] = st
    else:
        return 0

    count = 0
    for name, np_ in new_model.named_parameters():
        ost = old_by_name.get(name)
        if not ost:
            continue
        nst = new_opt.state[np_]
        if "momentum_buffer" in ost:
            mb_old = ost["momentum_buffer"]
            buf = torch.zeros_like(np_)
            sl = tuple(slice(0, min(s, np_.shape[i])) for i, s in enumerate(mb_old.shape))
            buf[sl].copy_(mb_old[sl].to(np_.device, np_.dtype))
            nst["momentum_buffer"] = buf
            count += 1
        elif "exp_avg" in ost and "exp_avg_sq" in ost:
            ea_old, eas_old = ost["exp_avg"], ost["exp_avg_sq"]
            ea = torch.zeros_like(np_)
            eas = torch.full_like(np_, float(eas_old.float().mean()))
            sl = tuple(slice(0, min(s, np_.shape[i])) for i, s in enumerate(ea_old.shape))
            ea[sl].copy_(ea_old[sl].to(np_.device, np_.dtype))
            eas[sl].copy_(eas_old[sl].to(np_.device, np_.dtype))
            nst["exp_avg"] = ea
            nst["exp_avg_sq"] = eas
            step_val = ost.get("step", 0)
            nst["step"] = int(step_val.item()) if isinstance(step_val, torch.Tensor) else int(step_val)
            count += 1
    return count


# =============================================================================
# Verification
# =============================================================================

@torch.no_grad()
def check_function_preserved(old_model: TensegrityLM, new_model: TensegrityLM, cfg: Optional[TensegrityConfig] = None,
                             device: Optional[str] = None, n_tokens: int = 48, batch: int = 2,
                             atol: float = 1e-4, verbose: bool = True) -> float:
    """Feed identical random byte sequences through both models (eval, fp32, TF32 off) and
    require the logits (and confidence scores) to agree within ``atol``. Raises on failure."""
    cfg = cfg or old_model.cfg
    device = device or str(next(old_model.parameters()).device)
    was = (old_model.training, new_model.training)
    old_model.eval()
    new_model.eval()
    prev = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    torch.backends.cuda.matmul.allow_tf32 = False  # TF32 rounding alone can exceed a tight atol
    torch.backends.cudnn.allow_tf32 = False
    try:
        g = torch.Generator().manual_seed(0)
        x = torch.randint(0, 256, (batch, min(n_tokens, cfg.block_size)), generator=g).to(device)
        lo, _, co = old_model(x)
        ln, _, cn = new_model(x)
        diff = float((lo - ln).abs().max())
        if co is not None and cn is not None:
            diff = max(diff, float((co - cn).abs().max()))
        ok = diff < atol
        if verbose:
            print(f"[grow] max |old - new| over logits/confidence = {diff:.3e} ({'PASS' if ok else 'FAIL'}, atol={atol:.0e})")
        if not ok:
            raise AssertionError("grow_model() did not preserve the model's function; do not train this model.")
        return diff
    finally:
        torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32 = prev
        old_model.train(was[0])
        new_model.train(was[1])


def _new_regions(old: TensegrityConfig, name: str, shape: Tuple[int, ...], layer: int) -> List[torch.Tensor]:
    """Boolean masks of the entries of ``name`` that did not exist in the old model AND are
    expected to receive gradient. Entries that *read* the new residual channels are excluded:
    their input is exactly zero at growth time (that is what keeps the function unchanged), so
    their gradient is legitimately zero until a writer has moved."""
    full = lambda: torch.ones(shape, dtype=torch.bool)  # noqa: E731
    if layer >= old.n_layer:
        return [full()] if len(shape) >= 2 else []
    hd, D0 = old.head_dim, old.n_embd
    q0, kv0, h0 = old.n_head * hd, old.n_kv_head * hd, old.ffn_hidden
    regions = []
    if name.endswith(("attn.q_proj.weight", "attn.k_proj.weight", "attn.v_proj.weight")):
        out0 = q0 if name.endswith("q_proj.weight") else kv0
        m = torch.zeros(shape, dtype=torch.bool)
        m[out0:, :D0] = True                        # new heads reading old channels
        regions.append(m)
    elif name.endswith("attn.o_proj.weight"):
        m = torch.zeros(shape, dtype=torch.bool)
        m[:, q0:] = True                            # new heads writing
        regions.append(m)
        m = torch.zeros(shape, dtype=torch.bool)
        m[D0:, :] = True                            # writing into new channels
        regions.append(m)
    elif name.endswith(("ffn.w_gate.weight", "ffn.w_up.weight")):
        m = torch.zeros(shape, dtype=torch.bool)
        m[h0:, :D0] = True                          # new hidden units reading old channels
        regions.append(m)
    elif name.endswith("ffn.w_down.weight"):
        m = torch.zeros(shape, dtype=torch.bool)
        m[:, h0:] = True                            # new hidden units writing
        regions.append(m)
        m = torch.zeros(shape, dtype=torch.bool)
        m[D0:, :] = True                            # writing into new channels
        regions.append(m)
    return [r for r in regions if r.any()]


def check_growth_trainable(new_model: TensegrityLM, old_cfg: TensegrityConfig, x: Optional[torch.Tensor] = None,
                           y: Optional[torch.Tensor] = None, probe_step: float = 0.05,
                           verbose: bool = True) -> Dict[str, float]:
    """
    Prove the new capacity can learn. On a throw-away copy of ``new_model`` this runs two
    forward/backward passes with one normalised SGD step in between, and requires that every
    region added by growth (new heads, new hidden units, new residual channels, new blocks)
    receives a non-zero gradient in at least one pass.

    Two passes are needed because some gradients are *legitimately* zero at the instant of
    growth -- a ReZero-style block with gain 0 only wakes its inner weights once the gain has
    moved; the last block's writers into new channels only get signal once an earlier block has
    written into them. A *dead* region (the v1 width-growth failure: a product of two zeros) stays
    at exactly 0 in both passes. Raises AssertionError listing dead regions. Does not modify
    ``new_model``.
    """
    new = new_model.cfg
    probe = copy.deepcopy(new_model)
    dev = next(probe.parameters()).device
    if x is None:
        g = torch.Generator().manual_seed(1)
        x = torch.randint(0, 256, (4, min(32, new.block_size)), generator=g).to(dev)
        y = torch.randint(0, 256, (4, x.shape[1]), generator=g).to(dev)
    probe.train()
    probe._drop_ps = [0.0] * len(probe._drop_ps)
    probe.grad_checkpoint = False

    best: Dict[str, float] = {}

    def record() -> None:
        for name, p in probe.named_parameters():
            if p.grad is None:
                continue
            if name.startswith("blocks."):
                layer = int(name.split(".")[1])
                if layer >= old_cfg.n_layer and name.endswith(("attn_gain", "ffn_gain")):
                    best[name] = max(best.get(name, 0.0), float(p.grad.abs().max()))
                    continue
                if new.n_loops > old_cfg.n_loops and name.endswith(("loop_attn_gains", "loop_ffn_gains")):
                    best[name] = max(best.get(name, 0.0), float(p.grad[old_cfg.n_loops - 1:].abs().max()))
                    continue
                for k, mask in enumerate(_new_regions(old_cfg, name, tuple(p.shape), layer)):
                    key = f"{name}#{k}"
                    best[key] = max(best.get(key, 0.0), float(p.grad[mask.to(p.grad.device)].abs().max()))
            elif name == "token_emb.weight" and new.n_embd > old_cfg.n_embd:
                best[name] = max(best.get(name, 0.0), float(p.grad[:, old_cfg.n_embd:].abs().max()))

    for pass_idx in range(2):
        probe.zero_grad(set_to_none=True)
        _, losses, _ = probe(x, y)
        losses["total_loss"].backward()
        record()
        if pass_idx == 0:
            gn = math.sqrt(sum(float(p.grad.float().pow(2).sum()) for p in probe.parameters() if p.grad is not None))
            with torch.no_grad():
                for p in probe.parameters():
                    if p.grad is not None:
                        p.add_(p.grad, alpha=-probe_step / (gn + 1e-12))
    dead = sorted(k for k, v in best.items() if v == 0.0)
    if verbose:
        print(f"[grow] trainability: {len(best)} new regions probed, "
              f"min max-|grad| = {min(best.values()) if best else float('nan'):.3e}, dead = {len(dead)}")
    if dead:
        raise AssertionError("grown model has regions that can never learn (gradient stays exactly 0): "
                             + ", ".join(dead[:8]))
    return best


# =============================================================================
# Elastic Weight Consolidation (continual learning)
# =============================================================================

def compute_fisher_diagonal(model: TensegrityLM, data_dir: str, device: Optional[str] = None,
                            n_sequences: int = 64, seq_len: Optional[int] = None, dtype: str = "auto",
                            seed: int = 0) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    """
    Diagonal *empirical* Fisher: the mean over sequences of the squared per-sequence gradient of
    the cross-entropy. (v1 squared the gradient of an 8-sequence batch mean, which underestimates
    the Fisher by roughly the batch size; here every sequence gets its own backward pass.)
    Gradients are computed in fp32 for stability. Returns ``(fisher, params_snapshot)``.
    """
    cfg = model.cfg
    device = device or str(next(model.parameters()).device)
    seq_len = seq_len or cfg.block_size
    ds = ByteDataset(data_dir, seed=seed)
    batches = ds.fixed_batches("train", n_sequences, 1, seq_len, device, seed=seed)
    was_training = model.training
    model.eval()
    fisher = {n: torch.zeros_like(p, dtype=torch.float32) for n, p in model.named_parameters() if p.requires_grad}
    snapshot = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
    for x, y in batches:
        model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            _, losses, _ = model(x, y)
            losses["ce_loss"].backward()
        for n, p in model.named_parameters():
            if p.grad is not None:
                fisher[n] += p.grad.detach().float().pow(2)
    for n in fisher:
        fisher[n] /= float(len(batches))
    model.zero_grad(set_to_none=True)
    model.train(was_training)
    return fisher, snapshot


@torch.no_grad()
def expand_reference_state(old_fisher: Dict[str, torch.Tensor], old_params: Dict[str, torch.Tensor],
                           new_model: nn.Module) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    """Re-map an (old_fisher, old_params) pair onto a grown model: carried-over entries keep
    their Fisher and anchor; new entries get Fisher 0 and anchor = their current value, so EWC
    puts no constraint on the new capacity."""
    if not old_fisher or not old_params:
        return {}, {}
    new_f, new_p = {}, {}
    for name, p in new_model.named_parameters():
        if name not in old_fisher:
            continue
        of, op = old_fisher[name], old_params[name]
        if of.shape == p.shape:
            new_f[name], new_p[name] = of.to(p.device), op.to(p.device, p.dtype)
        else:
            nf = torch.zeros_like(p, dtype=torch.float32)
            nop = p.detach().clone()
            sl = tuple(slice(0, min(s, p.shape[i])) for i, s in enumerate(of.shape))
            nf[sl] = of[sl].to(p.device)
            nop[sl] = op[sl].to(p.device, p.dtype)
            new_f[name], new_p[name] = nf, nop
    return new_f, new_p


def ewc_penalty(model: nn.Module, fisher: Dict[str, torch.Tensor], old_params: Dict[str, torch.Tensor]) -> torch.Tensor:
    """``sum_i F_i (theta_i - theta*_i)^2`` over parameters present in ``fisher``."""
    total = None
    for name, p in model.named_parameters():
        f = fisher.get(name)
        if f is None:
            continue
        term = (f * (p.float() - old_params[name].float()).pow(2)).sum()
        total = term if total is None else total + term
    return total if total is not None else torch.zeros((), device=next(model.parameters()).device)


def make_ewc_loss(fisher: Dict[str, torch.Tensor], old_params: Dict[str, torch.Tensor], lam: float):
    """Adapter for ``Trainer(extra_loss=...)``."""
    return lambda model: lam * ewc_penalty(model, fisher, old_params)

In [ ]:
%%writefile tensegrity_src/tensegrity/presets.py
"""Model-size presets: the **growth ladder** 1M -> 2M -> 5M -> 10M -> 50M -> 115M, plus the
original named presets (unchanged, so old checkpoints/configs keep their meaning).

Ladder design rules (so ``grow_model`` can walk the whole ladder with exact function
preservation -- verified for every adjacent pair in tests/test_presets.py):

  * ``head_dim = 32`` and GQA ratio 2:1 everywhere  (heads = d/32, kv heads = d/64);
  * ``n_embd``, ``n_layer``, FFN width and ``block_size`` are non-decreasing rung to rung;
  * the attention pattern (``global_every``, ``sliding_window``) is identical, so layer *i*
    keeps its type when new layers are appended;
  * FFN width is solved per rung so the parameter count lands within ~1% of the name.

Training defaults (batch size, learning rate, byte budget) are starting points chosen from
standard width-scaling heuristics and the small ablations in tools/ablate.py; they are not
tuned per-size on large runs. Treat them as defaults, not optima.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

from .config import TensegrityConfig
from .schedule import RunPlan, make_plan
from .train import TrainConfig, default_ema_decay

# --------------------------------------------------------------------------- legacy presets
BASE15M_CONFIG_KWARGS: Dict[str, Any] = dict(
    n_layer=4, n_embd=512, n_head=4, n_kv_head=2, ffn_mult=4.0, ffn_align=64,
    block_size=256, sliding_window=124, global_every=4)

BALANCED135M_CONFIG_KWARGS: Dict[str, Any] = dict(
    n_layer=12, n_embd=1024, n_head=16, n_kv_head=4, ffn_mult=8.0 / 3.0, ffn_align=64,
    block_size=2048, sliding_window=512, global_every=4)

HIGHINTEL400M_CONFIG_KWARGS: Dict[str, Any] = dict(
    n_layer=16, n_embd=1536, n_head=12, n_kv_head=4, ffn_mult=8.0 / 3.0, ffn_align=64,
    block_size=2048, sliding_window=512, global_every=4)

LEGACY_PRESETS: Dict[str, Dict[str, Any]] = {
    "base15m": BASE15M_CONFIG_KWARGS,
    "balanced135m": BALANCED135M_CONFIG_KWARGS,
    "highintel400m": HIGHINTEL400M_CONFIG_KWARGS,
}

# ------------------------------------------------------------------------------ the ladder
_HEAD_DIM = 32

# Architecture shared by every rung (and therefore frozen along the ladder).
LADDER_COMMON: Dict[str, Any] = dict(
    block_size=1024, sliding_window=256, global_every=3, ffn_align=32,
    layerdrop_max=0.0, grad_checkpoint=False,
    use_qk_norm=True, embed_scale=False, logit_soft_cap=30.0, z_loss_weight=1e-4,
    confidence_detach=True, byte_mix_kernel=4, byte_mix_bilinear=True,
    x0_mix=True, v0_mix=True, n_loops=1, homeo_weight=0.01,
)  # embed_scale / byte_mix / x0_mix / v0_mix / optimizer choices: see tools/ablation_*.log and the README


@dataclass(frozen=True)
class Preset:
    name: str                 # "5m"
    label: str                # "5M"
    n_embd: int
    n_layer: int
    ffn_hidden: int
    batch_tokens: int         # tokens per optimizer step
    learning_rate: float      # peak AdamW LR (embeddings, norms, scalars; whole model when optimizer="adamw")
    default_tokens: int       # default training byte budget
    min_vram_gb: float        # rough floor for comfortable training at the default micro-batch
    blurb: str

    def model_kwargs_params(self) -> int:
        """Parameter count of this rung's default configuration."""
        return TensegrityConfig(**self.model_kwargs).n_params()

    @property
    def model_kwargs(self) -> Dict[str, Any]:
        heads = self.n_embd // _HEAD_DIM
        return {**LADDER_COMMON, "n_layer": self.n_layer, "n_embd": self.n_embd, "n_head": heads,
                "n_kv_head": heads // 2, "ffn_mult": self.ffn_hidden / self.n_embd}


LADDER: Tuple[Preset, ...] = (
    Preset("1m", "1M", 128, 5, 384, 32_768, 4.0e-3, 300_000_000, 1.0,
           "Smoke-test scale. Learns spelling, word shapes and simple sentence rhythm in minutes."),
    Preset("2m", "2M", 192, 5, 480, 32_768, 2.8e-3, 400_000_000, 1.0,
           "Fastest rung that produces locally coherent sentences on narrow corpora."),
    Preset("5m", "5M", 256, 7, 672, 65_536, 2.0e-3, 600_000_000, 2.0,
           "Sweet spot for a free-tier GPU: a few hours for a clearly coherent story model."),
    Preset("10m", "10M", 320, 9, 832, 65_536, 1.6e-3, 800_000_000, 3.0,
           "Noticeably better long-range consistency; still comfortable on a T4."),
    Preset("50m", "50M", 512, 15, 1664, 131_072, 1.0e-3, 1_200_000_000, 8.0,
           "Serious small model. T4: ~a day of compute; L4/A100 recommended."),
    Preset("115m", "115M", 768, 18, 2016, 262_144, 6.5e-4, 1_750_000_000, 12.0,
           "Top of the ladder (GPT-2-small class parameter count, byte-level). A100/L4 or patience."),
)

_BY_NAME: Dict[str, Preset] = {p.name: p for p in LADDER}
LADDER_NAMES: Tuple[str, ...] = tuple(p.name for p in LADDER)
MODEL_PRESETS: Dict[str, Dict[str, Any]] = {**LEGACY_PRESETS, **{p.name: p.model_kwargs for p in LADDER}}


def _norm(name: str) -> str:
    return name.strip().lower().replace(" ", "")


def get_preset(name: str) -> Preset:
    """Ladder preset (training defaults + shape) by name, case-insensitive: "5m", "5M"."""
    key = _norm(name)
    if key not in _BY_NAME:
        raise ValueError(f"unknown ladder preset {name!r}; available: {list(LADDER_NAMES)}")
    return _BY_NAME[key]


def get_preset_config(name: str, **overrides: Any) -> TensegrityConfig:
    """TensegrityConfig for a ladder or legacy preset, with optional field overrides
    (e.g. ``get_preset_config("5m", dropout=0.1, block_size=2048)``)."""
    key = _norm(name)
    if key not in MODEL_PRESETS:
        raise ValueError(f"unknown preset {name!r}; available presets: {sorted(MODEL_PRESETS)}")
    kwargs = dict(MODEL_PRESETS[key])
    kwargs.update(overrides)
    return TensegrityConfig(**kwargs)


def next_preset(name: str) -> Optional[Preset]:
    """The rung above ``name`` on the ladder (None at the top)."""
    i = LADDER_NAMES.index(get_preset(name).name)
    return LADDER[i + 1] if i + 1 < len(LADDER) else None


def describe_presets() -> str:
    rows = [f"{'preset':7s}{'params':>10s}{'layers':>7s}{'d_model':>8s}{'heads/kv':>9s}{'ffn':>6s}"
            f"{'ctx':>6s}{'batch tok':>10s}{'peak lr':>9s}{'byte budget':>12s}"]
    for p in LADDER:
        c = get_preset_config(p.name)
        rows.append(f"{p.label:7s}{c.n_params() / 1e6:9.2f}M{c.n_layer:7d}{c.n_embd:8d}"
                    f"{c.n_head:5d}/{c.n_kv_head:<3d}{c.ffn_hidden:6d}{c.block_size:6d}"
                    f"{p.batch_tokens:10,d}{p.learning_rate:9.1e}{p.default_tokens / 1e6:10.0f}M")
    return "\n".join(rows)


def train_setup(name: str, micro_batch: int, grad_accum: int, total_tokens: Optional[int] = None,
                optimizer: str = "muon", schedule: str = "wsd", ema: bool = True,
                plan: Optional[RunPlan] = None, **tc_overrides: Any) -> Tuple[TrainConfig, RunPlan]:
    """Preset training defaults -> ``(TrainConfig, RunPlan)`` for a concrete micro-batch / accumulation.

    ``total_tokens`` defaults to the preset's budget. Pass ``plan`` to reuse an existing plan
    (e.g. one derived from a wall-clock budget by ``plan_for_time_budget``)."""
    p = get_preset(name)
    cfg = get_preset_config(p.name)
    tokens_per_step = micro_batch * grad_accum * cfg.block_size
    plan = plan or make_plan(total_tokens or p.default_tokens, tokens_per_step, schedule)
    tc = TrainConfig(
        batch_size=micro_batch, grad_accum_steps=grad_accum, learning_rate=p.learning_rate,
        min_lr=p.learning_rate / 10.0, optimizer=optimizer,
        muon_weight_decay=0.0,   # measured: decay on the Muon matrices hurt at 500 steps (tools/ablation_round2.log)
        ema_decay=default_ema_decay(plan.max_steps) if ema else 0.0,
        eval_interval=max(50, plan.max_steps // 20), log_interval=max(1, plan.max_steps // 100))
    for k, v in tc_overrides.items():
        if not hasattr(tc, k):
            raise TypeError(f"TrainConfig has no field {k!r}")
        setattr(tc, k, v)
    return tc, plan

In [ ]:
%%writefile tensegrity_src/tensegrity/ladder.py
"""Climb the size ladder: train a small model, grow it (function-preserving), keep training.

    results = climb_ladder(["1m", "2m", "5m"], data_dir, "runs/ladder", hours=[0.2, 0.4, 1.0])

Each rung is trained with its own preset hyper-parameters; between rungs ``grow_model`` carries
the previous rung's best weights into the next architecture, *verifying* that the function was
preserved and that the new capacity is trainable before spending any compute on it.

The whole climb is resumable: finished rungs are recorded (``rung_done.json``) and skipped, an
interrupted rung resumes from its own checkpoint, and per-rung plans are persisted, so re-running
the same call after a Colab disconnect simply continues.
"""

from __future__ import annotations

import json
import math
import os
import time
from dataclasses import asdict, dataclass
from typing import Any, Callable, Dict, List, Optional, Sequence

import torch

from .checkpoint import load_checkpoint, load_model
from .config import TensegrityConfig
from .growth import check_function_preserved, check_growth_trainable, grow_model, grow_optimizer_state, retarget
from .hw import autotune_micro_batch, measure_throughput, pick_device, pick_dtype
from .model import TensegrityLM
from .presets import get_preset, get_preset_config, train_setup
from .schedule import make_plan, plan_for_time_budget
from .train import Trainer, load_run_plan, preflight_or_fallback


@dataclass
class RungResult:
    rung: str
    params: int
    status: str                   # "completed" | "time_limit"
    steps: int
    tokens_seen: int
    best_val_loss: float
    best_val_bpb: float
    run_dir: str
    growth_max_logit_diff: Optional[float] = None
    seconds: float = 0.0
    skipped: bool = False         # True when loaded from a previous session


def _per_rung(value, n: int, name: str):
    if value is None:
        return [None] * n
    if isinstance(value, (int, float)):
        return [value] * n
    value = list(value)
    if len(value) != n:
        raise ValueError(f"{name} needs {n} entries (one per rung), got {len(value)}")
    return value


def climb_ladder(rungs: Sequence[str], data_dirs, root: str, *, hours=None, tokens=None, device: str = "auto",
                 dtype: str = "auto", optimizer: str = "muon", micro_batch: Optional[int] = None,
                 model_overrides: Optional[Dict[str, Any]] = None, tc_overrides: Optional[Dict[str, Any]] = None,
                 batch_tokens: Optional[int] = None, intermediate_cooldown: Optional[float] = None, data_weights: Optional[Sequence[float]] = None, lr_scale_after_growth: float = 1.0,
                 callbacks: Optional[Sequence[Callable]] = None, seed: int = 1337, verbose: bool = True,
                 ) -> List[RungResult]:
    """
    Train every rung in ``rungs`` in order, growing between them.

    hours / tokens : per-rung budgets (a number applies to every rung; a list gives one per rung).
        ``hours`` measures this machine's throughput on each rung and sizes the run to fit;
        ``tokens`` is an explicit byte budget; neither -> each preset's default budget.
    model_overrides : architecture overrides for the FIRST rung (e.g. ``block_size``); later rungs
        inherit every function-defining field through ``retarget``.
    lr_scale_after_growth : multiply the preset peak LR for grown rungs (1.0 = the preset's own).
    batch_tokens : tokens per optimizer step for EVERY rung (default: each preset's own; use a small
        value such as 4096 on CPU).
    intermediate_cooldown : fraction of a NON-final rung's steps spent cooling the LR down (default
        None = the usual 20%). A small value (e.g. 0.03) skips most of it, because the next rung re-warms
        anyway; in tools/ladder_vs_scratch.py this recovered ~0.01 bits/byte at equal compute, at the price
        of intermediate checkpoints that are not annealed (worse as standalone models).
    """
    rungs = [get_preset(r).name for r in rungs]
    if len(set(rungs)) != len(rungs):
        raise ValueError("rungs must be distinct")
    n = len(rungs)
    hours_l = _per_rung(hours, n, "hours")
    tokens_l = _per_rung(tokens, n, "tokens")
    if hours is not None and tokens is not None:
        raise ValueError("pass hours OR tokens, not both")
    dev = pick_device(device)
    dt = pick_dtype(dtype, dev)
    os.makedirs(root, exist_ok=True)
    say = print if verbose else (lambda *a, **k: None)

    results: List[RungResult] = []
    prev_dir: Optional[str] = None
    prev_cfg: Optional[TensegrityConfig] = None
    for i, name in enumerate(rungs):
        preset = get_preset(name)
        run_dir = os.path.join(root, f"rung{i}_{name}")
        done_path = os.path.join(run_dir, "rung_done.json")
        if os.path.exists(done_path):
            with open(done_path) as fh:
                r = RungResult(**{**json.load(fh), "skipped": True})
            say(f"[ladder] rung {i} ({preset.label}) already finished: val {r.best_val_bpb:.3f} bpb -- skipping")
            results.append(r)
            prev_dir = run_dir
            prev_cfg = TensegrityConfig.from_dict(load_checkpoint(os.path.join(run_dir, "ckpt_best.pt"))["config"])
            continue

        t0 = time.time()
        saved = load_run_plan(run_dir)
        growth_diff: Optional[float] = None
        init_opt_fn: Optional[Callable] = None
        if i == 0:
            cfg = get_preset_config(name, **(model_overrides or {}))
            model = TensegrityLM(cfg)
        else:
            assert prev_dir is not None and prev_cfg is not None
            cfg = retarget(prev_cfg, get_preset_config(name)).replace(
                **{k: v for k, v in (model_overrides or {}).items() if k in ("dropout", "layerdrop_max", "homeo_weight")})
            if saved is not None:  # interrupted mid-rung: the checkpoint will overwrite these weights anyway
                model = TensegrityLM(saved[2])
                cfg = saved[2]
            else:
                parent, parent_cfg, _ = load_model(os.path.join(prev_dir, "ckpt_best.pt"), "cpu")
                model, cfg = grow_model(parent, parent_cfg, target=cfg, device="cpu")
                growth_diff = check_function_preserved(parent, model, parent_cfg, "cpu", verbose=verbose)
                check_growth_trainable(model, parent_cfg, verbose=verbose)
                prev_latest = os.path.join(prev_dir, "ckpt_latest.pt")
                if os.path.exists(prev_latest):
                    prev_opt_state = load_checkpoint(prev_latest, "cpu").get("optimizer_state_dict")
                    if prev_opt_state is not None:
                        init_opt_fn = lambda new_opt, pm=parent, pos=prev_opt_state, nm=model: (  # noqa: E731
                            grow_optimizer_state(pm, pos, nm, new_opt))
                say(f"[ladder] grew {parent_cfg.n_params() / 1e6:.2f}M -> {cfg.n_params() / 1e6:.2f}M "
                    f"(function preserved, new capacity trainable)")
        model.to(dev)
        T = cfg.block_size

        if saved is not None:
            plan, tc, _ = saved
            for k, v in (tc_overrides or {}).items():     # this session's settings (limits, cadence) win
                setattr(tc, k, v)
            say(f"[ladder] rung {i} ({preset.label}): resuming its saved plan ({plan.max_steps:,} steps)")
        else:
            target = batch_tokens or preset.batch_tokens
            if micro_batch is not None:
                mb, accum = micro_batch, max(1, round(target / (micro_batch * T)))
            else:
                tune = autotune_micro_batch(model, T, dev, dt, target, verbose=False)
                mb, accum = tune["micro_batch"], tune["grad_accum"]
                model.grad_checkpoint = tune["grad_checkpoint"]
            tps_step = mb * accum * T
            plan_kw = {"decay_frac": intermediate_cooldown} if (intermediate_cooldown is not None and i < n - 1) else {}
            if hours_l[i] is not None:
                tps = measure_throughput(model, mb, T, dev, dt, steps=3, warmup=1)
                plan = plan_for_time_budget(tps, float(hours_l[i]), tps_step, max_tokens=preset.default_tokens, **plan_kw)
                say(f"[ladder] rung {i} ({preset.label}): {tps / 1e3:.1f}k tok/s -> {plan.total_tokens / 1e6:.0f}M bytes "
                    f"in ~{hours_l[i]} h")
            else:
                plan = make_plan(int(tokens_l[i] or preset.default_tokens), tps_step, **plan_kw)
            tc, plan = train_setup(name, mb, accum, plan=plan, optimizer=optimizer, dtype=dtype, seed=seed + i,
                                   **(tc_overrides or {}))
            if i > 0 and lr_scale_after_growth != 1.0:
                tc.learning_rate *= lr_scale_after_growth
                tc.min_lr *= lr_scale_after_growth
            tc = preflight_or_fallback(model, tc, dev, verbose=verbose)
            for k, v in (tc_overrides or {}).items():
                setattr(tc, k, v)

        trainer = Trainer(model, tc, plan, data_dirs, run_dir, data_weights=data_weights, device=dev,
                          callbacks=callbacks, init_optimizer=init_opt_fn)
        res = trainer.fit(resume=True)
        r = RungResult(rung=name, params=cfg.n_params(), status=res["status"], steps=res["step"],
                       tokens_seen=res["tokens_seen"], best_val_loss=res["best_val_loss"],
                       best_val_bpb=res["best_val_loss"] / math.log(2), run_dir=run_dir,
                       growth_max_logit_diff=growth_diff, seconds=time.time() - t0)
        results.append(r)
        if res["status"] != "completed":
            say(f"[ladder] stopped inside rung {i} ({preset.label}) with status {res['status']!r}; "
                f"re-run the same call to continue")
            return results
        with open(done_path, "w") as fh:
            json.dump({k: v for k, v in asdict(r).items() if k != "skipped"}, fh, indent=2)
        prev_dir, prev_cfg = run_dir, cfg
        if dev.startswith("cuda"):
            del trainer, model
            torch.cuda.empty_cache()
    return results

In [ ]:
%%writefile tensegrity_src/tensegrity/analysis.py
"""Model analysis helpers: confidence-head calibration and a per-preset hardware benchmark."""

from __future__ import annotations

from typing import Any, Dict, List, Optional

import torch

from .data import ByteDataset
from .hw import amp_settings, autotune_micro_batch, device_type_of, gpu_info, mfu, measure_throughput, pick_dtype
from .model import TensegrityLM


@torch.no_grad()
def confidence_calibration(model: TensegrityLM, data_dir: str, device: str = "cpu", n_batches: int = 20,
                           batch_size: int = 8, n_bins: int = 10, dtype: str = "auto",
                           seq_len: Optional[int] = None) -> Dict[str, Any]:
    """
    How trustworthy is the confidence head? It predicts P(argmax prediction == target). Tokens are
    bucketed by predicted confidence on the validation split and the empirical accuracy of each
    bucket is measured. A calibrated head has accuracy ~ mean confidence in every bin.

    Returns ``{"bins": [{bin_lo, bin_hi, count, accuracy, mean_confidence}], "ece": float,
    "n_tokens": int, "base_accuracy": float}``; ECE is the count-weighted mean |accuracy - confidence|
    (0 = perfectly calibrated).
    """
    if model.confidence_head is None:
        raise ValueError("model has no confidence head (cfg.use_confidence_head=False)")
    cfg = model.cfg
    amp = amp_settings(pick_dtype(dtype, device), device)
    ds = ByteDataset(data_dir)
    batches = ds.fixed_batches("val", n_batches, batch_size, seq_len or cfg.block_size, device)
    was_training = model.training
    model.eval()
    edges = torch.linspace(0.0, 1.0, n_bins + 1)
    count = torch.zeros(n_bins)
    correct_sum = torch.zeros(n_bins)
    conf_sum = torch.zeros(n_bins)
    try:
        for x, y in batches:
            with torch.autocast(device_type=amp["autocast_device"], dtype=amp["amp_dtype"], enabled=amp["use_amp"]):
                logits, _, conf = model(x)
            correct = (logits.argmax(-1) == y).float().reshape(-1).cpu()
            c = conf.float().reshape(-1).clamp(0, 1).cpu()
            idx = torch.bucketize(c, edges[1:-1])
            count += torch.bincount(idx, minlength=n_bins).float()
            correct_sum += torch.bincount(idx, weights=correct, minlength=n_bins).float()
            conf_sum += torch.bincount(idx, weights=c, minlength=n_bins).float()
    finally:
        model.train(was_training)
    total = float(count.sum())
    rows: List[Dict[str, Any]] = []
    ece = 0.0
    for b in range(n_bins):
        n = float(count[b])
        acc = float(correct_sum[b] / n) if n else float("nan")
        mc = float(conf_sum[b] / n) if n else float("nan")
        if n:
            ece += n / total * abs(acc - mc)
        rows.append({"bin_lo": float(edges[b]), "bin_hi": float(edges[b + 1]), "count": int(n),
                     "accuracy": acc, "mean_confidence": mc})
    return {"bins": rows, "ece": ece, "n_tokens": int(total),
            "base_accuracy": float(correct_sum.sum() / max(total, 1.0))}


def plot_reliability(result: Dict[str, Any]) -> None:
    """Reliability diagram for :func:`confidence_calibration` (no-op without matplotlib)."""
    try:
        import matplotlib.pyplot as plt
    except Exception:  # noqa: BLE001
        return
    rows = [r for r in result["bins"] if r["count"]]
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax.plot([r["mean_confidence"] for r in rows], [r["accuracy"] for r in rows], "o-", label="model")
    ax.set(xlabel="predicted confidence", ylabel="empirical accuracy", xlim=(0, 1), ylim=(0, 1),
           title=f"confidence calibration (ECE {result['ece']:.3f})")
    ax.legend()
    ax.grid(alpha=.3)
    plt.show()


def benchmark_presets(names, device: str = "auto", seq_len: Optional[int] = None, dtype: str = "auto",
                      verbose: bool = True) -> List[Dict[str, Any]]:
    """Measure training throughput of each preset on this machine (autotuned micro-batch)."""
    from .presets import get_preset, get_preset_config
    from .hw import pick_device
    device = pick_device(device)
    dt = pick_dtype(dtype, device)
    info = gpu_info()
    rows = []
    for name in names:
        cfg = get_preset_config(name)
        T = seq_len or cfg.block_size
        model = TensegrityLM(cfg).to(device)
        tune = autotune_micro_batch(model, T, device, dt, get_preset(name).batch_tokens, verbose=False)
        model.grad_checkpoint = tune["grad_checkpoint"]
        tps = measure_throughput(model, tune["micro_batch"], T, device, dt, steps=4, warmup=2)
        budget = get_preset(name).default_tokens
        rows.append({"preset": name, "params": cfg.n_params(), "micro_batch": tune["micro_batch"],
                     "grad_accum": tune["grad_accum"], "grad_checkpoint": tune["grad_checkpoint"],
                     "tokens_per_sec": tps, "mfu": mfu(cfg, tps, T, info["peak_tflops"]),
                     "hours_for_default_budget": budget / tps / 3600.0})
        if verbose:
            print(f"{name:>5s}: {tps / 1e3:8.1f}k tok/s  micro-batch {tune['micro_batch']:3d} x{tune['grad_accum']:<3d}"
                  f"  -> default {budget / 1e6:.0f}M-byte budget in {rows[-1]['hours_for_default_budget']:.1f} h")
        del model
        if device_type_of(device) == "cuda":
            torch.cuda.empty_cache()
    return rows

In [ ]:
%%writefile tensegrity_src/tensegrity/export.py
"""Export / import: safetensors (+ self-describing config), model card, INT8 dynamic quantisation."""

from __future__ import annotations

import json
import os
import warnings
from typing import Any, Dict, Optional, Tuple

import torch

from .config import TensegrityConfig
from .model import TensegrityLM

_DTYPES = {"float32": torch.float32, "fp32": torch.float32, "float16": torch.float16, "fp16": torch.float16,
           "bfloat16": torch.bfloat16, "bf16": torch.bfloat16}


def export_safetensors(model: TensegrityLM, cfg: Optional[TensegrityConfig] = None, out_dir: str = "export",
                       name: str = "model", dtype: str = "float32",
                       extra_metadata: Optional[Dict[str, Any]] = None) -> Tuple[str, str]:
    """
    Write ``{name}.safetensors`` and ``{name}.config.json``.

    * The tied ``lm_head.weight`` is not stored (it is the same tensor as ``token_emb.weight``;
      the loader restores the tie), so the file is smaller and safetensors' no-shared-memory
      rule is respected. Files written by the v1 exporter (which stored both) load fine too.
    * The config is also embedded in the safetensors header, so the single file is self-describing.
    * ``dtype``: "float32" (lossless), "float16" or "bfloat16" for a smaller deployment copy.
    """
    from safetensors.torch import save_file
    cfg = cfg or model.cfg
    if dtype not in _DTYPES:
        raise ValueError(f"dtype must be one of {sorted(set(_DTYPES))}, got {dtype!r}")
    target = _DTYPES[dtype]
    os.makedirs(out_dir, exist_ok=True)
    tensors = {}
    for k, v in model.state_dict().items():
        if k == "lm_head.weight":
            continue
        t = v.detach().cpu().contiguous()
        tensors[k] = t.to(target) if t.is_floating_point() else t
    meta = {"format": "pt", "architecture": "TensegrityLM", "tensegrity_config": json.dumps(cfg.to_dict()),
            "dtype": dtype, "n_params": str(model.num_parameters())}
    if extra_metadata:
        meta.update({k: str(v) for k, v in extra_metadata.items()})
    weights_path = os.path.join(out_dir, f"{name}.safetensors")
    save_file(tensors, weights_path, metadata=meta)
    config_path = os.path.join(out_dir, f"{name}.config.json")
    with open(config_path, "w") as fh:
        json.dump(cfg.to_dict(), fh, indent=2)
    return weights_path, config_path


def load_safetensors(weights_path: str, config_path: Optional[str] = None, device: str = "cpu"
                     ) -> Tuple[TensegrityLM, TensegrityConfig]:
    """Rebuild a model from a ``.safetensors`` file (config from ``config_path`` or the file header)."""
    from safetensors import safe_open
    from safetensors.torch import load_file
    if config_path is not None:
        with open(config_path) as fh:
            cfg_dict = json.load(fh)
    else:
        with safe_open(weights_path, framework="pt") as f:
            raw = (f.metadata() or {}).get("tensegrity_config")
        if raw is None:
            raise ValueError(f"{weights_path} has no embedded config; pass config_path=")
        cfg_dict = json.loads(raw)
    cfg = TensegrityConfig.from_dict(cfg_dict)
    model = TensegrityLM(cfg)
    state = load_file(weights_path)
    state.pop("lm_head.weight", None)  # tied to token_emb.weight
    missing, unexpected = model.load_state_dict(state, strict=False)
    missing = [k for k in missing if k != "lm_head.weight"]
    if missing or unexpected:
        raise ValueError(f"state mismatch loading {weights_path}: missing={missing[:5]} unexpected={unexpected[:5]}")
    assert model.lm_head.weight is model.token_emb.weight, "weight tying lost"
    return model.to(device), cfg


def quantize_int8_dynamic(model: TensegrityLM) -> Optional[torch.nn.Module]:
    """PyTorch dynamic INT8 quantisation of every nn.Linear (CPU inference). Returns None (with a
    warning) when the installed torch no longer ships eager-mode dynamic quantisation.

    The custom architecture has no GGUF/llama.cpp backend, so this is the portable low-memory path."""
    try:
        from torch.ao.quantization import quantize_dynamic
    except Exception as e:  # noqa: BLE001
        warnings.warn(f"dynamic quantisation unavailable in this torch build ({e})")
        return None
    cpu_model = TensegrityLM(model.cfg)
    cpu_model.load_state_dict(model.state_dict())
    cpu_model.eval()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return quantize_dynamic(cpu_model, {torch.nn.Linear}, dtype=torch.qint8)


def write_model_card(out_dir: str, cfg: TensegrityConfig, name: str = "model",
                     metrics: Optional[Dict[str, Any]] = None, training: Optional[Dict[str, Any]] = None,
                     dataset: str = "unspecified") -> str:
    """Write an honest ``README.md`` model card next to the exported weights."""
    os.makedirs(out_dir, exist_ok=True)
    m = metrics or {}
    t = training or {}
    lines = [
        f"# {name}", "",
        f"A **{cfg.n_params() / 1e6:.1f}M-parameter byte-level language model** (TensegrityLM). "
        "It reads and writes raw UTF-8 bytes (vocabulary 259: 256 bytes + BOS/EOS/PAD); there is no tokenizer file.", "",
        "## Architecture", "",
        f"- layers {cfg.n_layer}, width {cfg.n_embd}, {cfg.n_head} heads / {cfg.n_kv_head} KV heads (head_dim {cfg.head_dim})",
        f"- context {cfg.block_size} bytes; every {cfg.global_every}rd layer global, the rest sliding-window {cfg.sliding_window}",
        f"- SwiGLU FFN width {cfg.ffn_hidden}; RoPE (base {cfg.rope_base:g}); RMSNorm; learnable residual gains",
        f"- Q/K-norm {cfg.use_qk_norm}, logit soft-cap {cfg.logit_soft_cap}, ByteMix kernel {cfg.byte_mix_kernel or 'off'}"
        f"{' (bilinear)' if cfg.byte_mix_bilinear else ''}, "
        f"x0-mix {cfg.x0_mix}, v0-mix {cfg.v0_mix}, loops {cfg.n_loops}, confidence head {cfg.use_confidence_head}", "",
        "## Training", "",
        f"- dataset: {dataset}",
    ]
    for k, v in t.items():
        lines.append(f"- {k}: {v}")
    if m:
        lines += ["", "## Evaluation (held-out split of the training corpus)", ""]
        for k, v in m.items():
            lines.append(f"- {k}: {v:.4f}" if isinstance(v, float) else f"- {k}: {v}")
    lines += [
        "", "## Usage", "", "```python",
        "from tensegrity import load_safetensors, ByteTokenizer", "import torch",
        f"model, cfg = load_safetensors('{name}.safetensors'); model.eval()",
        "tok = ByteTokenizer()",
        "ids = torch.tensor([[tok.BOS] + tok.encode('Once upon a time')])",
        "out = model.generate(ids, max_new_tokens=200, temperature=0.8, top_k=40)",
        "print(tok.decode(out[0].tolist()))", "```", "",
        "## Limitations", "",
        "- Small research model trained on a narrow corpus: expect fluent-looking but unreliable text; "
        "it has no safety tuning and should not be used for decisions.",
        "- Byte-level modelling needs ~4x more positions than a subword model for the same text.",
        "- Quality claims should be checked on your own data; the architecture combination is not benchmarked "
        "against other models in this repository.", "",
    ]
    path = os.path.join(out_dir, "README.md")
    with open(path, "w") as fh:
        fh.write("\n".join(lines))
    return path

In [ ]:
%%writefile tensegrity_src/tensegrity/viz.py
"""Notebook visualisation: a live training dashboard and confidence-shaded text.

Everything imports lazily so the library works without matplotlib / IPython installed."""

from __future__ import annotations

import codecs
import html
import math
from typing import Any, Dict, List, Optional, Sequence


class LivePlot:
    """Callback for ``Trainer``: redraws loss / bits-per-byte / LR / throughput after each eval.

    ``trainer = Trainer(..., callbacks=[LivePlot()])`` -- in a notebook the figure updates in
    place; elsewhere it silently does nothing."""

    def __init__(self, redraw_every_evals: int = 1):
        self.train: List[Dict[str, Any]] = []
        self.val: List[Dict[str, Any]] = []
        self.samples: List[Dict[str, Any]] = []
        self.redraw_every = redraw_every_evals
        self._n_eval = 0
        self._handle = None

    def __call__(self, event: str, payload: Dict[str, Any]) -> None:
        if event == "log":
            self.train.append(payload)
        elif event == "eval":
            self.val.append(payload)
            self._n_eval += 1
            if self._n_eval % self.redraw_every == 0:
                self.draw()
        elif event == "sample":
            self.samples.append(payload)

    def draw(self) -> None:
        """(Re)draw the dashboard. The figure lives in a display handle that is updated in place, so
        the log lines printed by the trainer around it are never erased."""
        try:
            import matplotlib.pyplot as plt
            from IPython import get_ipython
            from IPython.display import display
        except Exception:  # noqa: BLE001
            return
        if get_ipython() is None or not self.train:
            return
        fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
        s = [r["step"] for r in self.train]
        ax[0].plot(s, [r["ce_loss"] / math.log(2) for r in self.train], lw=1, alpha=.8, label="train")
        if self.val:
            ax[0].plot([r["step"] for r in self.val], [r["val_bpb"] for r in self.val], "o-", ms=3, label="val")
            if "val_bpb_ema" in self.val[0]:
                ax[0].plot([r["step"] for r in self.val], [r["val_bpb_ema"] for r in self.val], "s--", ms=3, label="val (EMA)")
        ax[0].set(title="bits per byte  (lower is better)", xlabel="step")
        ax[0].legend()
        ax[0].grid(alpha=.3)
        ax[1].plot(s, [r["lr"] for r in self.train], color="C3")
        ax[1].set(title="learning rate", xlabel="step")
        ax[1].grid(alpha=.3)
        tps = [(r["step"], r["tok_per_s"] / 1e3) for r in self.train if r.get("tok_per_s")]
        if tps:
            ax[2].plot(*zip(*tps), color="C2")
        ax[2].set(title="throughput (k tokens/s)", xlabel="step")
        ax[2].grid(alpha=.3)
        fig.tight_layout()
        if self._handle is None:
            self._handle = display(fig, display_id=True)
        else:
            self._handle.update(fig)
        plt.close(fig)


def confidence_html(token_ids: Sequence[int], confidences: Optional[Sequence[float]]) -> str:
    """Render generated bytes as HTML, shading each character by the confidence head's score
    (red = the model expected to be wrong, green = confident). Multi-byte UTF-8 characters take
    the mean score of their bytes. Falls back to plain escaped text without confidences."""
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    out: List[str] = []
    pending: List[float] = []
    for i, t in enumerate(token_ids):
        t = int(t)
        if t > 255:
            continue
        if confidences is not None:
            pending.append(float(confidences[i]))
        ch = dec.decode(bytes([t]))
        if not ch:
            continue
        c = sum(pending) / len(pending) if pending else None
        pending = []
        esc = html.escape(ch).replace("\n", "<br>").replace(" ", "&nbsp;")
        if c is None:
            out.append(esc)
        else:
            hue = 120.0 * max(0.0, min(1.0, c))  # 0 = red .. 120 = green
            out.append(f'<span style="background:hsla({hue:.0f},70%,50%,0.35)" title="conf {c:.2f}">{esc}</span>')
    return '<div style="font-family:monospace;line-height:1.5">' + "".join(out) + "</div>"

In [ ]:
%%writefile tensegrity_src/tensegrity/colab.py
"""Google Colab (and Kaggle / local) helpers. Every function degrades gracefully outside Colab,
so the same notebook runs unchanged on a laptop or in CI."""

from __future__ import annotations

import importlib.util
import os
import platform
import shutil
import sys
from typing import Optional


def in_colab() -> bool:
    if "google.colab" in sys.modules:
        return True
    try:  # find_spec imports parent packages and raises if there is no `google` namespace at all
        return importlib.util.find_spec("google.colab") is not None
    except (ImportError, ValueError):
        return False


def in_kaggle() -> bool:
    return os.path.isdir("/kaggle/working")


def mount_drive(mount_point: str = "/content/drive") -> bool:
    """Mount Google Drive. Returns True on success, False (with a message) anywhere it is not possible."""
    if not in_colab():
        print("[drive] not running in Colab; skipping Drive mount.")
        return False
    if os.path.isdir(os.path.join(mount_point, "MyDrive")):
        return True
    try:
        from google.colab import drive
        drive.mount(mount_point)
        return os.path.isdir(os.path.join(mount_point, "MyDrive"))
    except Exception as e:  # noqa: BLE001 - user may decline the prompt
        print(f"[drive] could not mount Drive ({type(e).__name__}: {e}); continuing with local disk.")
        return False


def project_dir(use_drive: bool = True, name: str = "tensegrity") -> str:
    """Where checkpoints live: Drive (survives disconnects) when available, else local disk.
    ``TENSEGRITY_PROJECT_DIR`` overrides the location (used by CI)."""
    env = os.environ.get("TENSEGRITY_PROJECT_DIR")
    if env:
        os.makedirs(env, exist_ok=True)
        return env
    if use_drive and mount_drive():
        base = "/content/drive/MyDrive"
    elif in_kaggle():
        base = "/kaggle/working"
    else:
        base = os.getcwd()
    path = os.path.join(base, name)
    os.makedirs(path, exist_ok=True)
    return path


def local_scratch(name: str = "tensegrity_data") -> str:
    """Fast local disk for datasets (never Drive: random reads from FUSE are slow).
    ``TENSEGRITY_DATA_DIR`` overrides the location (used by CI)."""
    env = os.environ.get("TENSEGRITY_DATA_DIR")
    if env:
        os.makedirs(env, exist_ok=True)
        return env
    base = "/content" if os.path.isdir("/content") else ("/kaggle/working" if in_kaggle() else os.getcwd())
    path = os.path.join(base, name)
    os.makedirs(path, exist_ok=True)
    return path


def get_secret(name: str) -> Optional[str]:
    """Colab Secrets (key icon in the sidebar) or an environment variable."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:  # noqa: BLE001 - not in Colab / secret missing / access not granted
        return os.environ.get(name)


def download_to_browser(path: str) -> None:
    if in_colab():
        from google.colab import files
        files.download(path)
    else:
        print(f"[download] not in Colab; file is at {os.path.abspath(path)}")


def free_disk_gb(path: str = ".") -> float:
    return shutil.disk_usage(path).free / 2**30


def total_ram_gb() -> float:
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
    except (ValueError, OSError, AttributeError):
        return float("nan")


def runtime_summary() -> str:
    import torch
    from .hw import gpu_info
    g = gpu_info()
    gpu = (f"{g['name']} ({g['total_gb']:.0f} GB, sm{g['capability'][0]}{g['capability'][1]}, "
           f"{'native bf16' if g['native_bf16'] else 'fp16 (no native bf16)'})") if g["available"] else "none (CPU)"
    where = "Google Colab" if in_colab() else ("Kaggle" if in_kaggle() else "local")
    return (f"{where} | python {platform.python_version()} | torch {torch.__version__} | GPU: {gpu} | "
            f"RAM {total_ram_gb():.0f} GB | free disk {free_disk_gb('/content' if os.path.isdir('/content') else '.'):.0f} GB")


def push_to_hub(folder: str, repo_id: str, token: str, private: bool = True) -> str:
    """Upload an export folder to the Hugging Face Hub (opt-in; needs ``huggingface_hub``)."""
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(repo_id, private=private, exist_ok=True)
    api.upload_folder(folder_path=folder, repo_id=repo_id)
    return f"https://huggingface.co/{repo_id}"

In [ ]:
%%writefile tensegrity_src/tensegrity/cli.py
"""Command line: ``python -m tensegrity <command>``.

    presets                          list the size ladder
    prepare   --data_path P --out_dir D [--doc_sep '<|endoftext|>']
    train     --preset 5m --data_dir D --out_dir R [--hours 3 | --tokens 5e8] [--optimizer muon]
    generate  --ckpt R/ckpt_best.pt --prompt "Once upon a time"
    eval      --ckpt R/ckpt_best.pt --data_dir D
    grow      --ckpt R/ckpt_best.pt --to 10m --out R2/ckpt_latest.pt
    export    --ckpt R/ckpt_best.pt --out_dir export [--dtype float16]

Multi-GPU: ``torchrun --nproc_per_node=2 -m tensegrity train ...`` (DDP is picked up from the
environment).
"""

from __future__ import annotations

import argparse
import math
import os
from typing import Optional

import torch

from . import __version__
from .checkpoint import load_model, save_checkpoint
from .data import prepare_data
from .export import export_safetensors, write_model_card
from .growth import check_function_preserved, check_growth_trainable, grow_model, retarget
from .hw import autotune_micro_batch, measure_throughput, pick_device, pick_dtype
from .model import TensegrityLM
from .presets import (LADDER_NAMES, describe_presets, get_preset, get_preset_config, train_setup)
from .schedule import make_plan, plan_for_time_budget
from .tokenizer import ByteTokenizer
from .train import TrainConfig, Trainer, evaluate, load_run_plan, preflight_or_fallback


def _bool(x: str) -> bool:
    return str(x).lower() in ("1", "true", "yes", "y")


def cmd_presets(args) -> None:
    print(describe_presets())


def cmd_prepare(args) -> None:
    prepare_data(args.data_path, args.out_dir, args.val_fraction, args.seed, doc_sep=args.doc_sep,
                 max_bytes=int(args.max_mb * 1e6) if args.max_mb else None)


def cmd_train(args) -> None:
    device = pick_device(args.device)
    dtype = pick_dtype(args.dtype, device)
    os.makedirs(args.out_dir, exist_ok=True)

    saved = load_run_plan(args.out_dir)
    overrides = {k: getattr(args, k) for k in ("dropout", "block_size", "sliding_window", "layerdrop_max",
                                               "byte_mix_kernel", "byte_mix_bilinear", "x0_mix", "v0_mix",
                                               "n_loops", "homeo_weight")
                 if getattr(args, k, None) is not None}
    if saved is not None and args.resume:
        plan, tc, cfg = saved
        print(f"[cli] resuming run in {args.out_dir} with its saved plan ({plan.max_steps:,} steps)")
        model = TensegrityLM(cfg)
    else:
        if args.init_weights:
            # architecture comes from the checkpoint; only training-time knobs may be overridden
            arch = sorted(set(overrides) - {"dropout", "layerdrop_max", "homeo_weight"})
            if arch:
                raise SystemExit(f"--init_weights fixes the architecture; remove {['--' + a for a in arch]}")
            src, src_cfg, _ = load_model(args.init_weights, "cpu")
            cfg = src_cfg.replace(**overrides)
            model = TensegrityLM(cfg)
            model.load_state_dict(src.state_dict())
            print(f"[cli] initialised {cfg.n_params() / 1e6:.2f}M-parameter model from {args.init_weights}")
        else:
            cfg = get_preset_config(args.preset, **overrides)
            model = TensegrityLM(cfg)
        preset = get_preset(args.preset) if args.preset in LADDER_NAMES else None
        target_tokens = preset.batch_tokens if preset else 131_072
        if args.micro_batch:
            mb, accum = args.micro_batch, args.grad_accum or 1
        else:
            model.to(device)
            tune = autotune_micro_batch(model, cfg.block_size, device, dtype, target_tokens)
            mb, accum = tune["micro_batch"], tune["grad_accum"]
            model.grad_checkpoint = tune["grad_checkpoint"]
        tokens_per_step = mb * accum * cfg.block_size
        total = int(args.tokens) if args.tokens else (preset.default_tokens if preset else 100_000_000)
        if args.hours:
            tps = measure_throughput(model.to(device), mb, cfg.block_size, device, dtype)
            plan = plan_for_time_budget(tps, args.hours, tokens_per_step, max_tokens=total)
            print(f"[cli] measured {tps / 1e3:.1f}k tok/s -> {plan.total_tokens / 1e6:.0f}M tokens in ~{args.hours}h")
        else:
            plan = make_plan(total, tokens_per_step)
        if preset:
            tc, plan = train_setup(preset.name, mb, accum, plan=plan, optimizer=args.optimizer,
                                   dtype=args.dtype, seed=args.seed, max_minutes=args.max_minutes)
        else:
            tc = TrainConfig(batch_size=mb, grad_accum_steps=accum, learning_rate=args.learning_rate or 3e-4,
                             min_lr=(args.learning_rate or 3e-4) / 10, optimizer=args.optimizer,
                             dtype=args.dtype, seed=args.seed, max_minutes=args.max_minutes)
        if args.learning_rate:
            tc.learning_rate, tc.min_lr = args.learning_rate, args.learning_rate / 10
    if not (saved is not None and args.resume):
        tc = preflight_or_fallback(model, tc, args.device)
    trainer = Trainer(model, tc, plan, args.data_dir, args.out_dir, device=args.device)
    res = trainer.fit(resume=args.resume)
    print({k: res[k] for k in ("status", "step", "best_val_loss", "tokens_seen")})


def cmd_generate(args) -> None:
    device = pick_device(args.device)
    model, cfg, _ = load_model(args.ckpt, device, use_ema=args.ema)
    model.eval()
    tok = ByteTokenizer()
    if args.seed is not None:
        torch.manual_seed(args.seed)
    ids = ([ByteTokenizer.BOS] if (args.add_bos or not args.prompt) else []) + tok.encode(args.prompt or "")
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(idx, args.max_new_tokens, temperature=args.temperature, top_k=args.top_k,
                         top_p=args.top_p, min_p=args.min_p, use_confidence_temp=args.use_confidence_temp,
                         stop_on_eos=not args.no_stop_on_eos, use_cache=not args.no_kv_cache)
    print(tok.decode(out[0].tolist()))


def cmd_eval(args) -> None:
    device = pick_device(args.device)
    model, cfg, _ = load_model(args.ckpt, device, use_ema=args.ema)
    r = evaluate(model, args.data_dir, cfg, device, pick_dtype(args.dtype, device), args.eval_batches)
    print(f"val_loss = {r['loss']:.4f}  bits/byte = {r['bpb']:.4f}  perplexity = {r['ppl']:.3f}")


def cmd_grow(args) -> None:
    device = pick_device(args.device)
    old, old_cfg, ckpt = load_model(args.ckpt, device, use_ema=args.ema)
    target = retarget(old_cfg, get_preset_config(args.to))   # size from the preset, function-defining fields from the checkpoint
    new, new_cfg = grow_model(old, old_cfg, target=target, device=device)
    check_function_preserved(old, new, old_cfg, device)
    check_growth_trainable(new, old_cfg)
    path = args.out
    save_checkpoint(path, new, None, new_cfg, 0, float("inf"), extra={"grown_from": os.path.abspath(args.ckpt)})
    print(f"[grow] {old_cfg.n_params() / 1e6:.2f}M -> {new_cfg.n_params() / 1e6:.2f}M saved to {path} "
          f"(weights only; pass --init_weights to `train`)")


def cmd_export(args) -> None:
    model, cfg, ckpt = load_model(args.ckpt, "cpu", use_ema=args.ema)
    w, c = export_safetensors(model, cfg, args.out_dir, args.name, args.dtype)
    write_model_card(args.out_dir, cfg, args.name, metrics={"best_val_loss (nats/byte)": ckpt["best_val_loss"],
                                                             "best_val_bpb": ckpt["best_val_loss"] / math.log(2)})
    print(f"[export] {w}\n[export] {c}")


def build_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(prog="tensegrity", description=f"TensegrityLM {__version__}")
    sub = ap.add_subparsers(dest="command", required=True)

    sub.add_parser("presets", help="list the model-size ladder").set_defaults(fn=cmd_presets)

    p = sub.add_parser("prepare", help="tokenise text/markdown/parquet/jsonl into train.bin/val.bin")
    p.add_argument("--data_path", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--val_fraction", type=float, default=0.05)
    p.add_argument("--seed", type=int, default=1337)
    p.add_argument("--doc_sep", default=None, help="literal separator mapped to EOS, e.g. '<|endoftext|>'")
    p.add_argument("--max_mb", type=float, default=None)
    p.set_defaults(fn=cmd_prepare)

    p = sub.add_parser("train", help="train or resume")
    p.add_argument("--preset", default="5m", help=f"ladder {list(LADDER_NAMES)} or legacy name")
    p.add_argument("--data_dir", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--hours", type=float, default=None, help="size the run to this wall-clock budget")
    p.add_argument("--tokens", type=float, default=None, help="byte budget (default: preset's)")
    p.add_argument("--optimizer", choices=["muon", "adamw"], default="muon")
    p.add_argument("--learning_rate", type=float, default=None)
    p.add_argument("--micro_batch", type=int, default=None, help="skip autotune")
    p.add_argument("--grad_accum", type=int, default=None)
    p.add_argument("--max_minutes", type=float, default=None, help="stop cleanly after this long")
    p.add_argument("--no_resume", dest="resume", action="store_false")
    p.add_argument("--init_weights", default=None, help="start from these weights (e.g. a grown checkpoint)")
    p.add_argument("--device", default="auto")
    p.add_argument("--dtype", default="auto", choices=["auto", "float32", "bfloat16", "float16"])
    p.add_argument("--seed", type=int, default=1337)
    for name, typ in (("dropout", float), ("block_size", int), ("sliding_window", int), ("layerdrop_max", float),
                      ("byte_mix_kernel", int), ("n_loops", int), ("homeo_weight", float)):
        p.add_argument(f"--{name}", type=typ, default=None)
    for bname in ("x0_mix", "v0_mix", "byte_mix_bilinear"):
        p.add_argument(f"--{bname}", type=_bool, default=None)
    p.set_defaults(fn=cmd_train, resume=True)

    p = sub.add_parser("generate", help="sample text from a checkpoint")
    p.add_argument("--ckpt", required=True)
    p.add_argument("--prompt", default="")
    p.add_argument("--add_bos", action="store_true")
    p.add_argument("--max_new_tokens", type=int, default=300)
    p.add_argument("--temperature", type=float, default=0.8)
    p.add_argument("--top_k", type=int, default=40)
    p.add_argument("--top_p", type=float, default=None)
    p.add_argument("--min_p", type=float, default=None)
    p.add_argument("--use_confidence_temp", action="store_true")
    p.add_argument("--no_stop_on_eos", action="store_true")
    p.add_argument("--no_kv_cache", action="store_true")
    p.add_argument("--ema", action="store_true", help="use EMA weights if the checkpoint has them")
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--device", default="auto")
    p.set_defaults(fn=cmd_generate)

    p = sub.add_parser("eval", help="validation loss / bits-per-byte of a checkpoint")
    p.add_argument("--ckpt", required=True)
    p.add_argument("--data_dir", required=True)
    p.add_argument("--eval_batches", type=int, default=50)
    p.add_argument("--dtype", default="auto", choices=["auto", "float32", "bfloat16", "float16"])
    p.add_argument("--ema", action="store_true")
    p.add_argument("--device", default="auto")
    p.set_defaults(fn=cmd_eval)

    p = sub.add_parser("grow", help="grow a checkpoint to the next ladder rung (function-preserving)")
    p.add_argument("--ckpt", required=True)
    p.add_argument("--to", required=True, choices=list(LADDER_NAMES))
    p.add_argument("--out", required=True)
    p.add_argument("--ema", action="store_true")
    p.add_argument("--device", default="auto")
    p.set_defaults(fn=cmd_grow)

    p = sub.add_parser("export", help="write safetensors + config + model card")
    p.add_argument("--ckpt", required=True)
    p.add_argument("--out_dir", required=True)
    p.add_argument("--name", default="tensegrity")
    p.add_argument("--dtype", default="float32", choices=["float32", "float16", "bfloat16"])
    p.add_argument("--ema", action="store_true")
    p.set_defaults(fn=cmd_export)
    return ap


def main(argv: Optional[list] = None) -> None:
    args = build_parser().parse_args(argv)
    args.fn(args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tensegrity_src/tensegrity/__init__.py
"""Tensegrity: a byte-level language-model family with a verified, trainable growth ladder.

Quick tour::

    from tensegrity import get_preset_config, TensegrityLM, ByteTokenizer
    cfg = get_preset_config("5m")          # 1m 2m 5m 10m 50m 115m
    model = TensegrityLM(cfg)
"""

__version__ = "2.0.0"

from .checkpoint import (CHECKPOINT_FORMAT_VERSION, capture_rng_state, load_checkpoint, load_model,
                         restore_rng_state, save_checkpoint)
from .config import TensegrityConfig, ffn_hidden_dim
from .data import (DATASETS, ByteDataset, download_file, fetch_dataset, fetch_hf_streaming, load_meta,
                   prepare_data)
from .analysis import benchmark_presets, confidence_calibration, plot_reliability
from .colab import (download_to_browser, free_disk_gb, get_secret, in_colab, in_kaggle, local_scratch, mount_drive,
                    project_dir, push_to_hub, runtime_summary)
from .export import (export_safetensors, load_safetensors, quantize_int8_dynamic, write_model_card)
from .growth import (check_function_preserved, check_growth_trainable, compute_fisher_diagonal,
                     expand_reference_state, ewc_penalty, grow_model, grow_optimizer_state, make_ewc_loss, retarget)
from .hw import (autotune_micro_batch, gpu_info, measure_throughput, mfu, pick_device, pick_dtype)
from .ladder import RungResult, climb_ladder
from .model import TensegrityLM
from .presets import (LADDER, LADDER_NAMES, MODEL_PRESETS, describe_presets, get_preset, get_preset_config,
                      next_preset, train_setup)
from .schedule import RunPlan, extend_plan, get_lr, get_lr_wsd, make_plan, plan_for_time_budget
from .tokenizer import ByteTokenizer
from .train import (EMA, TrainConfig, Trainer, default_ema_decay, evaluate, evaluate_batches,
                    load_run_plan, preflight, preflight_or_fallback)
from .viz import LivePlot, confidence_html

__all__ = [n for n in dir() if not n.startswith("_")]

In [ ]:
%%writefile tensegrity_src/tensegrity/__main__.py
from .cli import main

main()

In [ ]:
#@title 0.2 · Load the library & inspect this runtime { display-mode: "form" }
import os, sys, warnings
sys.path.insert(0, os.path.abspath("tensegrity_src"))
for _name in [m for m in sys.modules if m == "tensegrity" or m.startswith("tensegrity.")]:
    del sys.modules[_name]                      # re-running this cell always picks up the freshly written source
warnings.filterwarnings("ignore", message=".*(legacy|TypedStorage).*")
import torch  # shared by the cells below
import tensegrity as tg

print(f"Tensegrity {tg.__version__}")
print(tg.runtime_summary())
_gpu = tg.gpu_info()
if not _gpu["available"]:
    print("\n⚠️  No GPU detected. Use Runtime → Change runtime type → GPU. "
          "On CPU only the 1M model with a short context is practical (set SMOKE_TEST in §1 to try the whole notebook).")
elif _gpu["native_bf16"]:
    print("\n✅ Ampere-or-newer GPU: training in bfloat16.")
else:
    print("\n✅ Turing/Volta-class GPU (e.g. the free T4): training in float16 with loss scaling "
          "(bfloat16 would be emulated and slow here; the library knows).")

### Where to keep your work
Checkpoints are written to **Google Drive** (`MyDrive/tensegrity/`) so a disconnect costs you at most a few minutes. Datasets stay on Colab's fast local disk (reading training data through Drive is slow). Decline the Drive prompt and everything still works — it just lives on the VM's disk until the session ends.

## 1 · Configure
Change the values below, then **Run all** (or run the cells in order). Everything is resumable: if Colab disconnects, reconnect, run the cells again, and training continues from the last checkpoint with the *same* schedule.

In [ ]:
#@title 1 · Your run { display-mode: "form" }
MODEL_SIZE = "5M"  #@param ["1M", "2M", "5M", "10M", "50M", "115M"]
TRAIN_HOURS = 2.0  #@param {type:"number"}
DATASET = "tinystories"  #@param ["tinystories", "tinystories-valid", "shakespeare", "custom"]
DATA_MEGABYTES = 400  #@param {type:"slider", min:50, max:1900, step:50}
CUSTOM_DATA_PATH = ""  #@param {type:"string"}
CUSTOM_DOC_SEPARATOR = ""  #@param {type:"string"}
OPTIMIZER = "muon"  #@param ["muon", "adamw"]
INIT_FROM_WEIGHTS = ""  #@param {type:"string"}
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
RUN_NAME = ""  #@param {type:"string"}
START_FRESH = False  #@param {type:"boolean"}
STOP_AFTER_MINUTES = 0  #@param {type:"number"}
SMOKE_TEST = False  #@param {type:"boolean"}

# SMOKE_TEST: run the entire notebook in a couple of minutes (tiny budget, short context) to
# check your setup before committing hours to a real run.
if SMOKE_TEST:
    MODEL_SIZE, DATASET, DATA_MEGABYTES, TRAIN_HOURS = "1M", "shakespeare", 2, 0.01
    print("SMOKE_TEST: 1M model, Shakespeare, ~1 minute of training.")

## 2 · Data
Downloads are resumable and cached; preparing tokenises to a flat `uint16` file on local disk. `<|endoftext|>` separators in TinyStories become real **EOS tokens** (not 13 spelled-out bytes), which is what lets the model learn to end a story.

In [ ]:
#@title 2 · Download & prepare the dataset { display-mode: "form" }
import hashlib, os, shutil

PROJECT_DIR = tg.project_dir(USE_GOOGLE_DRIVE, "tensegrity")
SCRATCH = tg.local_scratch("tensegrity_data")
_free = shutil.disk_usage(SCRATCH).free / 2**30
print(f"project dir : {PROJECT_DIR}\nlocal scratch: {SCRATCH}  ({_free:.0f} GB free)")

if DATASET == "custom":
    if not CUSTOM_DATA_PATH or not os.path.exists(CUSTOM_DATA_PATH):
        raise FileNotFoundError(
            "DATASET='custom' needs CUSTOM_DATA_PATH to point at a .txt/.md/.jsonl/.parquet file or a folder "
            "of them, e.g. /content/drive/MyDrive/my_corpus.txt")
    _src, _sep, _val = CUSTOM_DATA_PATH, (CUSTOM_DOC_SEPARATOR or None), 0.02
    _key = "custom-" + hashlib.md5(os.path.abspath(_src).encode()).hexdigest()[:8]
else:
    _mb = DATA_MEGABYTES if DATASET == "tinystories" else None
    _need = (_mb or tg.DATASETS[DATASET]["approx_mb"]) * 2.5 / 1024
    if _free < _need + 1:
        raise RuntimeError(f"need ~{_need:.1f} GB of local disk, only {_free:.1f} GB free")
    _spec = tg.fetch_dataset(DATASET, os.path.join(SCRATCH, "raw"), max_mb=_mb)
    _src, _sep, _val = _spec["path"], _spec["doc_sep"], _spec["val_fraction"]
    _key = f"{DATASET}-{_mb or 'full'}mb"

DATA_DIR = os.path.join(SCRATCH, "prepared", _key)
if not os.path.exists(os.path.join(DATA_DIR, "meta.json")):
    tg.prepare_data(_src, DATA_DIR, val_fraction=_val, doc_sep=_sep)
DATA_META = tg.load_meta(DATA_DIR)
print(f"\ndata ready: {DATA_META['n_train_tokens'] / 1e6:.1f}M train bytes, {DATA_META['n_val_tokens'] / 1e6:.2f}M val bytes, "
      f"{DATA_META['n_documents']:,} documents  ->  {DATA_DIR}")

## 3 · Plan the run
The notebook measures **your GPU's real throughput** on the chosen model, finds the largest micro-batch that fits in memory, and sizes the training run to your `TRAIN_HOURS` using a warmup-stable-decay schedule (so the learning rate always finishes decaying by the end of the time you gave it). If a plan for this run already exists in Drive it is reused unchanged, which is what makes resuming exact.

In [ ]:
#@title 3 · Hardware check, autotune & time budget { display-mode: "form" }
import shutil, time

DEVICE = tg.pick_device("auto")
DTYPE = tg.pick_dtype("auto", DEVICE)
PRESET = tg.get_preset(MODEL_SIZE)
_CPU_MODE = SMOKE_TEST or DEVICE == "cpu"
MODEL_OVERRIDES = dict(block_size=128, sliding_window=32) if _CPU_MODE else {}
BATCH_TOKENS = 4096 if _CPU_MODE else PRESET.batch_tokens      # small optimizer steps keep CPU runs responsive
if _CPU_MODE:
    print(f"(CPU / smoke-test mode: {MODEL_OVERRIDES['block_size']}-byte context, {BATCH_TOKENS}-byte optimizer steps)")

RUN_NAME_ = RUN_NAME.strip() or f"{PRESET.label}-{DATASET}"
RUN_DIR = os.path.join(PROJECT_DIR, "runs", RUN_NAME_)
if START_FRESH and os.path.exists(RUN_DIR):
    _bak = f"{RUN_DIR}.old-{int(time.time())}"
    shutil.move(RUN_DIR, _bak)                      # moved, never deleted
    print(f"START_FRESH: previous run kept at {_bak}")

_saved = tg.load_run_plan(RUN_DIR)
if _saved is not None:
    PLAN, TC, CFG = _saved
    model = tg.TensegrityLM(CFG)
    print(f"Resuming run '{RUN_NAME_}' with its saved plan ({PLAN.max_steps:,} steps).")
elif INIT_FROM_WEIGHTS:
    _src, CFG, _ = tg.load_model(INIT_FROM_WEIGHTS, "cpu")           # architecture comes from the weights
    model = tg.TensegrityLM(CFG)
    model.load_state_dict(_src.state_dict())
    del _src
    print(f"Initialised a {CFG.n_params() / 1e6:.2f}M-parameter model from {INIT_FROM_WEIGHTS}")
    if abs(CFG.n_params() / PRESET.model_kwargs_params() - 1) > 0.15:
        print(f"⚠️  These weights are not a {PRESET.label} model; hyper-parameters below come from the {PRESET.label} preset.")
else:
    CFG = tg.get_preset_config(PRESET.name, **MODEL_OVERRIDES)
    model = tg.TensegrityLM(CFG)
model.to(DEVICE)

if _saved is None:
    _tune = tg.autotune_micro_batch(model, CFG.block_size, DEVICE, DTYPE, BATCH_TOKENS, verbose=False)
    model.grad_checkpoint = _tune["grad_checkpoint"]
    _tps_step = _tune["micro_batch"] * _tune["grad_accum"] * CFG.block_size
    _tps = tg.measure_throughput(model, _tune["micro_batch"], CFG.block_size, DEVICE, DTYPE, steps=4, warmup=2)
    PLAN = tg.plan_for_time_budget(_tps, TRAIN_HOURS, _tps_step, max_tokens=PRESET.default_tokens)
    TC, PLAN = tg.train_setup(PRESET.name, _tune["micro_batch"], _tune["grad_accum"], plan=PLAN, optimizer=OPTIMIZER)
    _mfu = tg.mfu(CFG, _tps, CFG.block_size, tg.gpu_info()["peak_tflops"])
    print(f"measured throughput : {_tps / 1e3:,.1f}k bytes/s" + (f"  (~{_mfu * 100:.0f}% MFU)" if _mfu else ""))
    print(f"micro-batch         : {_tune['micro_batch']} x {_tune['grad_accum']} accumulation x {CFG.block_size} bytes"
          f"{'  [gradient checkpointing on: memory-bound]' if _tune['grad_checkpoint'] else ''}")
if _saved is None:                                  # new run: prove the optimizer path is numerically sound on THIS device
    TC = tg.preflight_or_fallback(model, TC, DEVICE)  # falls back to AdamW automatically if e.g. Muon misbehaves
TC.max_minutes = STOP_AFTER_MINUTES or None         # per-session limit, never persisted
_epochs = PLAN.total_tokens / max(1, DATA_META["n_train_tokens"])
print(f"model               : {CFG.describe()}")
def _human(n):
    return f"{n / 1e6:,.1f}M" if n >= 1e6 else f"{n / 1e3:,.0f}K"
print(f"training plan       : {PLAN.max_steps:,} steps x {PLAN.tokens_per_step:,} bytes = {_human(PLAN.total_tokens)} bytes "
      f"({_epochs:.2f} passes over your data) | peak LR {TC.learning_rate:.1e} | EMA decay {TC.ema_decay:.4f}")
print(f"schedule            : {PLAN.warmup_steps} warmup -> {PLAN.stable_steps:,} stable -> {PLAN.decay_steps:,} cool-down")
if _epochs > 8:
    print("⚠️  This plan repeats your data many times; consider more data (§1) or fewer hours to avoid memorisation.")

## 4 · Train
Run this cell as often as you like — it resumes automatically. The dashboard shows **bits per byte** (lower is better; 8.0 is chance, ≈1.0–1.5 is good story text), the learning-rate schedule and throughput. Every evaluation also prints a short sample so you can *read* the progress.

In [ ]:
#@title 4 · Train (resumable) { display-mode: "form" }
import math
live = tg.LivePlot()
trainer = tg.Trainer(model, TC, PLAN, DATA_DIR, RUN_DIR, device=DEVICE, callbacks=[live])
RESULT = trainer.fit(resume=True)
print("\n" + "=" * 70)
if RESULT["status"] == "completed":
    print(f"✅ Finished. Best validation: {RESULT['best_val_loss'] / math.log(2):.3f} bits/byte "
          f"({RESULT['best_source']} weights) after {RESULT['tokens_seen'] / 1e6:,.2f}M bytes.")
else:
    print(f"⏸  Stopped by your STOP_AFTER_MINUTES limit at step {RESULT['step']:,} of {PLAN.max_steps:,}. "
          f"Run this cell again to continue exactly where it left off.")

## 5 · Evaluate & talk to it

In [ ]:
#@title 5.1 · Load the best checkpoint & measure it { display-mode: "form" }
import math
BEST_PATH = os.path.join(RUN_DIR, "ckpt_best.pt")
BEST, BEST_CFG, CKPT = tg.load_model(BEST_PATH, DEVICE)
BEST.eval()
EVAL = tg.evaluate(BEST, DATA_DIR, BEST_CFG, DEVICE, DTYPE, n_batches=50, batch_size=16)
print(f"checkpoint from step {CKPT['step']:,} ({CKPT.get('best_source', 'raw')} weights)")
print(f"validation: {EVAL['loss']:.4f} nats/byte | {EVAL['bpb']:.3f} bits/byte | perplexity {EVAL['ppl']:.2f}  "
      f"(chance = {math.log(259):.2f} nats = {math.log2(259):.2f} bits)")
if BEST.confidence_head is not None:
    _cal = tg.confidence_calibration(BEST, DATA_DIR, DEVICE, n_batches=20, batch_size=8, dtype="auto")
    print(f"confidence head: next-byte accuracy {_cal['base_accuracy']:.3f}, calibration error (ECE) {_cal['ece']:.3f}")
    tg.plot_reliability(_cal)

In [ ]:
#@title 5.2 · Generate text (confidence-shaded) { display-mode: "form" }
PROMPT = "Once upon a time, there was a little"  #@param {type:"string"}
MAX_NEW_BYTES = 400  #@param {type:"integer"}
TEMPERATURE = 0.8  #@param {type:"slider", min:0.0, max:1.5, step:0.05}
TOP_K = 40  #@param {type:"integer"}
TOP_P = 0.95  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
MIN_P = 0.02  #@param {type:"slider", min:0.0, max:0.3, step:0.01}
CONFIDENCE_TEMPERATURE = True  #@param {type:"boolean"}
SEED = 0  #@param {type:"integer"}

import codecs
from IPython.display import HTML, display
tok = tg.ByteTokenizer()
torch.manual_seed(SEED)
_ids = [tok.BOS] + tok.encode(PROMPT)
_idx = torch.tensor([_ids], dtype=torch.long, device=DEVICE)
_dec = codecs.getincrementaldecoder("utf-8")("replace")
_toks, _conf = list(_ids), [None] * len(_ids)
print(PROMPT, end="", flush=True)
for _t, _c in BEST.stream(_idx, MAX_NEW_BYTES, temperature=TEMPERATURE, top_k=TOP_K or None, top_p=TOP_P,
                          min_p=MIN_P or None, use_confidence_temp=CONFIDENCE_TEMPERATURE and BEST.confidence_head is not None):
    _toks.append(_t); _conf.append(_c)
    if _t < 256:
        print(_dec.decode(bytes([_t])), end="", flush=True)
print("\n")
if BEST.confidence_head is not None:
    print("Shaded by the model's own confidence (green = sure, red = guessing):")
    display(HTML(tg.confidence_html(_toks[len(_ids):], _conf[len(_ids):])))

## 6 · Climb the ladder — grow a trained model into the next size
Train a small model quickly, then **grow it into the next size and keep going**. Growth keeps the model's function *exactly* (the notebook prints the maximum logit difference and fails loudly if it isn't ~0) and gives every new head, hidden unit and layer a non-zero gradient path — a property the previous release of this project silently lacked for width growth (new width was untrainable). Each rung is checkpointed and resumable; re-running this cell after a disconnect skips finished rungs.

**What to expect.** Growing preserves what was learned and lets you extend a run incrementally. In the repository's small CPU measurement (1M → 2M, equal compute) it did **not** train faster or reach a lower loss than training the 2M model from scratch, so use it for reuse and flexibility rather than as a shortcut; at larger scales that comparison is untested here.

Typical free-tier plan: `1M,2M,5M` with `0.15,0.3,1.0` hours.

In [ ]:
#@title 6 · Train → grow → train ... { display-mode: "form" }
RUN_THE_LADDER = False  #@param {type:"boolean"}
LADDER_RUNGS = "1M,2M,5M"  #@param {type:"string"}
HOURS_PER_RUNG = "0.15,0.3,1.0"  #@param {type:"string"}

if RUN_THE_LADDER:
    _rungs = [r.strip() for r in LADDER_RUNGS.split(",") if r.strip()]
    _hours = [float(h) for h in HOURS_PER_RUNG.split(",") if h.strip()]
    if len(_hours) == 1:
        _hours = _hours * len(_rungs)
    LADDER_ROOT = os.path.join(PROJECT_DIR, "ladders", f"{'-'.join(_rungs)}-{DATASET}")
    LADDER = tg.climb_ladder(_rungs, DATA_DIR, LADDER_ROOT, hours=_hours, optimizer=OPTIMIZER,
                             model_overrides=MODEL_OVERRIDES, batch_tokens=BATCH_TOKENS, callbacks=[tg.LivePlot()],
                             tc_overrides=dict(max_minutes=STOP_AFTER_MINUTES or None))
    print(f"\n{'rung':>6s} {'params':>9s} {'bits/byte':>10s} {'steps':>8s}  status")
    for r in LADDER:
        print(f"{r.rung.upper():>6s} {r.params / 1e6:8.2f}M {r.best_val_bpb:10.3f} {r.steps:8,d}  {r.status}"
              f"{'  (resumed from an earlier session)' if r.skipped else ''}")
else:
    print("Set RUN_THE_LADDER = True to train the rungs above in sequence.")

In [ ]:
#@title 6b · Or grow the model you just trained by ONE rung { display-mode: "form" }
GROW_NOW = False  #@param {type:"boolean"}

if GROW_NOW:
    _nxt = tg.next_preset(MODEL_SIZE)
    if _nxt is None:
        raise SystemExit("already at the top of the ladder (115M)")
    _target = tg.retarget(BEST_CFG, tg.get_preset_config(_nxt.name))
    GROWN, GROWN_CFG = tg.grow_model(BEST, BEST_CFG, target=_target, device=DEVICE)
    tg.check_function_preserved(BEST, GROWN, BEST_CFG, DEVICE)
    tg.check_growth_trainable(GROWN, BEST_CFG)
    _gm = tg.evaluate(GROWN, DATA_DIR, GROWN_CFG, DEVICE, DTYPE, n_batches=20, batch_size=16)
    GROWN_PATH = os.path.join(PROJECT_DIR, "grown", f"{RUN_NAME_}-to-{_nxt.label}.pt")
    tg.save_checkpoint(GROWN_PATH, GROWN, None, GROWN_CFG, 0, float("inf"))
    print(f"\n{BEST_CFG.n_params() / 1e6:.2f}M -> {GROWN_CFG.n_params() / 1e6:.2f}M parameters; "
          f"validation unchanged at {_gm['bpb']:.3f} bits/byte (just grown, not yet trained).")
    print(f"saved {GROWN_PATH}\n\nTo keep training it: in §1 set MODEL_SIZE = '{_nxt.label}' and "
          f"INIT_FROM_WEIGHTS = '{GROWN_PATH}', then re-run §3 and §4 (use a new RUN_NAME or START_FRESH).")
else:
    print("Set GROW_NOW = True to grow the trained model by one rung and save it.")

## 7 · Teach it something new without forgetting (EWC + replay)
Fine-tune on a *second* corpus while protecting what the model already knows, using two complementary tools: **replay** (a fraction of every batch still comes from the original data) and **Elastic Weight Consolidation** (a quadratic penalty, weighted by each weight's Fisher information, that resists moving the weights that mattered for the old data). The cell prints validation before and after on *both* corpora so you can see the trade-off and tune `EWC_LAMBDA`.

In [ ]:
#@title 7 · Continual learning on a second corpus { display-mode: "form" }
RUN_CONTINUAL = False  #@param {type:"boolean"}
NEW_DATASET = "shakespeare"  #@param ["shakespeare", "tinystories-valid", "custom"]
NEW_CUSTOM_PATH = ""  #@param {type:"string"}
REPLAY_FRACTION = 0.3  #@param {type:"slider", min:0.0, max:0.9, step:0.05}
EWC_LAMBDA = 100.0  #@param {type:"number"}
CONTINUAL_MEGABYTES_OF_TRAINING = 20  #@param {type:"number"}

if RUN_CONTINUAL:
    if NEW_DATASET == "custom":
        _nsrc, _nsep, _nval, _nkey = NEW_CUSTOM_PATH, None, 0.05, "newcustom"
    else:
        _nspec = tg.fetch_dataset(NEW_DATASET, os.path.join(SCRATCH, "raw"))
        _nsrc, _nsep, _nval, _nkey = _nspec["path"], _nspec["doc_sep"], _nspec["val_fraction"], NEW_DATASET
    NEW_DIR = os.path.join(SCRATCH, "prepared", _nkey)
    if not os.path.exists(os.path.join(NEW_DIR, "meta.json")):
        tg.prepare_data(_nsrc, NEW_DIR, val_fraction=_nval, doc_sep=_nsep)

    _old_model, _old_cfg, _ = tg.load_model(BEST_PATH, DEVICE)
    def _val(m, d):
        return tg.evaluate(m, d, _old_cfg, DEVICE, DTYPE, 20, 8)["bpb"]
    _before = (_val(_old_model, DATA_DIR), _val(_old_model, NEW_DIR))
    _fisher, _anchor = tg.compute_fisher_diagonal(_old_model, DATA_DIR, DEVICE, n_sequences=48)
    _ewc = tg.make_ewc_loss(_fisher, _anchor, EWC_LAMBDA)

    _tokens = int(CONTINUAL_MEGABYTES_OF_TRAINING * 1e6)
    _cplan = tg.make_plan(_tokens, TC.batch_size * TC.grad_accum_steps * _old_cfg.block_size, "wsd")
    _ctc = tg.TrainConfig(**{**TC.persistent_dict(), "learning_rate": TC.learning_rate * 0.3, "min_lr": TC.min_lr * 0.3,
                             "ema_decay": 0.0, "max_minutes": STOP_AFTER_MINUTES or None})
    _cdir = os.path.join(PROJECT_DIR, "runs", f"{RUN_NAME_}+{_nkey}")
    _model2 = tg.TensegrityLM(_old_cfg)
    _model2.load_state_dict(_old_model.state_dict())
    _fine = tg.Trainer(_model2, _ctc, _cplan, [NEW_DIR, DATA_DIR], _cdir,
                       data_weights=[1.0 - REPLAY_FRACTION, REPLAY_FRACTION] if REPLAY_FRACTION > 0 else [1.0, 0.0],
                       device=DEVICE, extra_loss=_ewc if EWC_LAMBDA > 0 else None, callbacks=[tg.LivePlot()])
    _fine.fit(resume=True)
    _m2, _, _ = tg.load_model(os.path.join(_cdir, "ckpt_best.pt"), DEVICE)
    _after = (_val(_m2, DATA_DIR), _val(_m2, NEW_DIR))
    print(f"\n{'':22s}{'original corpus':>18s}{'new corpus':>14s}   (bits/byte, lower is better)")
    print(f"{'before fine-tuning':22s}{_before[0]:18.3f}{_before[1]:14.3f}")
    print(f"{'after (replay+EWC)':22s}{_after[0]:18.3f}{_after[1]:14.3f}")
else:
    print("Set RUN_CONTINUAL = True to fine-tune on a second corpus with replay + EWC.")

## 8 · Export & share
Writes `tensegrity.safetensors` (a single self-describing file — the config is embedded in its header), a `config.json`, and an honest model card. Float16 halves the size for deployment; the INT8 option produces a CPU-inference copy.

In [ ]:
#@title 8 · Export (safetensors, model card, optional INT8 / Hugging Face) { display-mode: "form" }
EXPORT_DTYPE = "float16"  #@param ["float32", "float16", "bfloat16"]
ALSO_INT8_CPU_COPY = False  #@param {type:"boolean"}
DOWNLOAD_ZIP = True  #@param {type:"boolean"}
PUSH_TO_HUGGING_FACE = False  #@param {type:"boolean"}
HF_REPO_ID = ""  #@param {type:"string"}
HF_PRIVATE = True  #@param {type:"boolean"}

import shutil
EXPORT_DIR = os.path.join(PROJECT_DIR, "exports", RUN_NAME_)
_w, _c = tg.export_safetensors(BEST, BEST_CFG, EXPORT_DIR, "tensegrity", EXPORT_DTYPE,
                               extra_metadata={"val_bpb": f"{EVAL['bpb']:.4f}", "dataset": DATASET})
tg.write_model_card(EXPORT_DIR, BEST_CFG, RUN_NAME_, metrics={"validation bits/byte": EVAL["bpb"], "perplexity": EVAL["ppl"]},
                    training={"steps": CKPT["step"], "tokens seen": CKPT.get("tokens_seen", "n/a"), "optimizer": OPTIMIZER},
                    dataset=DATASET)
print(f"wrote {_w} ({os.path.getsize(_w) / 1e6:.1f} MB)")
if ALSO_INT8_CPU_COPY:
    _q = tg.quantize_int8_dynamic(BEST)
    if _q is not None:
        _qp = os.path.join(EXPORT_DIR, "tensegrity_int8_cpu.pt")
        torch.save(_q.state_dict(), _qp)
        print(f"wrote {_qp} ({os.path.getsize(_qp) / 1e6:.1f} MB)")
if PUSH_TO_HUGGING_FACE:
    _tok = tg.get_secret("HF_TOKEN")
    if not (_tok and HF_REPO_ID):
        print("To upload: add a Colab secret named HF_TOKEN (🔑 in the left sidebar) and set HF_REPO_ID = 'user/name'.")
    else:
        print("uploaded to", tg.push_to_hub(EXPORT_DIR, HF_REPO_ID, _tok, private=HF_PRIVATE))
if DOWNLOAD_ZIP:
    _zip = shutil.make_archive(os.path.join(SCRATCH, f"{RUN_NAME_}-export"), "zip", EXPORT_DIR)
    tg.download_to_browser(_zip)

## 9 · Optional: how fast is every size on *this* GPU?
Measures real training throughput for each preset (autotuned micro-batch) so you can choose a size that fits your time. Takes ~1–2 minutes.

In [ ]:
#@title 9 · Benchmark all sizes on this GPU { display-mode: "form" }
RUN_BENCHMARK = False  #@param {type:"boolean"}
BENCHMARK_SIZES = "1M,2M,5M,10M,50M,115M"  #@param {type:"string"}
if RUN_BENCHMARK:
    BENCH = tg.benchmark_presets([s.strip() for s in BENCHMARK_SIZES.split(",") if s.strip()], device="auto",
                                 **({"seq_len": 128} if DEVICE == "cpu" else {}))
else:
    print("Set RUN_BENCHMARK = True to time every preset.")

## Troubleshooting & FAQ

**Colab disconnected.** Reconnect, run §0 → §4 again. Datasets are re-downloaded to local disk (cached copies were on the lost VM); the run itself resumes from Drive with the same schedule, optimizer state, EMA and data order.

**"CUDA out of memory".** §3 autotunes the micro-batch and falls back to gradient checkpointing, so this should only happen if another process holds GPU memory — Runtime → Restart session, then run again. Smaller model or shorter `block_size` always fits.

**Which size should I pick?** 5M is the sweet spot on a free T4 (a few hours → clearly coherent stories). 1M/2M are for fast experiments. 10M if you have ~4 h. 50M/115M need an L4/A100 or many hours; use the ladder (§6) to get there incrementally.

**Why bytes?** No tokenizer to train or ship, any language or file type works, and there are no out-of-vocabulary failures — at the cost of ~4× longer sequences than subword models for the same text.

**Is the loss comparable to other models'?** Bits per byte is the tokenizer-independent number; compare *that*, on the same held-out text.

**My own data.** `DATASET = "custom"` with `CUSTOM_DATA_PATH` (file or folder; `.txt .md .jsonl .parquet`). For documents joined by a marker in one big file, set `CUSTOM_DOC_SEPARATOR`; for jsonl/parquet each record becomes one document automatically.

**Files in Drive.** `MyDrive/tensegrity/runs/<run>/` holds `ckpt_latest.pt` (full, resumable), `ckpt_best.pt` (weights only), `train_log.jsonl`, `run_plan.json`. Exports go to `exports/`.

**Reporting a bug / contributing.** https://github.com/recursive-ai-dev/tensegrity